# 美联储利率决议预测：多智能体协同（System C）

本笔记本展示以货币政策理论智能体、分析整合层和决策层构成的实验框架。运行前请阅读仓库 README，并准备本地数据与 API 密钥。默认不会自动开始训练或预测。


In [ ]:
from __future__ import annotations

import csv
import json
import os
import sys
import types
from datetime import datetime, timezone
from pathlib import Path

# 从仓库根目录启动 Jupyter；数据放在 data/，实验产物写入仓库根目录下的 artifacts/。
framework_dir = Path.cwd().resolve()
project_root = framework_dir / "data"

class _NotebookRuntimeModule(types.ModuleType):
    def __getattr__(self, name: str):
        notebook_globals = sys.modules["__main__"].__dict__
        if name in notebook_globals:
            return notebook_globals[name]
        raise AttributeError(f"module {self.__name__!r} has no attribute {name!r}")


def _install_notebook_runtime_modules() -> None:
    module_names = (
        "schemas", "schemas.labels", "schemas.weights", "schemas.theory_outputs",
        "schemas.decision_outputs", "schemas.analysis_outputs", "schemas.samples",
        "theory_specs", "theory_specs.base", "theory_specs.catalog",
        "sample_builder", "sample_builder.xlsx_reader", "sample_builder.hp_filter", "sample_builder.us_builder",
        "theory_input_builder", "theory_input_builder.theory_input_models",
        "theory_input_builder.history_access_guard", "theory_input_builder.variable_router",
        "theory_input_builder.builder",
        "theory_agents", "theory_agents.base", "theory_agents.output_parser",
        "theory_agents.prompt_renderer", "theory_agents.runner",
        "analysis_handover", "analysis_handover.builder",
        "decision_agent", "decision_agent.context", "decision_agent.base",
        "decision_agent.output_parser", "decision_agent.prompt_renderer", "decision_agent.runner",
        "state_models", "state_models.pipeline",
        "prediction_pipeline", "prediction_pipeline.orchestrator",
        "inference_loop", "inference_loop.driver",
        "model_backends", "model_backends.openai_compat",
        "runtime_assembly", "runtime_assembly.factory",
        "application_entry", "application_entry.service",
    )
    for module_name in module_names:
        sys.modules.pop(module_name, None)
        module = _NotebookRuntimeModule(module_name)
        module.__dict__["__package__"] = module_name.rpartition(".")[0]
        if "." not in module_name:
            module.__dict__["__path__"] = []
        sys.modules[module_name] = module


_install_notebook_runtime_modules()
NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION = False
NOTEBOOK_SYSTEM_VARIANT = "system_c_training_set_textual_memory"


## Core Schemas

In [ ]:
# 文件：schemas/labels.py
# 产出：产出利率标签枚举与标签顺序定义。
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from enum import StrEnum

    class RateMoveLabel(StrEnum):
        LESS_THAN_NEGATIVE_50 = "<-50bp"
        NEGATIVE_50_TO_25 = "[-50,-25)bp"
        NEGATIVE_25_TO_0 = "[-25,0)bp"
        ZERO = "0bp"
        POSITIVE_0_TO_25 = "(0,25]bp"
        POSITIVE_25_TO_50 = "(25,50]bp"
        GREATER_THAN_50 = ">50bp"

    LABEL_ORDER: tuple[RateMoveLabel, ...] = (
        RateMoveLabel.LESS_THAN_NEGATIVE_50,
        RateMoveLabel.NEGATIVE_50_TO_25,
        RateMoveLabel.NEGATIVE_25_TO_0,
        RateMoveLabel.ZERO,
        RateMoveLabel.POSITIVE_0_TO_25,
        RateMoveLabel.POSITIVE_25_TO_50,
        RateMoveLabel.GREATER_THAN_50,
    )

    LABEL_ALIASES: dict[str, RateMoveLabel] = {
        "< -50bp": RateMoveLabel.LESS_THAN_NEGATIVE_50,
        "<=-50bp": RateMoveLabel.NEGATIVE_50_TO_25,
        "-50bp": RateMoveLabel.NEGATIVE_50_TO_25,
        "-25bp": RateMoveLabel.NEGATIVE_25_TO_0,
        "0": RateMoveLabel.ZERO,
        "0 bp": RateMoveLabel.ZERO,
        "+0bp": RateMoveLabel.ZERO,
        "25bp": RateMoveLabel.POSITIVE_0_TO_25,
        "+25bp": RateMoveLabel.POSITIVE_0_TO_25,
        "50bp": RateMoveLabel.POSITIVE_25_TO_50,
        "+50bp": RateMoveLabel.POSITIVE_25_TO_50,
        ">=50bp": RateMoveLabel.GREATER_THAN_50,
        "> 50bp": RateMoveLabel.GREATER_THAN_50,
    }
    def validate_label(value: str | RateMoveLabel) -> RateMoveLabel:
        if isinstance(value, RateMoveLabel):
            return value
        normalized = str(value).strip()
        alias = LABEL_ALIASES.get(normalized)
        if alias is not None:
            return alias
        try:
            return RateMoveLabel(normalized)
        except ValueError as exc:
            allowed = ", ".join(label.value for label in LABEL_ORDER)
            raise ValueError(f"Invalid rate-move label: {value!r}. Allowed labels: {allowed}.") from exc


In [ ]:
# file: schemas/weights.py
# System C uses stable theory ids plus validated textual theory memory contracts.
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from math import isclose

    THEORY_IDS: tuple[str, ...] = (
        "standard_taylor",
        "forward_looking_taylor",
        "interest_rate_smoothing",
        "open_economy_rule",
        "financial_stability_rule",
        "new_keynesian_optimal_rule",
        "balanced_approach_rule",
    )

    NUMERIC_WEIGHT_TOLERANCE = 5e-3

    def uniform_theory_weight_profile() -> dict[str, float]:
        value = 1.0 / len(THEORY_IDS)
        return {theory_id: value for theory_id in THEORY_IDS}

    def validate_theory_weight_profile(field_name: str, value: dict[str, float]) -> dict[str, float]:
        if not isinstance(value, dict):
            raise TypeError(f"{field_name} must be a dict.")
        missing = [theory_id for theory_id in THEORY_IDS if theory_id not in value]
        extras = sorted(set(value) - set(THEORY_IDS))
        if missing or extras:
            raise ValueError(f"{field_name} must cover all registered theory ids exactly. Missing: {missing}; extras: {extras}.")
        normalized: dict[str, float] = {}
        for theory_id in THEORY_IDS:
            raw_value = value[theory_id]
            if not isinstance(raw_value, (int, float)):
                raise TypeError(f"{field_name}[{theory_id}] must be numeric.")
            numeric_value = float(raw_value)
            if numeric_value < 0.0:
                raise ValueError(f"{field_name}[{theory_id}] must be non-negative.")
            normalized[theory_id] = numeric_value
        total = sum(normalized.values())
        if total <= 0.0:
            raise ValueError(f"{field_name} must have a positive total.")
        if not isclose(total, 1.0, rel_tol=0.0, abs_tol=NUMERIC_WEIGHT_TOLERANCE):
            raise ValueError(f"{field_name} must sum to 1.0, got {total}.")
        return {theory_id: normalized[theory_id] / total for theory_id in THEORY_IDS}

    def normalize_positive_scores_to_weights(scores: dict[str, float]) -> dict[str, float]:
        if not isinstance(scores, dict):
            raise TypeError("scores must be a dict.")
        normalized_scores: dict[str, float] = {}
        for theory_id in THEORY_IDS:
            value = scores.get(theory_id, 0.0)
            if not isinstance(value, (int, float)):
                raise TypeError(f"scores[{theory_id}] must be numeric.")
            normalized_scores[theory_id] = max(float(value), 0.0)
        total = sum(normalized_scores.values())
        if total <= 0.0:
            return uniform_theory_weight_profile()
        return {theory_id: normalized_scores[theory_id] / total for theory_id in THEORY_IDS}

    CURRENT_NUMERIC_THEORY_WEIGHT_PROFILE: dict[str, float] = uniform_theory_weight_profile()


In [ ]:
# 文件：schemas/theory_outputs.py
# 产出：产出理论层输出与概率分布 schema。
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from dataclasses import dataclass, field
    from math import isclose



    def _validate_non_empty_text(field_name: str, value: str) -> str:
        if not isinstance(value, str):
            raise TypeError(f"{field_name} must be a string.")
        normalized = value.strip()
        if not normalized:
            raise ValueError(f"{field_name} must not be empty.")
        return normalized


    @dataclass(slots=True)
    class ProbabilityDistribution:
        probabilities: dict[RateMoveLabel, float]
        SUM_TOLERANCE: float = 5e-3

        def __post_init__(self) -> None:
            normalized: dict[RateMoveLabel, float] = {}
            for key, value in self.probabilities.items():
                label = validate_label(key)
                if not isinstance(value, (int, float)):
                    raise TypeError(f"Probability for {label.value} must be numeric.")
                probability = float(value)
                if probability < 0.0 or probability > 1.0:
                    raise ValueError(f"Probability for {label.value} must be within [0, 1].")
                normalized[label] = probability

            missing = [label.value for label in LABEL_ORDER if label not in normalized]
            extra_count = len(normalized) - len(LABEL_ORDER)
            if missing or extra_count != 0:
                raise ValueError(
                    "Probability distribution must cover all 7 fixed labels exactly once. "
                    f"Missing labels: {missing}."
                )

            total_probability = sum(normalized.values())
            if not isclose(total_probability, 1.0, rel_tol=0.0, abs_tol=self.SUM_TOLERANCE):
                raise ValueError(f"Probability distribution must sum to 1.0, got {total_probability}.")
            if total_probability <= 0.0:
                raise ValueError("Probability distribution total must be strictly positive.")

            normalized = {
                label: normalized[label] / total_probability
                for label in LABEL_ORDER
            }

            self.probabilities = normalized

        def probability_for(self, label: str | RateMoveLabel) -> float:
            return self.probabilities[validate_label(label)]

        def top_label(self) -> RateMoveLabel:
            return max(LABEL_ORDER, key=lambda label: self.probabilities[label])

        def as_ordered_dict(self) -> dict[str, float]:
            return {label.value: self.probabilities[label] for label in LABEL_ORDER}


    @dataclass(slots=True)
    class TheoryAgentOutput:
        recommendation: RateMoveLabel
        probability_distribution: ProbabilityDistribution
        confidence: int
        primary_policy_reason: str
        secondary_policy_reason: str
        key_variables_used: list[str]
        theory_explanation: str
        applicability_note: str
        risk_note: str
        policy_path_regime_assessment: dict[str, object] = field(default_factory=dict)

        def __post_init__(self) -> None:
            self.recommendation = validate_label(self.recommendation)

            if not isinstance(self.probability_distribution, ProbabilityDistribution):
                raise TypeError("probability_distribution must be a ProbabilityDistribution instance.")

            if self.recommendation != self.probability_distribution.top_label():
                raise ValueError(
                    "recommendation must match the highest-probability label in probability_distribution."
                )

            if not isinstance(self.confidence, int):
                raise TypeError("confidence must be an integer.")
            if not 0 <= self.confidence <= 100:
                raise ValueError("confidence must be within [0, 100].")

            self.primary_policy_reason = _validate_non_empty_text(
                "primary_policy_reason", self.primary_policy_reason
            )
            self.secondary_policy_reason = _validate_non_empty_text(
                "secondary_policy_reason", self.secondary_policy_reason
            )
            self.theory_explanation = _validate_non_empty_text("theory_explanation", self.theory_explanation)
            self.applicability_note = _validate_non_empty_text("applicability_note", self.applicability_note)
            self.risk_note = _validate_non_empty_text("risk_note", self.risk_note)

            if not isinstance(self.policy_path_regime_assessment, dict):
                raise TypeError("policy_path_regime_assessment must be a dict.")
            self.policy_path_regime_assessment = dict(self.policy_path_regime_assessment)

            if not isinstance(self.key_variables_used, list):
                raise TypeError("key_variables_used must be a list of variable names.")
            normalized_variables: list[str] = []
            for variable_name in self.key_variables_used:
                normalized_variables.append(_validate_non_empty_text("key_variables_used item", variable_name))
            if not normalized_variables:
                raise ValueError("key_variables_used must contain at least one variable name.")
            self.key_variables_used = normalized_variables


In [ ]:
# file: schemas/decision_outputs.py
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from dataclasses import dataclass

    def _validate_non_empty_text(field_name: str, value: str) -> str:
        if not isinstance(value, str):
            raise TypeError(f"{field_name} must be a string.")
        normalized = value.strip()
        if not normalized:
            raise ValueError(f"{field_name} must not be empty.")
        return normalized

    def _validate_text_list(field_name: str, value: list[str]) -> list[str]:
        if not isinstance(value, list):
            raise TypeError(f"{field_name} must be a list.")
        return [_validate_non_empty_text(field_name, str(item)) for item in value]

    def _validate_rule_family(field_name: str, value: str) -> str:
        if value not in THEORY_IDS:
            raise ValueError(f"{field_name} must be one of {THEORY_IDS}, got {value!r}.")
        return value

    def _validate_rule_family_list(field_name: str, value: list[str]) -> list[str]:
        if not isinstance(value, list):
            raise TypeError(f"{field_name} must be a list.")
        normalized = [_validate_rule_family(field_name, item) for item in value]
        if len(set(normalized)) != len(normalized):
            raise ValueError(f"{field_name} must not contain duplicates.")
        return normalized

    def _validate_first_contributor(first_contributor_rule: str, shares: dict[str, float]) -> str:
        first_contributor_rule = _validate_rule_family("first_contributor_rule", first_contributor_rule)
        top_share = max(shares.values())
        top_theories = {theory_id for theory_id, value in shares.items() if abs(value - top_share) <= 1e-9}
        if first_contributor_rule not in top_theories:
            raise ValueError("first_contributor_rule must be one of the highest-share theories in rule_contribution_shares.")
        return first_contributor_rule

    @dataclass(slots=True)
    class DecisionAgentOutput:
        final_recommendation: RateMoveLabel | str
        final_probability_distribution: ProbabilityDistribution
        final_confidence: int
        final_explanation: str
        dominant_rule_family: str
        supporting_rule_families: list[str]
        theory_weight_profile: dict[str, float]
        theory_weight_rationale: str
        rule_contribution_shares: dict[str, float]
        memory_usage_summary: str
        matched_memory_lessons: list[str]
        matched_conditional_rules: list[str]
        rule_application_summary: str
        direction_decision: str
        magnitude_decision_rationale: str
        first_contributor_rule: str
        current_sample_reasoning: str
        remaining_uncertainty: str
        decision_path: str

        def __post_init__(self) -> None:
            self.final_recommendation = validate_label(self.final_recommendation)
            if not isinstance(self.final_probability_distribution, ProbabilityDistribution):
                raise TypeError("final_probability_distribution must be a ProbabilityDistribution instance.")
            if self.final_recommendation != self.final_probability_distribution.top_label():
                raise ValueError("final_recommendation must match the highest-probability label in final_probability_distribution.")
            if not isinstance(self.final_confidence, int):
                raise TypeError("final_confidence must be an integer.")
            if not 0 <= self.final_confidence <= 100:
                raise ValueError("final_confidence must be within [0, 100].")
            self.final_explanation = _validate_non_empty_text("final_explanation", self.final_explanation)
            self.dominant_rule_family = _validate_rule_family("dominant_rule_family", self.dominant_rule_family)
            self.supporting_rule_families = _validate_rule_family_list("supporting_rule_families", self.supporting_rule_families)
            if self.dominant_rule_family in self.supporting_rule_families:
                raise ValueError("supporting_rule_families must not contain dominant_rule_family.")
            self.theory_weight_profile = validate_theory_weight_profile("theory_weight_profile", self.theory_weight_profile)
            self.theory_weight_rationale = _validate_non_empty_text("theory_weight_rationale", self.theory_weight_rationale)
            self.rule_contribution_shares = validate_theory_weight_profile("rule_contribution_shares", self.rule_contribution_shares)
            self.memory_usage_summary = _validate_non_empty_text("memory_usage_summary", self.memory_usage_summary)
            self.matched_memory_lessons = _validate_text_list("matched_memory_lessons", self.matched_memory_lessons)
            self.matched_conditional_rules = _validate_text_list("matched_conditional_rules", self.matched_conditional_rules)
            self.rule_application_summary = _validate_non_empty_text("rule_application_summary", self.rule_application_summary)
            self.direction_decision = _validate_non_empty_text("direction_decision", self.direction_decision)
            if self.direction_decision not in {"cut", "hold", "hike", "uncertain"}:
                raise ValueError("direction_decision must be one of cut, hold, hike, uncertain.")
            self.magnitude_decision_rationale = _validate_non_empty_text("magnitude_decision_rationale", self.magnitude_decision_rationale)
            self.first_contributor_rule = _validate_first_contributor(self.first_contributor_rule, self.rule_contribution_shares)
            self.current_sample_reasoning = _validate_non_empty_text("current_sample_reasoning", self.current_sample_reasoning)
            self.remaining_uncertainty = _validate_non_empty_text("remaining_uncertainty", self.remaining_uncertainty)
            if self.decision_path != "training_textual_memory_informed_aggregation":
                raise ValueError("decision_path must be 'training_textual_memory_informed_aggregation'.")


In [ ]:
# 文件：schemas/analysis_outputs.py
# 产出：产出分析交接层输出 schema。
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from dataclasses import dataclass
    from typing import Any

    def _validate_non_empty_text(field_name: str, value: str) -> str:
        if not isinstance(value, str):
            raise TypeError(f"{field_name} must be a string.")
        normalized = value.strip()
        if not normalized:
            raise ValueError(f"{field_name} must not be empty.")
        return normalized

    def _validate_bool(field_name: str, value: bool) -> bool:
        if not isinstance(value, bool):
            raise TypeError(f"{field_name} must be a bool.")
        return value

    def _validate_theory_mapping(field_name: str, value: dict[str, Any]) -> dict[str, Any]:
        if not isinstance(value, dict):
            raise TypeError(f"{field_name} must be a dict.")
        missing = [theory_id for theory_id in THEORY_IDS if theory_id not in value]
        extras = sorted(set(value) - set(THEORY_IDS))
        if missing or extras:
            raise ValueError(
                f"{field_name} must cover all registered theories exactly. Missing: {missing}; extras: {extras}."
            )
        return dict(value)

    def _validate_policy_phase_signal(field_name: str, value: dict[str, Any] | None) -> dict[str, Any]:
        if value is None:
            value = {}
        if not isinstance(value, dict):
            raise TypeError(f"{field_name} must be a dict.")
        normalized = dict(value)
        allowed_phases = {
            "near_zlb_waiting",
            "liftoff_start",
            "rapid_hiking_cycle",
            "high_rate_plateau",
            "easing_after_plateau",
            "crisis_cut",
            "normal",
            "uncertain",
        }
        allowed_direction_constraints = {"hold_bias", "hike_bias", "cut_bias", "no_constraint", "uncertain"}
        allowed_magnitude_constraints = {"avoid_large", "allow_large", "prefer_small", "prefer_hold", "no_constraint", "uncertain"}
        normalized.setdefault("phase", "uncertain")
        normalized.setdefault("phase_confidence", "low")
        normalized.setdefault("evidence", "")
        normalized.setdefault("direction_constraint", "uncertain")
        normalized.setdefault("magnitude_constraint", "uncertain")
        normalized.setdefault("anti_overfit_warning", "This signal is a current-sample diagnostic, not a date answer key.")
        if str(normalized["phase"]) not in allowed_phases:
            normalized["phase"] = "uncertain"
        if str(normalized["direction_constraint"]) not in allowed_direction_constraints:
            normalized["direction_constraint"] = "uncertain"
        if str(normalized["magnitude_constraint"]) not in allowed_magnitude_constraints:
            normalized["magnitude_constraint"] = "uncertain"
        normalized["evidence"] = str(normalized.get("evidence", "")).strip()
        normalized["phase_confidence"] = str(normalized.get("phase_confidence", "low")).strip() or "low"
        normalized["anti_overfit_warning"] = str(normalized.get("anti_overfit_warning", "")).strip()
        return normalized

    def _validate_label_mapping(field_name: str, value: dict[str, Any]) -> dict[str, Any]:
        if not isinstance(value, dict):
            raise TypeError(f"{field_name} must be a dict.")
        normalized: dict[str, Any] = {}
        for key, item in value.items():
            normalized[validate_label(key).value] = item
        missing = [label.value for label in LABEL_ORDER if label.value not in normalized]
        extras = sorted(set(normalized) - {label.value for label in LABEL_ORDER})
        if missing or extras:
            raise ValueError(
                f"{field_name} must cover all 7 labels exactly. Missing: {missing}; extras: {extras}."
            )
        return normalized

    def _validate_optional_text(field_name: str, value: str | None) -> str | None:
        if value is None:
            return None
        return _validate_non_empty_text(field_name, value)

    @dataclass(slots=True)
    class AnalysisSynthesisOutput:
        reason_summary_by_rule: dict[str, Any]
        applicability_summary_by_rule: dict[str, Any]
        risk_summary_by_rule: dict[str, Any]
        main_disagreement_dimension: str
        secondary_disagreement_dimension: str | None = None
        high_conflict_flag: bool = False
        minority_high_confidence_alert: bool = False
        policy_phase_signal: dict[str, Any] | None = None

        def __post_init__(self) -> None:
            self.reason_summary_by_rule = _validate_theory_mapping(
                "reason_summary_by_rule", self.reason_summary_by_rule
            )
            self.applicability_summary_by_rule = _validate_theory_mapping(
                "applicability_summary_by_rule", self.applicability_summary_by_rule
            )
            self.risk_summary_by_rule = _validate_theory_mapping(
                "risk_summary_by_rule", self.risk_summary_by_rule
            )
            self.main_disagreement_dimension = _validate_non_empty_text(
                "main_disagreement_dimension", self.main_disagreement_dimension
            )
            self.secondary_disagreement_dimension = _validate_optional_text(
                "secondary_disagreement_dimension", self.secondary_disagreement_dimension
            )
            self.high_conflict_flag = _validate_bool("high_conflict_flag", self.high_conflict_flag)
            self.minority_high_confidence_alert = _validate_bool(
                "minority_high_confidence_alert", self.minority_high_confidence_alert
            )
            self.policy_phase_signal = _validate_policy_phase_signal(
                "policy_phase_signal", self.policy_phase_signal
            )

    @dataclass(slots=True)
    class AnalysisPayload:
        label_support_table: dict[str, Any]
        agent_position_table: dict[str, Any]
        agent_distribution_table: dict[str, Any]
        agent_confidence_table: dict[str, Any]
        reason_summary_by_rule: dict[str, Any]
        applicability_summary_by_rule: dict[str, Any]
        risk_summary_by_rule: dict[str, Any]
        main_disagreement_dimension: str
        secondary_disagreement_dimension: str | None = None
        high_conflict_flag: bool = False
        minority_high_confidence_alert: bool = False
        policy_phase_signal: dict[str, Any] | None = None

        def __post_init__(self) -> None:
            self.label_support_table = _validate_label_mapping("label_support_table", self.label_support_table)
            self.agent_position_table = _validate_theory_mapping(
                "agent_position_table", self.agent_position_table
            )
            self.agent_distribution_table = _validate_theory_mapping(
                "agent_distribution_table", self.agent_distribution_table
            )
            self.agent_confidence_table = _validate_theory_mapping(
                "agent_confidence_table", self.agent_confidence_table
            )
            self.reason_summary_by_rule = _validate_theory_mapping(
                "reason_summary_by_rule", self.reason_summary_by_rule
            )
            self.applicability_summary_by_rule = _validate_theory_mapping(
                "applicability_summary_by_rule", self.applicability_summary_by_rule
            )
            self.risk_summary_by_rule = _validate_theory_mapping(
                "risk_summary_by_rule", self.risk_summary_by_rule
            )
            self.main_disagreement_dimension = _validate_non_empty_text(
                "main_disagreement_dimension", self.main_disagreement_dimension
            )
            self.secondary_disagreement_dimension = _validate_optional_text(
                "secondary_disagreement_dimension", self.secondary_disagreement_dimension
            )
            self.high_conflict_flag = _validate_bool("high_conflict_flag", self.high_conflict_flag)
            self.minority_high_confidence_alert = _validate_bool(
                "minority_high_confidence_alert", self.minority_high_confidence_alert
            )
            self.policy_phase_signal = _validate_policy_phase_signal(
                "policy_phase_signal", self.policy_phase_signal
            )

    @dataclass(slots=True)
    class DecisionHandoverPacket:
        label_support_table: dict[str, Any]
        agent_position_table: dict[str, Any]
        agent_distribution_table: dict[str, Any]
        agent_confidence_table: dict[str, Any]
        reason_summary_by_rule: dict[str, Any]
        applicability_summary_by_rule: dict[str, Any]
        risk_summary_by_rule: dict[str, Any]
        main_disagreement_dimension: str
        secondary_disagreement_dimension: str | None
        high_conflict_flag: bool
        minority_high_confidence_alert: bool
        policy_phase_signal: dict[str, Any] | None = None

        def __post_init__(self) -> None:
            self.label_support_table = _validate_label_mapping("label_support_table", self.label_support_table)
            self.agent_position_table = _validate_theory_mapping(
                "agent_position_table", self.agent_position_table
            )
            self.agent_distribution_table = _validate_theory_mapping(
                "agent_distribution_table", self.agent_distribution_table
            )
            self.agent_confidence_table = _validate_theory_mapping(
                "agent_confidence_table", self.agent_confidence_table
            )
            self.reason_summary_by_rule = _validate_theory_mapping(
                "reason_summary_by_rule", self.reason_summary_by_rule
            )
            self.applicability_summary_by_rule = _validate_theory_mapping(
                "applicability_summary_by_rule", self.applicability_summary_by_rule
            )
            self.risk_summary_by_rule = _validate_theory_mapping(
                "risk_summary_by_rule", self.risk_summary_by_rule
            )
            self.main_disagreement_dimension = _validate_non_empty_text(
                "main_disagreement_dimension", self.main_disagreement_dimension
            )
            self.secondary_disagreement_dimension = _validate_optional_text(
                "secondary_disagreement_dimension", self.secondary_disagreement_dimension
            )
            self.high_conflict_flag = _validate_bool("high_conflict_flag", self.high_conflict_flag)
            self.minority_high_confidence_alert = _validate_bool(
                "minority_high_confidence_alert", self.minority_high_confidence_alert
            )
            self.policy_phase_signal = _validate_policy_phase_signal(
                "policy_phase_signal", self.policy_phase_signal
            )


In [ ]:
# 文件：schemas/samples.py
# 产出：产出训练样本、预测样本与基础样本 schema。
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from dataclasses import dataclass, field
    from datetime import date
    from typing import Any

    def _validate_date(field_name: str, value: date) -> date:
        if not isinstance(value, date):
            raise TypeError(f"{field_name} must be a datetime.date instance.")
        return value

    def _validate_mapping(field_name: str, value: dict[str, Any]) -> dict[str, Any]:
        if not isinstance(value, dict):
            raise TypeError(f"{field_name} must be a dict.")
        return value

    @dataclass(slots=True)
    class BaseDecisionSample:
        country: str
        decision_date: date
        label_space_definition: tuple[RateMoveLabel, ...] = field(default_factory=lambda: LABEL_ORDER)
        current_macro_snapshot: dict[str, Any] = field(default_factory=dict)
        market_expectations: dict[str, Any] = field(default_factory=dict)
        policy_inputs: dict[str, Any] = field(default_factory=dict)
        textual_inputs: dict[str, Any] = field(default_factory=dict)

        def __post_init__(self) -> None:
            if not isinstance(self.country, str) or not self.country.strip():
                raise ValueError("country must be a non-empty string.")
            self.country = self.country.strip()
            self.decision_date = _validate_date("decision_date", self.decision_date)

            if tuple(self.label_space_definition) != LABEL_ORDER:
                raise ValueError("label_space_definition must match the fixed 7-label rate-decision space.")

            self.current_macro_snapshot = _validate_mapping(
                "current_macro_snapshot", self.current_macro_snapshot
            )
            self.market_expectations = _validate_mapping("market_expectations", self.market_expectations)
            self.policy_inputs = _validate_mapping("policy_inputs", self.policy_inputs)
            self.textual_inputs = _validate_mapping("textual_inputs", self.textual_inputs)

    @dataclass(slots=True)
    class TrainingDecisionSample(BaseDecisionSample):
        realized_label: RateMoveLabel | str | None = None

        def __post_init__(self) -> None:
            BaseDecisionSample.__post_init__(self)
            if self.realized_label is None:
                raise ValueError("TrainingDecisionSample must include realized_label.")
            self.realized_label = validate_label(self.realized_label)

    @dataclass(slots=True)
    class PredictionDecisionSample(BaseDecisionSample):
        realized_label: None = field(default=None, init=False)


In [ ]:
# file: schemas/__init__.py
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    __all__ = [
        "AnalysisPayload",
        "BaseDecisionSample",
        "DecisionAgentOutput",
        "DecisionHandoverPacket",
        "LABEL_ORDER",
        "PredictionDecisionSample",
        "ProbabilityDistribution",
        "RateMoveLabel",
        "THEORY_IDS",
        "TheoryAgentOutput",
        "TrainingDecisionSample",
        "validate_label",
    ]


## Theory Specs and Samples

In [ ]:
# 文件：theory_specs/base.py
# 产出：产出理论规格基础结构与历史访问策略枚举。
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from dataclasses import dataclass
    from enum import StrEnum

    class HistoryAccessPolicy(StrEnum):
        NONE = "none"
        PREVIOUS_POLICY_RATE_ONLY = "previous_policy_rate_only"
        FULL_DECISION_RATE_HISTORY = "full_decision_rate_history"

    @dataclass(frozen=True, slots=True)
    class PromptModuleSpec:
        theory_identity: str
        policy_logic: str
        theory_anchor_keys: tuple[str, ...]
        execution_protocol: str

    @dataclass(frozen=True, slots=True)
    class TheorySpec:
        theory_id: str
        display_name: str
        prompt_modules: PromptModuleSpec
        allowed_variables: tuple[str, ...]
        history_access_policy: HistoryAccessPolicy
        required_policy_history_keys: tuple[str, ...]
        forbidden_logic_crossovers: tuple[str, ...]
        output_schema_name: str = "TheoryAgentOutput"

        def __post_init__(self) -> None:
            if not self.theory_id.strip():
                raise ValueError("theory_id must be non-empty.")
            if not self.display_name.strip():
                raise ValueError("display_name must be non-empty.")
            if not self.allowed_variables:
                raise ValueError(f"{self.theory_id} must define allowed_variables.")
            if not self.prompt_modules.theory_anchor_keys:
                raise ValueError(f"{self.theory_id} must define theory_anchor_keys.")


In [ ]:
# 文件：theory_specs/catalog.py
# 产出：产出七个理论的正式规格、中文 prompt 模块与变量边界。
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    THEORY_SPECS: tuple[TheorySpec, ...] = (
        TheorySpec(
            theory_id="standard_taylor",
            display_name="Standard Taylor Rule",
            prompt_modules=PromptModuleSpec(
                theory_identity=(
                    "你代表标准泰勒规则理论。你的职责是依据经典规则式国内稳定化逻辑，"
                    "围绕通胀相对目标的偏离和当期产出缺口，对当前决议时点的利率变动标签作出判断。"
                    "你不是最终决策层，不负责整合其他理论，不负责根据系统历史表现调整自身立场。"
                ),
                policy_logic=(
                    "本理论的主判断逻辑固定为当期国内稳定化反应规则。第一步，识别 `inflation_current` 相对 "
                    "`inflation_target_mid` 的偏离方向与偏离强度；第二步，识别 `output_gap` 所反映的需求松紧与产出状态；"
                    "第三步，将通胀偏离与产出缺口的共同方向或冲突关系转化为对利率调整方向和幅度的判断。"
                    "当通胀信号与产出信号一致时，应说明二者如何共同强化某一标签；当二者冲突时，应明确说明冲突来源，"
                    "并仍以标准泰勒规则的国内稳定化逻辑给出结论。"
                ),
                theory_anchor_keys=(
                    "通胀偏离目标",
                    "当期产出缺口",
                    "国内稳定化取向",
                    "规则式当期反应",
                ),
                execution_protocol=(
                    "你只能使用当前样本中被授权暴露给本理论的字段。"
                    "主判断必须显式使用 `inflation_current`、`inflation_target_mid` 和 `output_gap`。"
                    "`key_variables_used` 必须优先反映本次判断真正主导结论的核心变量，不能只列非本理论变量。"
                    "你必须输出固定 9 字段结构，且 `recommendation` 必须与 `probability_distribution` 中最高概率标签一致。"
                    "你不得引入未授权变量，不得假设中性利率，不得讨论历史政策路径，不得越界到其他理论。"
                ),
            ),
            allowed_variables=(
                "inflation_current",
                "inflation_target_mid",
                "output_gap",
            ),
            history_access_policy=HistoryAccessPolicy.NONE,
            required_policy_history_keys=(),
            forbidden_logic_crossovers=(
                "不得把预期通胀或预期产出缺口写成主导机制。",
                "不得把政策利率惯性、平滑或历史路径写成主要依据。",
                "不得把汇率、资本流动或外部平衡压力写成核心机制。",
                "不得把金融周期稳定、资产价格或系统风险写成主要目标。",
                "不得用 DSGE 式跨期最优或福利最优化替代标准泰勒规则的当期国内稳定逻辑。",
            ),
        ),
        TheorySpec(
            theory_id="forward_looking_taylor",
            display_name="Forward-Looking Taylor Rule",
            prompt_modules=PromptModuleSpec(
                theory_identity=(
                    "你代表前瞻型泰勒规则理论。你的职责是依据前瞻性货币政策反应规则，"
                    "围绕未来通胀压力与未来需求压力，对当前决议时点的利率变动标签作出判断。"
                    "你不是最终决策层，不负责整合其他理论，也不负责根据系统历史表现调整自身立场。"
                ),
                policy_logic=(
                    "本理论的主判断逻辑固定为前瞻性稳定化反应规则。第一步，优先识别 `expected_inflation` 相对 `inflation_target_mid` 所反映的未来通胀压力；"
                    "第二步，识别 `expected_output_gap` 所反映的未来需求和活动压力；第三步，在承认货币政策传导存在时滞的前提下，"
                    "判断央行是否应在当前决议时点对未来压力进行预防性调整。"
                    "当未来通胀压力与未来需求压力方向一致时，应说明预防性反应如何被强化；当二者冲突时，应明确说明冲突来源，"
                    "并仍以前瞻型泰勒规则的逻辑给出结论。"
                ),
                theory_anchor_keys=(
                    "预期通胀优先",
                    "预期产出缺口优先",
                    "预防性政策反应",
                    "前瞻性稳定化",
                ),
                execution_protocol=(
                    "你只能使用当前样本中被授权暴露给本理论的字段。"
                    "主判断必须显式使用 `expected_inflation`、`inflation_target_mid` 和 `expected_output_gap`，并明确区分前瞻条件与当期条件。"
                    "`key_variables_used` 必须体现本次判断真正主导结论的前瞻变量。"
                    "你必须输出固定 9 字段结构，且 `recommendation` 必须与最高概率标签一致。"
                    "你不得退化为当期静态泰勒规则，不得越界到政策惯性、开放经济或 DSGE 结构最优化逻辑。"
                ),
            ),
            allowed_variables=(
                "expected_inflation",
                "inflation_target_mid",
                "expected_output_gap",
            ),
            history_access_policy=HistoryAccessPolicy.NONE,
            required_policy_history_keys=(),
            forbidden_logic_crossovers=(
                "当预期信号可用时，不得让当期通胀或当期活动指标压倒前瞻变量成为主导机制。",
                "不得把本理论写成利率平滑或政策惯性规则。",
                "不得把汇率机制、资本流动或金融稳定机制写成主要依据。",
                "不得用完整 DSGE 式结构最优或福利最优化替代前瞻型规则逻辑。",
            ),
        ),
        TheorySpec(
            theory_id="interest_rate_smoothing",
            display_name="Interest Rate Smoothing Rule",
            prompt_modules=PromptModuleSpec(
                theory_identity=(
                    "你代表利率平滑规则理论。你的职责是依据政策惯性、渐进调整和避免不必要大幅反转的逻辑，"
                    "对当前决议时点的利率变动标签作出判断。你不是最终决策层，不负责整合其他理论，也不负责根据系统历史表现调整自身立场。"
                ),
                policy_logic=(
                    "本理论的主判断逻辑固定为渐进调整与政策惯性。第一步，识别 `previous_policy_rate` 所代表的既有政策位置；"
                    "第二步，结合当前通胀与产出信号判断央行是否有必要继续维持、缓慢调整或谨慎改变既有政策方向；"
                    "第三步，从连续性、调整成本和预期管理角度判断本次利率变动步长是否应保持克制。"
                    "本理论允许通胀和产出信号影响判断，但这些信号必须通过“是否需要偏离既有路径、以及偏离幅度应多大”这一平滑逻辑进入结论，"
                    "不得把本理论退化成忽略既有利率位置的静态泰勒规则。"
                ),
                theory_anchor_keys=(
                    "政策惯性",
                    "渐进调整",
                    "连续性偏好",
                    "调整成本关注",
                ),
                execution_protocol=(
                    "你必须显式使用 `previous_policy_rate`，且它必须进入主解释而不是只出现在附带说明中。"
                    "你不能虚构当前框架未授权的更长历史路径，也不能把 `previous_policy_rate` 仅当作背景信息。"
                    "若使用通胀和产出变量，必须说明它们如何通过平滑逻辑影响步长判断。"
                    "`key_variables_used` 必须体现 `previous_policy_rate` 的主导地位。"
                    "你必须输出固定 9 字段结构，且 `recommendation` 必须与最高概率标签一致。"
                    "解释必须保持在利率平滑规则边界内，不得越界到前瞻型、开放经济、金融稳定或 DSGE 结构逻辑。"
                ),
            ),
            allowed_variables=(
                "previous_policy_rate",
                "inflation_current",
                "output_gap",
                "inflation_target_mid",
            ),
            history_access_policy=HistoryAccessPolicy.PREVIOUS_POLICY_RATE_ONLY,
            required_policy_history_keys=("previous_policy_rate",),
            forbidden_logic_crossovers=(
                "不得忽略 `previous_policy_rate`。",
                "不得把预期通胀或预期产出缺口写成主导机制。",
                "不得把汇率、资本流动或外部平衡约束写成中心机制。",
                "不得把金融稳定、信用周期或资产价格失衡写成主要理由。",
                "不得用 DSGE 式跨期最优或福利最优化替代渐进调整逻辑。",
            ),
        ),
        TheorySpec(
            theory_id="open_economy_rule",
            display_name="Open-Economy Rule",
            prompt_modules=PromptModuleSpec(
                theory_identity=(
                    "你代表开放经济规则理论。你的职责是依据汇率传导、输入型通胀和外部约束逻辑，"
                    "对当前决议时点的利率变动标签作出判断。你不是最终决策层，不负责整合其他理论，也不负责根据系统历史表现调整自身立场。"
                ),
                policy_logic=(
                    "本理论的主判断逻辑固定为外部约束修正规则。第一步，先基于 `inflation_current` 相对 `inflation_target_mid` 的偏离与产出状态形成一个初始政策倾向；"
                    "第二步，识别 `reer` 与 `import_price_yoy` 所反映的实际有效汇率压力和输入型通胀传导；"
                    "第三步，判断这些开放经济核心因素是否会通过汇率传导、进口通胀或外部约束改变原有国内判断。"
                    "当国内项与外部项一致时，应说明二者如何共同强化某一标签；当二者冲突时，应明确说明外部约束为何足以修正或不足以修正国内倾向。"
                ),
                theory_anchor_keys=(
                    "汇率敏感性",
                    "汇率传导约束",
                    "输入型通胀传导",
                    "开放经济约束修正",
                ),
                execution_protocol=(
                    "你只能使用当前样本中被授权暴露给本理论的字段。"
                    "你必须明确说明 `reer` 当前代表的是实际升值压力还是实际贬值压力，并说明 `import_price_yoy` 是否显示输入型通胀压力。"
                    "这些信号必须通过汇率传导、进口通胀或外部约束修正进入开放经济规则。"
                    "你不得把汇率变量降级为背景说明，也不得只复述国内通胀—产出逻辑。"
                    "`key_variables_used` 必须体现本次判断中真正起作用的开放经济变量。"
                    "你必须输出固定 9 字段结构，且 `recommendation` 必须与最高概率标签一致。"
                    "解释必须保持在开放经济规则边界内，不得越界到金融稳定、政策惯性或 DSGE 结构逻辑。"
                ),
            ),
            allowed_variables=(
                'reer',
                'import_price_yoy',
                'inflation_current',
                'inflation_target_mid',
                'output_gap',
            ),
            history_access_policy=HistoryAccessPolicy.NONE,
            required_policy_history_keys=(),
            forbidden_logic_crossovers=(
                "当样本中存在汇率和输入型通胀信号时，不得忽略这些机制。",
                "不得把本理论缩减为纯国内泰勒判断。",
                "不得用金融周期稳定、资产价格或信用失衡逻辑替代外部约束逻辑。",
                "不得把政策惯性或历史利率路径写成主导依据。",
                "不得把未经授权的资本流动叙述写成直接主变量依据，不得把本理论改写为 DSGE 式结构福利最优化解释。",
            ),
        ),
        TheorySpec(
            theory_id="financial_stability_rule",
            display_name="Financial Stability Rule",
            prompt_modules=PromptModuleSpec(
                theory_identity=(
                    "你代表金融稳定增强规则理论。你的职责是依据信用扩张、杠杆积累、资产价格脆弱性及其所指向的金融脆弱性风险逻辑，"
                    "判断当前利率决议是否应偏离单纯的通胀—产出稳定规则。你不是最终决策层，不负责整合其他理论，也不负责根据系统历史表现调整自身立场。"
                ),
                policy_logic=(
                    "本理论的主判断逻辑固定为金融失衡修正规则。第一步，先基于 `inflation_current` 相对 `inflation_target_mid` 的偏离与产出状态形成宏观稳定化倾向；"
                    "第二步，识别 `domestic_credit_yoy` 与 `credit_to_gdp_ratio` 所反映的信用扩张和杠杆积累，识别 `house_price_yoy` 与 `stock_price_yoy` 所反映的资产价格脆弱性，识别 `corporate_credit_spread` 所反映的风险溢价和金融压力；"
                    "第三步，判断这些金融稳定因素是否要求在当前样本中额外收紧、延后放松或调整步长。"
                    "你必须区分宏观稳定目标与金融稳定目标，并说明二者是一致还是冲突；若冲突，必须明确说明为何金融稳定考虑足以修正或不足以修正原有宏观倾向。"
                ),
                theory_anchor_keys=(
                    "信用周期关注",
                    "资产价格脆弱性",
                    "金融脆弱性代理",
                    "逆风而行取向",
                ),
                execution_protocol=(
                    "你只能使用当前样本中被授权暴露给本理论的字段。"
                    "你必须明确说明当前样本是否真的属于金融稳定相关情境，并区分信用/杠杆积累、资产价格脆弱性、风险溢价/金融压力三类机制。"
                    "你不得把金融变量当成噪声背景，也不得在金融失衡并不显著时硬写出强烈金融稳定结论。"
                    "`key_variables_used` 必须体现本次判断中真正起作用的金融变量。"
                    "你必须输出固定 9 字段结构，并保持 `recommendation`、`probability_distribution` 与 `confidence` 的一致性。"
                    "解释必须保持在金融稳定增强规则边界内，不得越界到开放经济、政策惯性或 DSGE 结构逻辑。"
                ),
            ),
            allowed_variables=(
                'domestic_credit_yoy',
                'credit_to_gdp_ratio',
                'house_price_yoy',
                'stock_price_yoy',
                'corporate_credit_spread',
                'inflation_current',
                'inflation_target_mid',
                'output_gap',
            ),
            history_access_policy=HistoryAccessPolicy.NONE,
            required_policy_history_keys=(),
            forbidden_logic_crossovers=(
                "当金融风险是主观察视角时，不得把标准通胀—产出稳定判断当成充分解释。",
                "不得用汇率或资本流动逻辑替代金融脆弱性逻辑。",
                "不得把本理论写成普通的利率惯性或平滑规则。",
                "不得用完整 DSGE 式跨期最优或福利最优化替代本理论。",
                "不得在金融失衡不显著时强行把每个样本都解释成金融稳定事件。",
            ),
        ),
        TheorySpec(
            theory_id="new_keynesian_optimal_rule",
            display_name="New Keynesian Optimal Rule",
            prompt_modules=PromptModuleSpec(
                theory_identity=(
                    "你代表新凯恩斯最优利率规则，具体是 history-dependent target-criterion optimal interest-rate rule。你的职责是依据跨期目标准则、预期传导、结构约束和历史依赖，"
                    "判断当前利率动作是否有助于未来通胀、产出/需求和政策路径满足最优目标准则。你不是最终决策层，不负责整合其他理论，也不负责根据系统历史表现调整自身立场。"

                ),
                policy_logic=(
                    "本理论的主判断逻辑固定为历史依赖的跨期目标准则判断。第一步，识别 `core_inflation` 和 `expected_inflation` 相对 `inflation_target_mid` 的目标偏离，以及 `output_gap` 和 `expected_output_gap` 所反映的当前与未来需求路径。"
                    "第二步，识别 `policy_rate_history_full` 与 `policy_path_diagnosis` 所代表的 history dependence、commitment 和 expectations management 约束；历史路径不是答案库，也不是趋势外推。"
                    "第三步，按 target criterion 比较加息、不变和降息三类候选动作：它们分别如何影响未来通胀路径、未来产出/需求路径、历史承诺和预期管理。"
                    "第四步，选择最符合跨期目标准则的固定7档标签。不得以“通胀高于目标 + 产出缺口为正”直接推出加息，也不得以历史上处于降息路径直接推出继续降息。"
                    "在 insurance_cut_cycle 或 easing_cycle_continuation 中，历史依赖和承诺管理通常先约束不要突然反转为加息；除非核心通胀、预期通胀和需求路径同时强烈再加速，否则高通胀信号更应解释为限制降息幅度或支持暂停，而不是直接反转政策方向。"

                ),
                theory_anchor_keys=(
                    "跨期一致性",
                    "结构均衡关注",
                    "预期传导",
                    "福利损失最小化",
                ),
                execution_protocol=(
                    "当前框架明确授权你读取 `policy_rate_history_full` 与 `policy_path_diagnosis`，且你必须把它作为 history dependence、commitment 和 expectations management 的正式输入，而不是可有可无的背景信息。"
                    "你必须同时覆盖目标变量偏离、预期路径和历史依赖约束，不能只抓住其中某一类变量就给出结论。"
                    "你必须明确比较加息、不变和降息三类候选动作对未来通胀路径、未来产出/需求路径、历史承诺和预期管理的相对影响。"
                    "历史路径不是用来修正一个预先给定的静态 Taylor 方向，而是作为最优目标准则中的历史依赖、承诺和预期管理约束。"
                    "若 policy_path_diagnosis 显示降息延续或保险式降息阶段，必须比较继续小幅降息、暂停、反转加息三者；反转加息需要比限制降息幅度更强的跨期目标准则证据。"
                    "你不得声称自己在字面意义上求解了完整 DSGE 模型，但必须以结构性、跨期性和目标准则的解释方式组织判断。"
                    "`key_variables_used` 必须体现本次判断中真正起作用的当前项、预期项和路径变量。"
                    "你必须输出固定 9 字段基础结构，且 `recommendation` 必须与最高概率标签一致。"
                    "解释必须保持在新凯恩斯最优利率规则边界内，不得退化为单期静态 Taylor 规则或简单路径外推。"

                ),
            ),
            allowed_variables=(
                "core_inflation",
                "inflation_target_mid",
                "expected_inflation",
                "output_gap",
                "expected_output_gap",
                "policy_rate_history_full",
                "policy_path_diagnosis",
            ),
            history_access_policy=HistoryAccessPolicy.FULL_DECISION_RATE_HISTORY,
            required_policy_history_keys=("policy_rate_history_full", "policy_path_diagnosis"),
            forbidden_logic_crossovers=(
                "不得把本理论缩减为单期 Taylor 判断或前瞻 Taylor 判断。",
                "不得以“通胀高于目标 + 产出缺口为正”直接推出加息；必须说明该动作相对不变/降息为何更符合跨期目标准则。",
                "不得让历史利率路径单独替代跨期目标准则，也不得完全忽略路径信息。",
                "不得把历史路径当作趋势外推；连续降息不自动意味继续降息，连续加息也不自动意味继续加息。",
                "不得把汇率机制或金融风险机制写成唯一主导逻辑。",
                "不得声称自己给出了字面意义上的 DSGE 求解结果；本理论只是一层结构性近似解释。",

            ),
        ),

        TheorySpec(
            theory_id="balanced_approach_rule",
            display_name="balanced-approach rule",
            prompt_modules=PromptModuleSpec(
                theory_identity=(
                    "你代表 Federal Reserve Monetary Policy Report 中的 `balanced-approach rule`。"
                    "你的职责是严格依据该规则的货币政策反应函数判断当前决议时点的利率变动标签。"
                    "你不是标准泰勒规则，不是前瞻型规则，不是最终决策层，也不根据系统历史表现调整自身立场。"
                ),
                policy_logic=(
                    "本理论的主判断逻辑固定为 Fed Monetary Policy Report 列出的 balanced-approach rule："
                    "R_t^BA = r_t^LR + pi_t + 0.5(pi_t - pi^LR) + 2(u_t^LR - u_t)。"
                    "第一步，识别 `inflation_current` 相对 `inflation_target_mid` 的通胀缺口项；"
                    "第二步，识别 `longer_run_unemployment_rate` 与 `unemployment_rate` 形成的 `unemployment_gap = u_t^LR - u_t`；"
                    "第三步，按公式符号理解两个反应项。若 u_t > u_t^LR，则 u_t^LR - u_t < 0，就业侧支持更低政策利率；"
                    "若 u_t < u_t^LR，则 u_t^LR - u_t > 0，劳动力市场偏紧支持更高政策利率。"
                    "第四步，将规则隐含的通胀项和就业项方向压力转换为标签判断，不得把政策惯性写成该理论的核心机制。"
                    "当 u_t > u_t^LR 且通胀缺口并非极端失控时，balanced-approach rule 的就业项不是普通背景变量，而是双重使命下的方向性宽松压力；必须在解释中说明它如何抵消或压过通胀项。"
                ),
                theory_anchor_keys=(
                    "balanced-approach rule",
                    "R_t^BA = r_t^LR + pi_t + 0.5(pi_t - pi^LR) + 2(u_t^LR - u_t)",
                    "inflation gap",
                    "unemployment gap",
                    "maximum employment statutory goal",
                ),
                execution_protocol=(
                    "你只能使用当前样本中被授权暴露给本理论的字段。"
                    "主判断必须显式使用 `inflation_current`、`inflation_target_mid`、`unemployment_rate`、`longer_run_unemployment_rate` 和 `unemployment_gap`。"
                    "`unemployment_gap` 的方向必须按 Fed 公式中的 `u_t^LR - u_t` 解释，不得写反。"
                    "`key_variables_used` 必须包含通胀缺口和失业率缺口的核心变量。"
                    "若 unemployment_rate 高于 longer_run_unemployment_rate，必须明确写出这是就业风险/劳动力市场松弛信号；不得仅因为该理论是少数派或缺口绝对值较小就把方向信号写成无关紧要。"
                    "你必须输出固定 9 字段结构，且 `recommendation` 必须与最高概率标签一致。"
                ),
            ),
            allowed_variables=(
                'inflation_current',
                'inflation_target_mid',
                'unemployment_rate',
                'longer_run_unemployment_rate',
                'unemployment_gap',
            ),
            history_access_policy=HistoryAccessPolicy.NONE,
            required_policy_history_keys=(),
            forbidden_logic_crossovers=(
                "不得把本理论写成产出缺口版的标准泰勒规则。",
                "不得把本理论写成产出缺口版的标准泰勒规则。",
                "不得把政策路径惯性、利率平滑或历史路径写成主要机制。",
                "不得把汇率、金融稳定、风险管理或 DSGE 最优控制写成本理论的核心逻辑。",
            ),
        ),
    )

    THEORY_SPECS_BY_ID: dict[str, TheorySpec] = {spec.theory_id: spec for spec in THEORY_SPECS}

    def get_theory_spec(theory_id: str) -> TheorySpec:
        try:
            return THEORY_SPECS_BY_ID[theory_id]
        except KeyError as exc:
            available = ", ".join(sorted(THEORY_SPECS_BY_ID))
            raise KeyError(f"Unknown theory_id {theory_id!r}. Available theory ids: {available}.") from exc


In [ ]:
# 文件：theory_specs/__init__.py
# 产出：导出理论规格接口。

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    __all__ = [
        "HistoryAccessPolicy",
        "PromptModuleSpec",
        "THEORY_SPECS",
        "THEORY_SPECS_BY_ID",
        "TheorySpec",
        "get_theory_spec",
    ]


In [ ]:
# 文件：sample_builder/xlsx_reader.py
# 产出：产出 Excel 原始数据读取工具。
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from dataclasses import dataclass
    from datetime import date, datetime, timedelta
    from pathlib import Path
    import re
    import zipfile
    import xml.etree.ElementTree as ET

    MAIN_NS = "http://schemas.openxmlformats.org/spreadsheetml/2006/main"
    REL_NS = "http://schemas.openxmlformats.org/package/2006/relationships"
    DOC_REL_NS = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"
    NS = {"main": MAIN_NS, "rel": REL_NS}

    DATE_NUMFMT_IDS = {
        14,
        15,
        16,
        17,
        22,
        27,
        30,
        36,
        45,
        46,
        47,
        50,
        57,
        176,
    }

    @dataclass(frozen=True, slots=True)
    class WorksheetRows:
        sheet_name: str
        rows: list[list[str]]

    class XlsxSheetReader:
        def __init__(self, workbook_path: Path) -> None:
            self.workbook_path = workbook_path

        def read_first_sheet_rows(self) -> WorksheetRows:
            with zipfile.ZipFile(self.workbook_path) as zf:
                shared_strings = self._read_shared_strings(zf)
                style_numfmts = self._read_style_numfmts(zf)
                sheet_name, sheet_path = self._resolve_first_sheet(zf)
                rows = self._read_sheet_rows(zf, sheet_path, shared_strings, style_numfmts)
            return WorksheetRows(sheet_name=sheet_name, rows=rows)

        @staticmethod
        def _read_shared_strings(zf: zipfile.ZipFile) -> list[str]:
            if "xl/sharedStrings.xml" not in zf.namelist():
                return []
            root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
            strings: list[str] = []
            for item in root.findall("main:si", NS):
                parts = [node.text or "" for node in item.findall(".//main:t", NS)]
                strings.append("".join(parts))
            return strings

        @staticmethod
        def _read_style_numfmts(zf: zipfile.ZipFile) -> dict[int, int]:
            if "xl/styles.xml" not in zf.namelist():
                return {}
            root = ET.fromstring(zf.read("xl/styles.xml"))
            cell_xfs = root.find("main:cellXfs", NS)
            if cell_xfs is None:
                return {}
            mapping: dict[int, int] = {}
            for idx, xf in enumerate(cell_xfs.findall("main:xf", NS)):
                mapping[idx] = int(xf.attrib.get("numFmtId", "0"))
            return mapping

        @staticmethod
        def _resolve_first_sheet(zf: zipfile.ZipFile) -> tuple[str, str]:
            workbook = ET.fromstring(zf.read("xl/workbook.xml"))
            rels = ET.fromstring(zf.read("xl/_rels/workbook.xml.rels"))
            rel_map = {
                rel.attrib["Id"]: rel.attrib["Target"]
                for rel in rels.findall(f"{{{REL_NS}}}Relationship")
            }
            first_sheet = workbook.find("main:sheets/main:sheet", NS)
            if first_sheet is None:
                raise ValueError(f"No worksheet found in workbook: {zf.filename}")
            rel_id = first_sheet.attrib[f"{{{DOC_REL_NS}}}id"]
            target = rel_map[rel_id]
            sheet_path = "xl/" + target if target.startswith("worksheets/") else "xl/worksheets/" + target
            return first_sheet.attrib.get("name", "Sheet1"), sheet_path

        def _read_sheet_rows(
            self,
            zf: zipfile.ZipFile,
            sheet_path: str,
            shared_strings: list[str],
            style_numfmts: dict[int, int],
        ) -> list[list[str]]:
            root = ET.fromstring(zf.read(sheet_path))
            rows = root.findall(".//main:sheetData/main:row", NS)
            parsed_rows: list[list[str]] = []

            for row in rows:
                values: list[str] = []
                for cell in row.findall("main:c", NS):
                    ref = cell.attrib.get("r", "")
                    col_index = self._column_index_from_ref(ref)
                    while len(values) <= col_index:
                        values.append("")
                    values[col_index] = self._read_cell(cell, shared_strings, style_numfmts)
                if values:
                    parsed_rows.append(values)
            return parsed_rows

        @staticmethod
        def _column_index_from_ref(ref: str) -> int:
            match = re.match(r"([A-Z]+)", ref)
            if match is None:
                return 0
            letters = match.group(1)
            index = 0
            for char in letters:
                index = index * 26 + (ord(char) - ord("A") + 1)
            return index - 1

        def _read_cell(
            self,
            cell: ET.Element,
            shared_strings: list[str],
            style_numfmts: dict[int, int],
        ) -> str:
            cell_type = cell.attrib.get("t")
            value_node = cell.find("main:v", NS)
            raw_value = "" if value_node is None or value_node.text is None else value_node.text

            if cell_type == "s":
                return shared_strings[int(raw_value)] if raw_value else ""
            if cell_type == "inlineStr":
                texts = [node.text or "" for node in cell.findall(".//main:t", NS)]
                return "".join(texts)
            if not raw_value:
                return ""

            style_id = int(cell.attrib.get("s", "0"))
            numfmt_id = style_numfmts.get(style_id, 0)
            if numfmt_id in DATE_NUMFMT_IDS:
                return self._excel_date_to_str(raw_value)
            return raw_value

        @staticmethod
        def _excel_date_to_str(raw: str) -> str:
            number = float(raw)
            base = datetime(1899, 12, 30)
            return (base + timedelta(days=number)).strftime("%Y-%m-%d")

    def parse_iso_date(value: str) -> date:
        return datetime.strptime(value, "%Y-%m-%d").date()


In [ ]:
# 文件：sample_builder/hp_filter.py
# 产出：产出单向 HP 滤波与趋势缺口计算。
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from typing import Iterable

    import numpy as np

    def hp_filter_trend(y: np.ndarray, lamb: float) -> np.ndarray:
        n = len(y)
        if n < 3:
            return y.copy()

        eye = np.eye(n)
        diff = np.zeros((n - 2, n))
        for i in range(n - 2):
            diff[i, i] = 1.0
            diff[i, i + 1] = -2.0
            diff[i, i + 2] = 1.0
        system = eye + lamb * (diff.T @ diff)
        return np.linalg.solve(system, y)

    def compute_one_sided_hp_gap(
        level_series: Iterable[tuple[str, float | None]], lamb: float = 129600.0
    ) -> dict[str, float | None]:
        output: dict[str, float | None] = {}
        history: list[float] = []

        for date_key, value in level_series:
            if value is None:
                output[date_key] = None
                continue

            history.append(float(value))
            if len(history) < 6:
                output[date_key] = None
                continue

            trend = hp_filter_trend(np.array(history, dtype=float), lamb=lamb)
            actual = history[-1]
            potential = float(trend[-1])
            if potential == 0.0:
                output[date_key] = None
            else:
                output[date_key] = ((actual - potential) / potential) * 100.0

        return output


In [ ]:
# 文件：sample_builder/us_builder.py
# 产出：产出美国样本构建器与美国数据样本切片逻辑。
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from dataclasses import dataclass
    from datetime import date, datetime, timedelta
    from pathlib import Path
    from typing import Any
    import zipfile
    import xml.etree.ElementTree as ET

    def _resolve_project_root() -> Path:
        if "__file__" in globals():
            return Path(__file__).resolve().parents[2]
        if "project_root" in globals():
            return Path(project_root).resolve()

        current_dir = Path.cwd().resolve()
        if (current_dir / "多智能体协同框架").exists():
            return current_dir
        if current_dir.name == "多智能体协同框架":
            return current_dir.parent
        return current_dir

    PROJECT_ROOT = _resolve_project_root()
    COUNTRY_CODE = "US"
    COUNTRY_NAME = "US"
    TRAIN_TEST_SPLIT_DATE = date(2015, 1, 1)
    US_DECISION_FILE_NAME = "美联储利率决议.xlsx"
    MASTER_FILE_NAME = "美国数据.xlsx"
    US_DECISION_DIRECTORY_NAME = "各国利率决议时间数据"

    US_HEADER_MAP = {
        'inflation': '个人消费支出价格指数：PCEPI：同比：月度：SA：美国',
        'core_inflation': '核心个人消费支出价格指数：PCEPILFE：同比：月度：SA：美国',
        'output_gap': '产出缺口：GDPC1-GDPPOT占GDPPOT百分比：季度：美国',
        'unemployment_rate': '失业率：月度：SA：美国',
        'longer_run_unemployment_rate': '自然失业率：美国',
        'unemployment_gap': '失业率缺口：自然失业率-失业率：百分点：月度：美国',
        'reer': '实际有效汇率指数：BIS：Narrow：Real：2020=100：月度：NSA：美国',
        'domestic_credit_yoy': '总信用：私营非金融部门：同比：季度：NSA：美国',
        'credit_to_gdp_ratio': '总信用：私营非金融部门：占GDP百分比：季度：NSA：美国',
        'import_price_yoy': '进口价格指数：全部进口：同比：月度：NSA：美国',
        'house_price_yoy': 'FHFA房价指数：全交易：同比：季度：NSA：美国',
        'stock_price_yoy': '股票价格指数：全部股票：广义：同比：月度：NSA：美国',
        'corporate_credit_spread': '企业信用利差：穆迪评级Baa级-Aaa级：月度：NSA：美国',
        'inflation_target_mid': '目标通货膨胀率：理论参数：美国',
        'expected_inflation': '密歇根大学消费者调查：一年期通胀预期：月度：NSA：美国',
        'expected_output_gap': '预测产出缺口：占潜在GDP百分比：美国',
    }

    MAIN_NS = "http://schemas.openxmlformats.org/spreadsheetml/2006/main"
    REL_NS = "http://schemas.openxmlformats.org/package/2006/relationships"
    DOC_REL_NS = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"
    XML_NS = {"main": MAIN_NS}

    REQUIRED_SNAPSHOT_FIELDS = (
        'inflation_current',
        'core_inflation',
        'output_gap',
        'unemployment_rate',
        'longer_run_unemployment_rate',
        'unemployment_gap',
        'reer',
        'domestic_credit_yoy',
        'import_price_yoy',
        'credit_to_gdp_ratio',
        'house_price_yoy',
        'stock_price_yoy',
        'corporate_credit_spread',
        'inflation_target_mid',
    )

    REQUIRED_EXPECTATION_FIELDS = (
        "expected_inflation",
        "expected_output_gap",
    )

    REQUIRED_POLICY_INPUT_FIELDS = (
        'previous_policy_rate',
        'policy_rate_history_full',
        'policy_path_diagnosis',
    )

    @dataclass(frozen=True, slots=True)
    class USDecisionRecord:
        decision_date: date
        observed_policy_rate: float

    class USSampleBuilder:
        def __init__(
            self,
            project_root: Path | None = None,
            hp_lambda: float = 129600.0,
            decision_cutoff_date: date | None = None,
        ) -> None:
            self.project_root = project_root or PROJECT_ROOT
            self.hp_lambda = hp_lambda
            if decision_cutoff_date is not None and not isinstance(decision_cutoff_date, date):
                raise TypeError("decision_cutoff_date must be a datetime.date instance or None.")
            self.decision_cutoff_date = decision_cutoff_date
            self.decision_file_path = self._find_us_decision_file()
            self.master_file_path = self._find_master_file()
            self._decision_records: list[USDecisionRecord] | None = None
            self._monthly_series_map: dict[str, dict[str, float | None]] | None = None
            self._longer_run_unemployment_map: dict[str, float] | None = None
            self._first_runnable_decision_index: int | None = None

        def list_decision_dates(self) -> list[date]:
            return [record.decision_date for record in self._load_decision_records()]

        def list_runnable_decision_dates(self) -> list[date]:
            start_index = self._first_runnable_index()
            return [record.decision_date for record in self._load_decision_records()[start_index:]]

        def list_training_runnable_decision_dates(self) -> list[date]:
            return [decision_date for decision_date in self.list_runnable_decision_dates() if decision_date < TRAIN_TEST_SPLIT_DATE]

        def list_test_runnable_decision_dates(self) -> list[date]:
            return [decision_date for decision_date in self.list_runnable_decision_dates() if decision_date >= TRAIN_TEST_SPLIT_DATE]

        @staticmethod
        def is_training_period(decision_date: date) -> bool:
            return decision_date < TRAIN_TEST_SPLIT_DATE

        @staticmethod
        def is_test_period(decision_date: date) -> bool:
            return decision_date >= TRAIN_TEST_SPLIT_DATE

        def first_runnable_decision_date(self) -> date:
            index = self._first_runnable_index()
            return self._load_decision_records()[index].decision_date

        def build_training_sample(self, decision_date: date) -> TrainingDecisionSample:
            record, record_index = self._get_record_and_index(decision_date)
            self._ensure_runnable_record(record_index)
            if not self.is_training_period(record.decision_date):
                raise ValueError(
                    f"Decision {record.decision_date.isoformat()} is outside the training period. "
                    f"Training samples must be before {TRAIN_TEST_SPLIT_DATE.isoformat()}."
                )

            payload = self._build_training_sample_payload(record_index)
            return TrainingDecisionSample(
                country=COUNTRY_NAME,
                decision_date=record.decision_date,
                current_macro_snapshot=payload["current_macro_snapshot"],
                market_expectations=payload["market_expectations"],
                policy_inputs=payload["policy_inputs"],
                textual_inputs=payload["textual_inputs"],
                realized_label=payload["realized_label"],
            )

        def build_prediction_sample(self, decision_date: date) -> PredictionDecisionSample:
            record, record_index = self._get_record_and_index(decision_date)
            self._ensure_runnable_record(record_index)
            if not self.is_test_period(record.decision_date):
                raise ValueError(
                    f"Decision {record.decision_date.isoformat()} is outside the prediction/test period. "
                    f"Prediction samples must be on or after {TRAIN_TEST_SPLIT_DATE.isoformat()}."
                )

            payload = self._build_prediction_sample_payload(record_index)
            return PredictionDecisionSample(
                country=COUNTRY_NAME,
                decision_date=record.decision_date,
                current_macro_snapshot=payload["current_macro_snapshot"],
                market_expectations=payload["market_expectations"],
                policy_inputs=payload["policy_inputs"],
                textual_inputs=payload["textual_inputs"],
            )

        def build_actual_label(self, decision_date: date) -> RateMoveLabel:
            record, record_index = self._get_record_and_index(decision_date)
            self._ensure_runnable_record(record_index)
            rate_change_bp = self._rate_change_bp(record_index)
            if rate_change_bp is None:
                raise ValueError(
                    f"Missing previous policy rate for decision {record.decision_date.isoformat()}; "
                    "cannot derive actual label."
                )
            return self._map_rate_change_to_label(rate_change_bp)

        def _build_shared_sample_payload(self, record_index: int) -> dict[str, Any]:
            records = self._load_decision_records()
            record = records[record_index]
            panel_date = self._panel_date_from_decision_date(record.decision_date)
            series_map = self._load_monthly_series_map()
            longer_run_unemployment_rate = self._safe_get_latest_available(
                series_map,
                "longer_run_unemployment_rate",
                panel_date,
                max_staleness_months=6,
            )
            unemployment_rate = self._safe_get(series_map, "unemployment_rate", panel_date)
            unemployment_gap = None if unemployment_rate is None or longer_run_unemployment_rate is None else longer_run_unemployment_rate - unemployment_rate

            previous_policy_rate = records[record_index - 1].observed_policy_rate if record_index > 0 else None

            current_macro_snapshot = {
                "inflation_current": self._safe_get(series_map, "inflation", panel_date),
                "core_inflation": self._safe_get(series_map, "core_inflation", panel_date),
                "output_gap": self._safe_get_latest_available(series_map, "output_gap", panel_date, max_staleness_months=6),
                "unemployment_rate": unemployment_rate,
                "longer_run_unemployment_rate": longer_run_unemployment_rate,
                "unemployment_gap": unemployment_gap,
                "reer": self._safe_get(series_map, "reer", panel_date),
                "domestic_credit_yoy": self._safe_get_latest_available(series_map, "domestic_credit_yoy", panel_date, max_staleness_months=12),
                "import_price_yoy": self._safe_get_latest_available(series_map, "import_price_yoy", panel_date, max_staleness_months=6),
                "credit_to_gdp_ratio": self._safe_get_latest_available(series_map, "credit_to_gdp_ratio", panel_date, max_staleness_months=12),
                "house_price_yoy": self._safe_get_latest_available(series_map, "house_price_yoy", panel_date, max_staleness_months=6),
                "stock_price_yoy": self._safe_get_latest_available(series_map, "stock_price_yoy", panel_date, max_staleness_months=6),
                "corporate_credit_spread": self._safe_get_latest_available(series_map, "corporate_credit_spread", panel_date, max_staleness_months=6),
                "inflation_target_mid": self._safe_get(series_map, "inflation_target_mid", panel_date),
            }

            market_expectations = {
                "expected_inflation": self._safe_get(series_map, "expected_inflation", panel_date),
                "expected_output_gap": self._safe_get(series_map, "expected_output_gap", panel_date),
            }

            policy_rate_history_full = self._policy_rate_history_before(record_index)
            policy_inputs = {
                "previous_policy_rate": previous_policy_rate,
                "policy_rate_history_full": policy_rate_history_full,
                "policy_path_diagnosis": self._build_policy_path_diagnosis(policy_rate_history_full),
            }

            self._assert_no_missing_values(
                decision_date=record.decision_date,
                current_macro_snapshot=current_macro_snapshot,
                market_expectations=market_expectations,
                policy_inputs=policy_inputs,
            )

            return {
                "current_macro_snapshot": current_macro_snapshot,
                "market_expectations": market_expectations,
                "policy_inputs": policy_inputs,
                "textual_inputs": {
                    "country_code": COUNTRY_CODE,
                    "panel_date": panel_date,
                    "train_test_split_date": TRAIN_TEST_SPLIT_DATE.isoformat(),
                },
            }

        def _build_training_sample_payload(self, record_index: int) -> dict[str, Any]:
            shared_payload = self._build_shared_sample_payload(record_index)
            record = self._load_decision_records()[record_index]
            rate_change_bp = self._rate_change_bp(record_index)

            if rate_change_bp is None:
                raise ValueError(
                    f"Missing previous policy rate for decision {record.decision_date.isoformat()}; "
                    "cannot derive training label."
                )

            shared_payload["realized_label"] = self._map_rate_change_to_label(rate_change_bp)
            return shared_payload

        def _build_prediction_sample_payload(self, record_index: int) -> dict[str, Any]:
            return self._build_shared_sample_payload(record_index)

        def _ensure_runnable_record(self, record_index: int) -> None:
            first_index = self._first_runnable_index()
            if record_index < first_index:
                first_date = self._load_decision_records()[first_index].decision_date.isoformat()
                current_date = self._load_decision_records()[record_index].decision_date.isoformat()
                raise ValueError(
                    f"Decision {current_date} is earlier than the first runnable US sample {first_date}. "
                    "The framework requires starting from the first time point where all required indicators exist."
                )

        def _first_runnable_index(self) -> int:
            if self._first_runnable_decision_index is not None:
                return self._first_runnable_decision_index

            records = self._load_decision_records()
            first_missing_error: ValueError | None = None
            for idx in range(1, len(records)):
                try:
                    self._build_shared_sample_payload(idx)
                except ValueError as exc:
                    if "Missing required US sample fields for decision" not in str(exc):
                        raise
                    if first_missing_error is None:
                        first_missing_error = exc
                    continue
                self._first_runnable_decision_index = idx
                return idx

            if first_missing_error is not None:
                raise ValueError("No runnable US decision sample found.") from first_missing_error
            raise ValueError("US decision record list is empty or too short to build runnable samples.")

        def _find_us_decision_file(self) -> Path:
            for directory in self.project_root.iterdir():
                if not directory.is_dir():
                    continue
                if directory.name != US_DECISION_DIRECTORY_NAME:
                    continue
                for file_path in directory.glob("*.xlsx"):
                    if file_path.name == US_DECISION_FILE_NAME or "美联储" in file_path.name:
                        return file_path
            raise FileNotFoundError(f"Could not locate {US_DECISION_FILE_NAME} under project root.")

        def _find_master_file(self) -> Path:
            candidate = self.project_root / MASTER_FILE_NAME
            if candidate.exists():
                return candidate
            raise FileNotFoundError(f"Could not locate {MASTER_FILE_NAME} under project root.")

        def _load_decision_records(self) -> list[USDecisionRecord]:
            if self._decision_records is not None:
                return self._decision_records

            records: list[USDecisionRecord] = []
            with zipfile.ZipFile(self.decision_file_path) as zf:
                rows = self._read_decision_sheet_rows(zf)
                for row in rows:
                    raw_date = row.get("B", "").strip()
                    raw_rate = row.get("D", "").strip() or row.get("C", "").strip()
                    if not raw_date or not raw_rate:
                        continue

                    record = USDecisionRecord(
                        decision_date=self._parse_us_decision_date(raw_date),
                        observed_policy_rate=float(raw_rate),
                    )
                    if self.decision_cutoff_date is not None and record.decision_date > self.decision_cutoff_date:
                        continue
                    records.append(record)

            self._decision_records = sorted(records, key=lambda item: item.decision_date)
            return self._decision_records

        def _read_decision_sheet_rows(self, zf: zipfile.ZipFile) -> list[dict[str, str]]:
            workbook = ET.fromstring(zf.read("xl/workbook.xml"))
            rels = ET.fromstring(zf.read("xl/_rels/workbook.xml.rels"))
            rel_map = {
                rel.attrib["Id"]: rel.attrib["Target"]
                for rel in rels.findall(f"{{{REL_NS}}}Relationship")
            }
            first_sheet = workbook.find("main:sheets/main:sheet", XML_NS)
            if first_sheet is None:
                raise ValueError("No worksheet found in US decision workbook.")

            rel_id = first_sheet.attrib[f"{{{DOC_REL_NS}}}id"]
            target = rel_map[rel_id]
            sheet_path = "xl/" + target if target.startswith("worksheets/") else "xl/worksheets/" + target
            sheet_root = ET.fromstring(zf.read(sheet_path))

            shared_strings = self._read_shared_strings(zf)
            parsed_rows: list[dict[str, str]] = []
            for row in sheet_root.findall(".//main:sheetData/main:row", XML_NS)[1:]:
                row_map: dict[str, str] = {}
                for cell in row.findall("main:c", XML_NS):
                    ref = cell.attrib.get("r", "")
                    column = "".join(char for char in ref if char.isalpha())
                    value = cell.findtext("main:v", default="", namespaces=XML_NS)
                    if cell.attrib.get("t") == "s" and value:
                        value = shared_strings[int(value)]
                    row_map[column] = value.strip()
                if row_map:
                    parsed_rows.append(row_map)
            return parsed_rows

        @staticmethod
        def _read_shared_strings(zf: zipfile.ZipFile) -> list[str]:
            if "xl/sharedStrings.xml" not in zf.namelist():
                return []
            root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
            strings: list[str] = []
            for item in root.findall("main:si", XML_NS):
                parts = [node.text or "" for node in item.findall(".//main:t", XML_NS)]
                strings.append("".join(parts))
            return strings

        @staticmethod
        def _excel_serial_to_date(raw: str) -> date:
            serial = float(raw)
            base = datetime(1899, 12, 30)
            return (base + timedelta(days=serial)).date()

        @classmethod
        def _parse_us_decision_date(cls, raw: str) -> date:
            text = raw.strip()
            if "/" in text:
                left, right = [part.strip() for part in text.split("/", 1)]
                if len(left) == 4:
                    return date(int(left), int(right), 1)
                if len(right) == 4:
                    return date(int(right), int(left), 1)
            return cls._excel_serial_to_date(text)

        def _load_monthly_series_map(self) -> dict[str, dict[str, float | None]]:
            if self._monthly_series_map is not None:
                return self._monthly_series_map

            rows = XlsxSheetReader(self.master_file_path).read_first_sheet_rows().rows
            if len(rows) < 3:
                raise ValueError("Master workbook does not contain enough rows.")

            headers = rows[0]
            data_rows = rows[2:]
            header_index = {header: idx for idx, header in enumerate(headers) if header}

            out: dict[str, dict[str, float | None]] = {field_name: {} for field_name in US_HEADER_MAP}
            date_col_idx = header_index.get("date", 0)

            for row in data_rows:
                if date_col_idx >= len(row):
                    continue
                raw_panel_date = row[date_col_idx].strip()
                if not raw_panel_date:
                    continue
                panel_date = self._parse_panel_date(raw_panel_date).strftime("%Y-%m-%d")

                for field_name, header_text in US_HEADER_MAP.items():
                    idx = header_index.get(header_text)
                    if idx is None:
                        raise ValueError(f"Master workbook is missing required US header: {header_text}")
                    raw_value = row[idx].strip() if idx < len(row) else ""
                    out[field_name][panel_date] = float(raw_value) if raw_value else None

            self._monthly_series_map = out
            return out

        def _get_record_and_index(self, decision_date: date) -> tuple[USDecisionRecord, int]:
            records = self._load_decision_records()
            for idx, record in enumerate(records):
                if record.decision_date == decision_date:
                    return record, idx
            raise KeyError(f"Decision date {decision_date.isoformat()} not found in US decision calendar.")

        @staticmethod
        def _panel_date_from_decision_date(decision_date: date) -> str:
            return f"{decision_date.year:04d}-{decision_date.month:02d}-01"

        @classmethod
        def _parse_panel_date(cls, raw: str) -> date:
            text = raw.strip()
            if "/" in text:
                left, right = [part.strip() for part in text.split("/", 1)]
                if len(left) == 4:
                    return date(int(left), int(right), 1)
                if len(right) == 4:
                    return date(int(right), int(left), 1)
            return parse_iso_date(text)

        @classmethod
        def _parse_panel_date(cls, raw: str) -> date:
            text = raw.strip()
            if "/" in text:
                left, right = [part.strip() for part in text.split("/", 1)]
                if len(left) == 4:
                    return date(int(left), int(right), 1)
                if len(right) == 4:
                    return date(int(right), int(left), 1)
            return parse_iso_date(text)

        @classmethod
        def _parse_panel_date(cls, raw: str) -> date:
            text = raw.strip()
            if "/" in text:
                left, right = [part.strip() for part in text.split("/", 1)]
                if len(left) == 4:
                    return date(int(left), int(right), 1)
                if len(right) == 4:
                    return date(int(right), int(left), 1)
            return parse_iso_date(text)

        @classmethod
        def _parse_panel_date(cls, raw: str) -> date:
            text = raw.strip()
            if "/" in text:
                left, right = [part.strip() for part in text.split("/", 1)]
                if len(left) == 4:
                    return date(int(left), int(right), 1)
                if len(right) == 4:
                    return date(int(right), int(left), 1)
            return parse_iso_date(text)

        @staticmethod
        def _safe_get(
            series_map: dict[str, dict[str, float | None]],
            field_name: str,
            panel_date: str,
        ) -> float | None:
            return series_map.get(field_name, {}).get(panel_date)

        @staticmethod
        def _safe_get_latest_available(
            series_map: dict[str, dict[str, float | None]],
            field_name: str,
            panel_date: str,
            *,
            max_staleness_months: int,
        ) -> float | None:
            exact_value = USSampleBuilder._safe_get(series_map, field_name, panel_date)
            if exact_value is not None:
                return exact_value

            panel_dt = parse_iso_date(panel_date)
            candidates: list[tuple[date, float]] = []
            for candidate_date_text, candidate_value in series_map.get(field_name, {}).items():
                if candidate_value is None:
                    continue
                candidate_dt = parse_iso_date(candidate_date_text)
                if candidate_dt <= panel_dt:
                    candidates.append((candidate_dt, float(candidate_value)))
            if not candidates:
                return None

            latest_date, latest_value = max(candidates, key=lambda item: item[0])
            month_gap = (panel_dt.year - latest_date.year) * 12 + (panel_dt.month - latest_date.month)
            if month_gap > max_staleness_months:
                return None
            return latest_value

        @staticmethod
        def _compute_year_over_year(
            series: dict[str, float | None],
            panel_date: str,
        ) -> float | None:
            current_value = series.get(panel_date)
            if current_value is None or current_value == 0:
                return None

            panel_dt = datetime.strptime(panel_date, "%Y-%m-%d").date()
            previous_year_date = f"{panel_dt.year - 1:04d}-{panel_dt.month:02d}-01"
            previous_value = series.get(previous_year_date)
            if previous_value is None or previous_value == 0:
                return None

            return ((current_value / previous_value) - 1.0) * 100.0

        def _rate_change_bp(self, record_index: int) -> int | None:
            if record_index == 0:
                return None
            records = self._load_decision_records()
            delta = records[record_index].observed_policy_rate - records[record_index - 1].observed_policy_rate
            return int(round(delta * 100.0))

        def _policy_rate_history_before(self, record_index: int) -> list[dict[str, Any]]:
            history: list[dict[str, Any]] = []
            for record in self._load_decision_records()[:record_index]:
                history.append(
                    {
                        "decision_date": record.decision_date.isoformat(),
                        "policy_rate": record.observed_policy_rate,
                    }
                )
            return history
        @staticmethod
        def _rate_change_label_from_bp(rate_change_bp: int) -> str:
            if rate_change_bp < -50:
                return "<-50bp"
            if -50 <= rate_change_bp < -25:
                return "[-50,-25)bp"
            if -25 <= rate_change_bp < 0:
                return "[-25,0)bp"
            if rate_change_bp == 0:
                return "0bp"
            if 0 < rate_change_bp <= 25:
                return "(0,25]bp"
            if 25 < rate_change_bp <= 50:
                return "(25,50]bp"
            return ">50bp"

        @classmethod
        def _build_policy_path_diagnosis(cls, policy_rate_history: list[dict[str, Any]]) -> dict[str, Any]:
            if not policy_rate_history:
                return {
                    "schema_version": "policy_path_diagnosis_v1",
                    "current_path_regime": "insufficient_history",
                    "selected_recent_window": "none",
                    "recent_path_points": [],
                    "recent_change_labels": [],
                    "last_policy_change_bp": None,
                    "last_policy_change_label": None,
                    "last_policy_change_date": None,
                    "consecutive_easing_count": 0,
                    "consecutive_hiking_count": 0,
                    "consecutive_hold_count": 0,
                    "cumulative_change_since_recent_peak_bp": 0,
                    "cumulative_change_since_recent_trough_bp": 0,
                    "turning_point_detected": False,
                }

            rates = [float(item["policy_rate"]) for item in policy_rate_history]
            dates = [str(item["decision_date"]) for item in policy_rate_history]
            changes: list[dict[str, Any]] = []
            for idx in range(1, len(rates)):
                change_bp = int(round((rates[idx] - rates[idx - 1]) * 100.0))
                changes.append({
                    "decision_date": dates[idx],
                    "change_bp": change_bp,
                    "change_label": cls._rate_change_label_from_bp(change_bp),
                    "policy_rate": rates[idx],
                })

            recent_changes = changes[-8:]
            nonzero_changes = [item for item in changes if int(item["change_bp"]) != 0]
            last_nonzero = nonzero_changes[-1] if nonzero_changes else None
            last_change_bp = int(last_nonzero["change_bp"]) if last_nonzero else 0

            consecutive_hold_count = 0
            for item in reversed(changes):
                if int(item["change_bp"]) == 0:
                    consecutive_hold_count += 1
                else:
                    break

            consecutive_easing_count = 0
            consecutive_hiking_count = 0
            for item in reversed(nonzero_changes):
                change_bp = int(item["change_bp"])
                if last_change_bp < 0 and change_bp < 0:
                    consecutive_easing_count += 1
                    continue
                if last_change_bp > 0 and change_bp > 0:
                    consecutive_hiking_count += 1
                    continue
                break

            current_rate = rates[-1]
            recent_peak_rate = max(rates[-12:])
            recent_trough_rate = min(rates[-12:])
            cumulative_since_peak_bp = int(round((current_rate - recent_peak_rate) * 100.0))
            cumulative_since_trough_bp = int(round((current_rate - recent_trough_rate) * 100.0))

            turning_point_detected = False
            if len(nonzero_changes) >= 2:
                previous_sign = 1 if int(nonzero_changes[-2]["change_bp"]) > 0 else -1
                current_sign = 1 if last_change_bp > 0 else -1
                turning_point_detected = previous_sign != current_sign

            if last_change_bp < 0 and cumulative_since_peak_bp <= -50:
                regime = "easing_continuation" if consecutive_easing_count >= 2 else "easing_after_peak"
                window = "since_recent_peak"
            elif last_change_bp > 0 and cumulative_since_trough_bp >= 50:
                regime = "tightening_continuation" if consecutive_hiking_count >= 2 else "tightening_after_trough"
                window = "since_recent_trough"
            elif consecutive_hold_count >= 3 and current_rate >= recent_peak_rate - 0.125:
                regime = "high_rate_hold"
                window = "recent_hold_plateau"
            elif consecutive_hold_count >= 3 and current_rate <= recent_trough_rate + 0.125:
                regime = "low_rate_hold"
                window = "recent_hold_plateau"
            elif consecutive_hold_count >= 3:
                regime = "hold_plateau"
                window = "recent_hold_plateau"
            else:
                regime = "mixed_or_unclear"
                window = "last_8_decisions"

            return {
                "schema_version": "policy_path_diagnosis_v1",
                "current_path_regime": regime,
                "selected_recent_window": window,
                "recent_path_points": policy_rate_history[-8:],
                "recent_change_labels": [item["change_label"] for item in recent_changes],
                "recent_change_bp": [item["change_bp"] for item in recent_changes],
                "last_policy_change_bp": last_change_bp if last_nonzero else None,
                "last_policy_change_label": cls._rate_change_label_from_bp(last_change_bp) if last_nonzero else None,
                "last_policy_change_date": last_nonzero["decision_date"] if last_nonzero else None,
                "consecutive_easing_count": consecutive_easing_count,
                "consecutive_hiking_count": consecutive_hiking_count,
                "consecutive_hold_count": consecutive_hold_count,
                "recent_peak_rate": recent_peak_rate,
                "recent_trough_rate": recent_trough_rate,
                "current_policy_rate": current_rate,
                "cumulative_change_since_recent_peak_bp": cumulative_since_peak_bp,
                "cumulative_change_since_recent_trough_bp": cumulative_since_trough_bp,
                "turning_point_detected": turning_point_detected,
            }


        @staticmethod
        def _map_rate_change_to_label(rate_change_bp: int) -> str:
            if rate_change_bp < -50:
                return validate_label("<-50bp").value
            if -50 <= rate_change_bp < -25:
                return validate_label("[-50,-25)bp").value
            if -25 <= rate_change_bp < 0:
                return validate_label("[-25,0)bp").value
            if rate_change_bp == 0:
                return validate_label("0bp").value
            if 0 < rate_change_bp <= 25:
                return validate_label("(0,25]bp").value
            if 25 < rate_change_bp <= 50:
                return validate_label("(25,50]bp").value
            return validate_label(">50bp").value

        @staticmethod
        def _assert_no_missing_values(
            decision_date: date,
            current_macro_snapshot: dict[str, Any],
            market_expectations: dict[str, Any],
            policy_inputs: dict[str, Any],
        ) -> None:
            missing_fields: list[str] = []

            for field_name in REQUIRED_SNAPSHOT_FIELDS:
                if current_macro_snapshot.get(field_name) is None:
                    missing_fields.append(f"current_macro_snapshot.{field_name}")

            for field_name in REQUIRED_EXPECTATION_FIELDS:
                if market_expectations.get(field_name) is None:
                    missing_fields.append(f"market_expectations.{field_name}")

            for field_name in REQUIRED_POLICY_INPUT_FIELDS:
                if policy_inputs.get(field_name) is None:
                    missing_fields.append(f"policy_inputs.{field_name}")

            if missing_fields:
                missing_text = ", ".join(missing_fields)
                raise ValueError(
                    f"Missing required US sample fields for decision {decision_date.isoformat()}: {missing_text}"
                )


In [ ]:
# file: sample_builder/scheduler.py
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from dataclasses import dataclass, field
    from datetime import date

    @dataclass(frozen=True, slots=True)
    class ScheduledSampleRef:
        split: str
        sample_index: int
        decision_date: date
        sample_id: str

        def __post_init__(self) -> None:
            if self.split not in {"train", "test"}:
                raise ValueError("System C scheduler supports train and test splits only.")
            if not isinstance(self.sample_index, int) or self.sample_index < 0:
                raise ValueError("sample_index must be a non-negative integer.")
            if not isinstance(self.decision_date, date):
                raise TypeError("decision_date must be a datetime.date instance.")
            if not isinstance(self.sample_id, str) or not self.sample_id.strip():
                raise ValueError("sample_id must be a non-empty string.")

    @dataclass(slots=True)
    class USSampleScheduler:
        sample_builder: USSampleBuilder
        _training_schedule_cache: tuple[ScheduledSampleRef, ...] | None = field(default=None, init=False, repr=False)
        _test_schedule_cache: tuple[ScheduledSampleRef, ...] | None = field(default=None, init=False, repr=False)

        def __post_init__(self) -> None:
            if not isinstance(self.sample_builder, USSampleBuilder):
                raise TypeError("sample_builder must be a USSampleBuilder instance.")

        def training_schedule(self) -> tuple[ScheduledSampleRef, ...]:
            if self._training_schedule_cache is not None:
                return self._training_schedule_cache
            decision_dates = self.sample_builder.list_training_runnable_decision_dates()
            self._training_schedule_cache = tuple(
                ScheduledSampleRef("train", index, decision_date, self._build_sample_id("train", index, decision_date))
                for index, decision_date in enumerate(decision_dates)
            )
            return self._training_schedule_cache

        def test_schedule(self) -> tuple[ScheduledSampleRef, ...]:
            if self._test_schedule_cache is not None:
                return self._test_schedule_cache
            raw_decision_dates = self.sample_builder.list_test_runnable_decision_dates()
            prediction_end_date = date(2025, 12, 31)
            decision_dates = tuple(decision_date for decision_date in raw_decision_dates if decision_date <= prediction_end_date)
            self._test_schedule_cache = tuple(
                ScheduledSampleRef("test", index, decision_date, self._build_sample_id("test", index, decision_date))
                for index, decision_date in enumerate(decision_dates)
            )
            return self._test_schedule_cache

        def build_training_sample(self, sample_index: int) -> tuple[TrainingDecisionSample, ScheduledSampleRef]:
            ref = self._get_ref(self.training_schedule(), sample_index, "Training")
            return self.sample_builder.build_training_sample(ref.decision_date), ref

        def build_test_sample(self, sample_index: int) -> tuple[PredictionDecisionSample, ScheduledSampleRef]:
            ref = self._get_ref(self.test_schedule(), sample_index, "Test")
            return self.sample_builder.build_prediction_sample(ref.decision_date), ref

        @staticmethod
        def _get_ref(schedule: tuple[ScheduledSampleRef, ...], sample_index: int, split_name: str) -> ScheduledSampleRef:
            if not isinstance(sample_index, int) or sample_index < 0:
                raise ValueError("sample_index must be a non-negative integer.")
            if sample_index >= len(schedule):
                raise IndexError(f"{split_name} sample index {sample_index} is out of range.")
            return schedule[sample_index]

        @staticmethod
        def _build_sample_id(split: str, sample_index: int, decision_date: date) -> str:
            prefix = "train_us" if split == "train" else "test_us"
            return f"{prefix}_{sample_index:04d}_{decision_date.isoformat()}"


In [ ]:
# 文件：sample_builder/__init__.py
# 产出：导出样本构建接口。

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    __all__ = [
        "USDecisionRecord",
        "USSampleBuilder",
        "compute_one_sided_hp_gap",
        "hp_filter_trend",
    ]


## Fixed Theory Prompt Inputs

In [ ]:
# Fixed theory prompt constants for System A.
SHARED_PROMPT_PART = (
    "你是七个独立货币政策理论Agent之一。"
    "你必须只基于自己被分配的理论身份、政策逻辑、允许变量和当前可见样本进行推理。"
    "你不是最终决策Agent，不得聚合其他理论。"
)

CURRENT_OUTPUT_CONSTRAINTS = (
    "只返回一个JSON对象。recommendation和probability_distribution必须使用固定的7个标签空间。"
    "probability_distribution必须包含所有标签，且总和为1。"
    "Use only variables visible to this theory agent."
)

CURRENT_SELF_CHECK_RULES = (
    "Before returning, check that recommendation is the top probability label, all required fields are present, "
    "the explanation stays within this theory boundary, and uncertainty is reflected in confidence and probabilities."
)


In [ ]:
# 文件：theory_input_builder/theory_input_models.py
# 产出：产出理论层输入数据结构。
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from dataclasses import dataclass, field
    from datetime import date
    from typing import Any

    def _validate_non_empty_string(field_name: str, value: str) -> str:
        if not isinstance(value, str):
            raise TypeError(f"{field_name} must be a string.")
        normalized = value.strip()
        if not normalized:
            raise ValueError(f"{field_name} must be a non-empty string.")
        return normalized

    def _validate_date(field_name: str, value: date) -> date:
        if not isinstance(value, date):
            raise TypeError(f"{field_name} must be a datetime.date instance.")
        return value

    def _validate_mapping(field_name: str, value: dict[str, Any]) -> dict[str, Any]:
        if not isinstance(value, dict):
            raise TypeError(f"{field_name} must be a dict.")
        normalized: dict[str, Any] = {}
        for key, item in value.items():
            normalized_key = _validate_non_empty_string(f"{field_name} key", key)
            normalized[normalized_key] = item
        return normalized

    @dataclass(slots=True)
    class TheoryVisibleSample:
        theory_id: str
        country: str
        decision_date: date
        visible_current_macro_snapshot: dict[str, Any] = field(default_factory=dict)
        visible_market_expectations: dict[str, Any] = field(default_factory=dict)
        visible_policy_inputs: dict[str, Any] = field(default_factory=dict)
        visible_textual_inputs: dict[str, Any] = field(default_factory=dict)

        def __post_init__(self) -> None:
            self.theory_id = _validate_non_empty_string("theory_id", self.theory_id)
            self.country = _validate_non_empty_string("country", self.country)
            self.decision_date = _validate_date("decision_date", self.decision_date)
            self.visible_current_macro_snapshot = _validate_mapping(
                "visible_current_macro_snapshot", self.visible_current_macro_snapshot
            )
            self.visible_market_expectations = _validate_mapping(
                "visible_market_expectations", self.visible_market_expectations
            )
            self.visible_policy_inputs = _validate_mapping(
                "visible_policy_inputs", self.visible_policy_inputs
            )
            self.visible_textual_inputs = _validate_mapping(
                "visible_textual_inputs", self.visible_textual_inputs
            )

    @dataclass(slots=True)
    class TheoryPromptContext:
        theory_id: str
        theory_spec: TheorySpec
        visible_sample: TheoryVisibleSample
        shared_prompt_part: str
        theory_identity: str
        policy_logic: str
        theory_anchor_keys: tuple[str, ...]
        execution_protocol: str
        output_constraints: str
        self_check_rules: str
        forbidden_logic_crossovers: tuple[str, ...]

        def __post_init__(self) -> None:
            self.theory_id = _validate_non_empty_string("theory_id", self.theory_id)
            if not isinstance(self.theory_spec, TheorySpec):
                raise TypeError("theory_spec must be a TheorySpec instance.")
            if not isinstance(self.visible_sample, TheoryVisibleSample):
                raise TypeError("visible_sample must be a TheoryVisibleSample instance.")
            if self.theory_id != self.theory_spec.theory_id:
                raise ValueError("theory_id must match theory_spec.theory_id.")
            if self.theory_id != self.visible_sample.theory_id:
                raise ValueError("theory_id must match visible_sample.theory_id.")
            self.shared_prompt_part = _validate_non_empty_string("shared_prompt_part", self.shared_prompt_part)
            self.theory_identity = _validate_non_empty_string("theory_identity", self.theory_identity)
            self.policy_logic = _validate_non_empty_string("policy_logic", self.policy_logic)
            self.execution_protocol = _validate_non_empty_string(
                "execution_protocol", self.execution_protocol
            )
            self.output_constraints = _validate_non_empty_string("output_constraints", self.output_constraints)
            self.self_check_rules = _validate_non_empty_string("self_check_rules", self.self_check_rules)
            if not isinstance(self.theory_anchor_keys, tuple) or not self.theory_anchor_keys:
                raise ValueError("theory_anchor_keys must be a non-empty tuple.")
            self.theory_anchor_keys = tuple(
                _validate_non_empty_string(f"theory_anchor_keys[{i}]", key)
                for i, key in enumerate(self.theory_anchor_keys)
            )
            if not isinstance(self.forbidden_logic_crossovers, tuple):
                raise TypeError("forbidden_logic_crossovers must be a tuple.")
            self.forbidden_logic_crossovers = tuple(
                _validate_non_empty_string(f"forbidden_logic_crossovers[{idx}]", item)
                for idx, item in enumerate(self.forbidden_logic_crossovers)
            )


In [ ]:
# 文件：theory_input_builder/history_access_guard.py
# 产出：产出理论历史可见性边界校验逻辑。
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    POLICY_RATE_FIELDS = (
        "previous_policy_rate",
        "policy_rate_history_full",
    )

    FUTURE_POLICY_FORBIDDEN_FIELDS = (
        "observed_policy_rate",
        "observed_policy_rate_after_decision",
        "current_policy_rate",
        "rate_change_bp",
    )

    def _assert_required_policy_history_keys(
        visible_policy_inputs: dict[str, object],
        theory_spec: TheorySpec,
    ) -> None:
        missing_keys = [
            field_name
            for field_name in theory_spec.required_policy_history_keys
            if field_name not in visible_policy_inputs
        ]
        if missing_keys:
            raise ValueError(
                f"Theory {theory_spec.theory_id} is missing required policy history keys after "
                f"history-access filtering: {missing_keys}."
            )

    def enforce_history_access_policy(
        visible_sample: TheoryVisibleSample,
        theory_spec: TheorySpec,
    ) -> TheoryVisibleSample:
        if not isinstance(visible_sample, TheoryVisibleSample):
            raise TypeError("visible_sample must be a TheoryVisibleSample instance.")
        if not isinstance(theory_spec, TheorySpec):
            raise TypeError("theory_spec must be a TheorySpec instance.")
        if visible_sample.theory_id != theory_spec.theory_id:
            raise ValueError("visible_sample.theory_id must match theory_spec.theory_id.")

        visible_policy_inputs = dict(visible_sample.visible_policy_inputs)

        if theory_spec.history_access_policy == HistoryAccessPolicy.NONE:
            for field_name in POLICY_RATE_FIELDS:
                visible_policy_inputs.pop(field_name, None)
        elif theory_spec.history_access_policy == HistoryAccessPolicy.PREVIOUS_POLICY_RATE_ONLY:
            allowed = {"previous_policy_rate"}
            for field_name in POLICY_RATE_FIELDS:
                if field_name not in allowed:
                    visible_policy_inputs.pop(field_name, None)
        elif theory_spec.history_access_policy == HistoryAccessPolicy.FULL_DECISION_RATE_HISTORY:
            allowed = {"policy_rate_history_full"}
            for field_name in POLICY_RATE_FIELDS:
                if field_name not in allowed:
                    visible_policy_inputs.pop(field_name, None)
        else:
            raise ValueError(
                f"Unsupported history access policy {theory_spec.history_access_policy!r}."
            )

        _assert_required_policy_history_keys(visible_policy_inputs, theory_spec)

        return TheoryVisibleSample(
            theory_id=visible_sample.theory_id,
            country=visible_sample.country,
            decision_date=visible_sample.decision_date,
            visible_current_macro_snapshot=dict(visible_sample.visible_current_macro_snapshot),
            visible_market_expectations=dict(visible_sample.visible_market_expectations),
            visible_policy_inputs=visible_policy_inputs,
            visible_textual_inputs=dict(visible_sample.visible_textual_inputs),
        )

    def assert_no_future_policy_information(visible_sample: TheoryVisibleSample) -> None:
        if not isinstance(visible_sample, TheoryVisibleSample):
            raise TypeError("visible_sample must be a TheoryVisibleSample instance.")

        visible_blocks = {
            "visible_current_macro_snapshot": visible_sample.visible_current_macro_snapshot,
            "visible_market_expectations": visible_sample.visible_market_expectations,
            "visible_policy_inputs": visible_sample.visible_policy_inputs,
            "visible_textual_inputs": visible_sample.visible_textual_inputs,
        }

        for field_name in FUTURE_POLICY_FORBIDDEN_FIELDS:
            for block_name, block in visible_blocks.items():
                if field_name in block:
                    raise ValueError(
                        "Theory-visible sample contains forbidden future/post-decision policy field "
                        f"{field_name!r} in {block_name}."
                    )


In [ ]:
# 文件：theory_input_builder/variable_router.py
# 产出：产出理论变量授权裁切逻辑。
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from typing import Any

    def collect_sample_field_pool(sample: BaseDecisionSample) -> dict[str, tuple[str, Any]]:
        if not isinstance(sample, BaseDecisionSample):
            raise TypeError("sample must be a BaseDecisionSample instance.")

        field_pool: dict[str, tuple[str, Any]] = {}
        source_blocks = (
            ("current_macro_snapshot", sample.current_macro_snapshot),
            ("market_expectations", sample.market_expectations),
            ("policy_inputs", sample.policy_inputs),
            ("textual_inputs", sample.textual_inputs),
        )

        for source_block, block in source_blocks:
            for field_name, field_value in block.items():
                if field_name in field_pool:
                    previous_source, _ = field_pool[field_name]
                    raise ValueError(
                        f"Duplicate field name {field_name!r} found in both {previous_source} and {source_block}."
                    )
                field_pool[field_name] = (source_block, field_value)

        return field_pool

    def route_variables_for_theory(
        sample: BaseDecisionSample,
        theory_spec: TheorySpec,
    ) -> TheoryVisibleSample:
        if not isinstance(sample, BaseDecisionSample):
            raise TypeError("sample must be a BaseDecisionSample instance.")
        if not isinstance(theory_spec, TheorySpec):
            raise TypeError("theory_spec must be a TheorySpec instance.")

        field_pool = collect_sample_field_pool(sample)

        visible_current_macro_snapshot: dict[str, Any] = {}
        visible_market_expectations: dict[str, Any] = {}
        visible_policy_inputs: dict[str, Any] = {}
        visible_textual_inputs: dict[str, Any] = {}

        for field_name in theory_spec.allowed_variables:
            try:
                source_block, field_value = field_pool[field_name]
            except KeyError as exc:
                raise KeyError(
                    f"Theory {theory_spec.theory_id} requires field {field_name!r}, "
                    "but it is not present in the sample."
                ) from exc

            if source_block == "current_macro_snapshot":
                visible_current_macro_snapshot[field_name] = field_value
            elif source_block == "market_expectations":
                visible_market_expectations[field_name] = field_value
            elif source_block == "policy_inputs":
                visible_policy_inputs[field_name] = field_value
            elif source_block == "textual_inputs":
                visible_textual_inputs[field_name] = field_value
            else:
                raise ValueError(f"Unsupported source_block {source_block!r} for field {field_name!r}.")

        return TheoryVisibleSample(
            theory_id=theory_spec.theory_id,
            country=sample.country,
            decision_date=sample.decision_date,
            visible_current_macro_snapshot=visible_current_macro_snapshot,
            visible_market_expectations=visible_market_expectations,
            visible_policy_inputs=visible_policy_inputs,
            visible_textual_inputs=visible_textual_inputs,
        )


In [ ]:
# file: theory_input_builder/builder.py
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    def build_theory_visible_sample(sample: BaseDecisionSample, theory_spec: TheorySpec) -> TheoryVisibleSample:
        if not isinstance(sample, BaseDecisionSample):
            raise TypeError("sample must be a BaseDecisionSample instance.")
        if not isinstance(theory_spec, TheorySpec):
            raise TypeError("theory_spec must be a TheorySpec instance.")
        visible_sample = route_variables_for_theory(sample, theory_spec)
        visible_sample = enforce_history_access_policy(visible_sample, theory_spec)
        assert_no_future_policy_information(visible_sample)
        return visible_sample

    def build_theory_prompt_context(sample: BaseDecisionSample, theory_spec: TheorySpec) -> TheoryPromptContext:
        if not isinstance(sample, BaseDecisionSample):
            raise TypeError("sample must be a BaseDecisionSample instance.")
        if not isinstance(theory_spec, TheorySpec):
            raise TypeError("theory_spec must be a TheorySpec instance.")
        visible_sample = build_theory_visible_sample(sample, theory_spec)
        prompt_modules = theory_spec.prompt_modules
        return TheoryPromptContext(
            theory_id=theory_spec.theory_id,
            theory_spec=theory_spec,
            visible_sample=visible_sample,
            shared_prompt_part=SHARED_PROMPT_PART,
            theory_identity=prompt_modules.theory_identity,
            policy_logic=prompt_modules.policy_logic,
            theory_anchor_keys=prompt_modules.theory_anchor_keys,
            execution_protocol=prompt_modules.execution_protocol,
            output_constraints=CURRENT_OUTPUT_CONSTRAINTS,
            self_check_rules=CURRENT_SELF_CHECK_RULES,
            forbidden_logic_crossovers=theory_spec.forbidden_logic_crossovers,
        )

    def build_all_theory_prompt_contexts(
        sample: BaseDecisionSample,
        theory_specs: tuple[TheorySpec, ...] | None = None,
    ) -> dict[str, TheoryPromptContext]:
        if not isinstance(sample, BaseDecisionSample):
            raise TypeError("sample must be a BaseDecisionSample instance.")
        selected_specs = THEORY_SPECS if theory_specs is None else theory_specs
        contexts: dict[str, TheoryPromptContext] = {}
        for theory_spec in selected_specs:
            if not isinstance(theory_spec, TheorySpec):
                raise TypeError("theory_specs must contain TheorySpec instances.")
            contexts[theory_spec.theory_id] = build_theory_prompt_context(sample, theory_spec)
        return contexts


In [ ]:
# 文件：theory_input_builder/__init__.py
# 产出：导出理论层输入构建接口。

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    __all__ = [
        "TheoryPromptContext",
        "TheoryVisibleSample",
        "assert_no_future_policy_information",
        "build_all_theory_prompt_contexts",
        "build_theory_prompt_context",
        "build_theory_visible_sample",
        "collect_sample_field_pool",
        "enforce_history_access_policy",
        "route_variables_for_theory",
    ]


## Theory Agents

In [ ]:
# 文件：theory_agents/base.py
# 产出：产出理论层抽象基类。
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from abc import ABC, abstractmethod
    from dataclasses import dataclass

    @dataclass(frozen=True, slots=True)
    class RenderedTheoryPrompt:
        theory_id: str
        system_prompt: str
        user_prompt: str

        def __post_init__(self) -> None:
            if not isinstance(self.theory_id, str) or not self.theory_id.strip():
                raise ValueError("theory_id must be a non-empty string.")
            if not isinstance(self.system_prompt, str) or not self.system_prompt.strip():
                raise ValueError("system_prompt must be a non-empty string.")
            if not isinstance(self.user_prompt, str) or not self.user_prompt.strip():
                raise ValueError("user_prompt must be a non-empty string.")

    class TheoryModelBackend(ABC):
        @abstractmethod
        def generate(
            self,
            prompt: RenderedTheoryPrompt,
            context: TheoryPromptContext,
        ) -> str:
            """
            Execute one theory-agent call and return the raw model response text.
            """

    class TheoryAgentRunner(ABC):
        @abstractmethod
        def run(self, context: TheoryPromptContext) -> TheoryAgentOutput:
            """
            Run one theory agent from prompt context to validated structured output.
            """


In [ ]:
# 文件：theory_agents/output_parser.py
# 产出：产出理论层 JSON 输出解析与校验逻辑。
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    import json
    import re
    from math import isfinite
    from typing import Any

    THEORY_REQUIRED_OUTPUT_FIELDS = (
        "recommendation",
        "probability_distribution",
        "confidence",
        "primary_policy_reason",
        "secondary_policy_reason",
        "key_variables_used",
        "theory_explanation",
        "applicability_note",
        "risk_note",
        "policy_path_regime_assessment",
    )

    THEORY_HIGH_CONFIDENCE_THRESHOLD = 80
    THEORY_MIN_TOP_PROBABILITY_FOR_HIGH_CONFIDENCE = 0.45

    def _theory_strip_markdown_code_fence(text: str) -> str:
        stripped = text.strip()
        if stripped.startswith("```") and stripped.endswith("```"):
            lines = stripped.splitlines()
            if len(lines) >= 3:
                return "\n".join(lines[1:-1]).strip()
        return stripped

    def _theory_extract_json_object_candidate(text: str) -> str:
        start = text.find("{")
        if start < 0:
            return text

        depth = 0
        in_string = False
        escaped = False
        for index in range(start, len(text)):
            char = text[index]
            if in_string:
                if escaped:
                    escaped = False
                elif char == "\\":
                    escaped = True
                elif char == '"':
                    in_string = False
                continue
            if char == '"':
                in_string = True
            elif char == "{":
                depth += 1
            elif char == "}":
                depth -= 1
                if depth == 0:
                    return text[start : index + 1]
        return text[start:].strip()

    def _theory_count_unclosed_outer_braces(text: str) -> int:
        depth = 0
        in_string = False
        escaped = False
        for char in text:
            if in_string:
                if escaped:
                    escaped = False
                elif char == "\\":
                    escaped = True
                elif char == '"':
                    in_string = False
                continue
            if char == '"':
                in_string = True
            elif char == "{":
                depth += 1
            elif char == "}":
                depth = max(0, depth - 1)
        return depth

    def _theory_candidate_json_variants(text: str) -> list[str]:
        candidates: list[str] = []

        def add(candidate: str) -> None:
            cleaned = candidate.strip()
            if cleaned and cleaned not in candidates:
                candidates.append(cleaned)

        add(text)
        stripped = _theory_strip_markdown_code_fence(text)
        add(stripped)
        extracted = _theory_extract_json_object_candidate(stripped)
        add(extracted)

        for candidate in list(candidates):
            add(re.sub(r",\s*([}\]])", r"\1", candidate))
            missing_braces = _theory_count_unclosed_outer_braces(candidate)
            if missing_braces > 0:
                add(candidate + ("}" * missing_braces))

        return candidates

    def _theory_coerce_mapping(raw_output: str | dict[str, Any]) -> dict[str, Any]:
        if isinstance(raw_output, dict):
            return dict(raw_output)
        if not isinstance(raw_output, str):
            raise TypeError("raw_output must be either a JSON string or a dict.")

        normalized = raw_output.strip()
        if not normalized:
            raise ValueError("raw_output must not be empty.")

        parsed = None
        last_error: json.JSONDecodeError | None = None
        for candidate in _theory_candidate_json_variants(normalized):
            try:
                parsed = json.loads(candidate)
                break
            except json.JSONDecodeError as exc:
                last_error = exc
        if parsed is None:
            raise ValueError("Theory agent output must be valid JSON.") from last_error

        if not isinstance(parsed, dict):
            raise ValueError("Theory agent output JSON must be an object.")
        return parsed

    def _theory_assert_exact_field_set(payload: dict[str, Any]) -> None:
        actual_fields = set(payload)
        required_fields = set(THEORY_REQUIRED_OUTPUT_FIELDS)
        missing = sorted(required_fields - actual_fields)
        extra = sorted(actual_fields - required_fields)
        if missing or extra:
            raise ValueError(
                "Theory agent output must contain exactly the fixed 10 fields. "
                f"Missing: {missing}. Extra: {extra}."
            )

    def _theory_validate_key_variables_used(
        key_variables_used: list[str],
        context: TheoryPromptContext,
    ) -> None:
        allowed_variables = set(context.theory_spec.allowed_variables)
        visible_variables = (
            set(context.visible_sample.visible_current_macro_snapshot)
            | set(context.visible_sample.visible_market_expectations)
            | set(context.visible_sample.visible_policy_inputs)
            | set(context.visible_sample.visible_textual_inputs)
        )

        for variable_name in key_variables_used:
            if variable_name not in allowed_variables:
                raise ValueError(
                    f"key_variables_used contains unauthorized variable {variable_name!r} for theory {context.theory_id}."
                )
            if variable_name not in visible_variables:
                raise ValueError(
                    f"key_variables_used contains non-visible variable {variable_name!r} for theory {context.theory_id}."
                )

        missing_required_history_keys = [
            field_name
            for field_name in context.theory_spec.required_policy_history_keys
            if field_name not in key_variables_used
        ]
        if missing_required_history_keys:
            raise ValueError(
                f"key_variables_used must include required policy-history variables "
                f"{missing_required_history_keys} for theory {context.theory_id}."
            )

    def _theory_coerce_key_variables_used(raw_value: Any) -> list[str]:
        if isinstance(raw_value, list):
            return raw_value
        if isinstance(raw_value, str):
            normalized = raw_value.strip()
            if not normalized:
                return []
            if normalized.startswith("[") and normalized.endswith("]"):
                try:
                    parsed = json.loads(normalized)
                except json.JSONDecodeError:
                    parsed = None
                if isinstance(parsed, list):
                    return parsed
            pieces = [
                item.strip().strip("\"' `")
                for item in re.split(r"[,，、;\n]+", normalized)
            ]
            return [item for item in pieces if item]
        return [raw_value]

    def _theory_normalize_key_variables_used(
        raw_variables: list[Any],
        context: TheoryPromptContext,
    ) -> list[str]:
        allowed_variables = tuple(context.theory_spec.allowed_variables)
        allowed_set = set(allowed_variables)
        visible_variables = (
            set(context.visible_sample.visible_current_macro_snapshot)
            | set(context.visible_sample.visible_market_expectations)
            | set(context.visible_sample.visible_policy_inputs)
            | set(context.visible_sample.visible_textual_inputs)
        )
        usable_variables = [name for name in allowed_variables if name in visible_variables]
        alias_map = {
            "output_gap_current": "output_gap",
            "current_output_gap": "output_gap",
            "output gap current": "output_gap",
            "current output gap": "output_gap",
            "inflation_gap": "inflation_current",
            "inflation deviation": "inflation_current",
            "inflation_deviation": "inflation_current",
            "policy_rate_previous": "previous_policy_rate",
            "previous rate": "previous_policy_rate",
            "expected output gap": "expected_output_gap",
            "expected inflation": "expected_inflation",
            "core inflation": "core_inflation",
            "house prices": "house_price_yoy",
            "credit growth": "domestic_credit_yoy",
            "import price inflation": "import_price_yoy",
            "stock prices": "stock_price_yoy",
            "equity prices": "stock_price_yoy",
            "credit spread": "corporate_credit_spread",
            "import price inflation": "import_price_yoy",
            "stock prices": "stock_price_yoy",
            "equity prices": "stock_price_yoy",
            "credit spread": "corporate_credit_spread",
            "import price inflation": "import_price_yoy",
            "stock prices": "stock_price_yoy",
            "equity prices": "stock_price_yoy",
            "credit spread": "corporate_credit_spread",
            "import price inflation": "import_price_yoy",
            "stock prices": "stock_price_yoy",
            "equity prices": "stock_price_yoy",
            "credit spread": "corporate_credit_spread",
            "import price inflation": "import_price_yoy",
            "stock prices": "stock_price_yoy",
            "equity prices": "stock_price_yoy",
            "credit spread": "corporate_credit_spread",
            "import price inflation": "import_price_yoy",
            "stock prices": "stock_price_yoy",
            "equity prices": "stock_price_yoy",
            "credit spread": "corporate_credit_spread",
            "import price inflation": "import_price_yoy",
            "stock prices": "stock_price_yoy",
            "equity prices": "stock_price_yoy",
            "credit spread": "corporate_credit_spread",
            "import price inflation": "import_price_yoy",
            "stock prices": "stock_price_yoy",
            "equity prices": "stock_price_yoy",
            "credit spread": "corporate_credit_spread",
            "import price inflation": "import_price_yoy",
            "stock prices": "stock_price_yoy",
            "equity prices": "stock_price_yoy",
            "credit spread": "corporate_credit_spread",
        }
        keyword_map = (
            ("产出缺口", ("output_gap", "expected_output_gap")),
            ("通胀目标", ("inflation_target_mid",)),
            ("目标", ("inflation_target_mid",)),
            ("核心通胀", ("core_inflation",)),
            ("通胀", ("inflation_current", "expected_inflation")),
            ("失业", ("unemployment_rate",)),
            ("汇率", ("reer",)),
            ("信用", ("domestic_credit_yoy", "credit_to_gdp_ratio")),
            ("房价", ("house_price_yoy",)),
            ("上一期", ("previous_policy_rate",)),
            ("历史", ("policy_rate_history_full",)),
        )

        normalized: list[str] = []

        def add_if_usable(candidate: str) -> None:
            if candidate in allowed_set and candidate in visible_variables and candidate not in normalized:
                normalized.append(candidate)

        for raw_item in raw_variables:
            text = str(raw_item).strip().strip("`'\"")
            if not text:
                continue
            before_count = len(normalized)
            add_if_usable(text)
            lowered = re.sub(r"\s+", " ", text.replace("_", " ").lower()).strip()
            compact = text.strip().lower()
            alias_candidate = alias_map.get(compact) or alias_map.get(lowered)
            if alias_candidate:
                add_if_usable(alias_candidate)
            for allowed_name in usable_variables:
                if allowed_name.lower() in compact:
                    add_if_usable(allowed_name)
            for keyword, candidates in keyword_map:
                if keyword in text.lower() or keyword in text:
                    for candidate in candidates:
                        add_if_usable(candidate)
            if len(normalized) == before_count:
                raise ValueError(
                    f"key_variables_used item {text!r} cannot be mapped to an authorized "
                    f"visible variable for theory {context.theory_id}."
                )

        for required_key in context.theory_spec.required_policy_history_keys:
            add_if_usable(required_key)
        if normalized:
            return normalized
        raise ValueError(
            f"key_variables_used must contain at least one authorized visible variable "
            f"for theory {context.theory_id}."
        )

    def _theory_validate_confidence_consistency(
        parsed_output: TheoryAgentOutput,
    ) -> None:
        top_probability = parsed_output.probability_distribution.probability_for(
            parsed_output.recommendation
        )
        if (
            parsed_output.confidence >= THEORY_HIGH_CONFIDENCE_THRESHOLD
            and top_probability < THEORY_MIN_TOP_PROBABILITY_FOR_HIGH_CONFIDENCE
        ):
            raise ValueError(
                "confidence is too high relative to the support concentration in "
                "probability_distribution."
            )

        applicability_note = parsed_output.applicability_note
        if (
            ("受限" in applicability_note or "不适用" in applicability_note)
            and parsed_output.confidence >= THEORY_HIGH_CONFIDENCE_THRESHOLD
        ):
            raise ValueError(
                "confidence must not remain in the high-confidence band when "
                "applicability_note already states that the theory is limited or not applicable."
            )

    def _theory_coerce_policy_path_regime_assessment(
        raw_value: Any,
        context: TheoryPromptContext,
    ) -> dict[str, object]:
        if not isinstance(context, TheoryPromptContext):
            raise TypeError("context must be a TheoryPromptContext instance.")

        if isinstance(raw_value, dict):
            assessment = dict(raw_value)
            coerced = False
            raw_text = ""
        else:
            assessment = {}
            coerced = True
            raw_text = "not_applicable" if raw_value is None else str(raw_value)

        if context.theory_id == "new_keynesian_optimal_rule":
            default_regime = "unparsed_policy_path_regime" if coerced else "unspecified_policy_path_regime"
            default_reason = "new_keynesian_policy_path_assessment_was_missing_or_malformed" if coerced else "new_keynesian_policy_path_assessment_missing_fields"
        else:
            default_regime = "not_applicable"
            default_reason = "this_theory_is_not_the_new_keynesian_optimal_rule"

        normalized: dict[str, object] = {
            "regime": assessment.get("regime", default_regime),
            "selected_recent_window": assessment.get("selected_recent_window", "not_applicable"),
            "path_signal_for_this_theory": assessment.get("path_signal_for_this_theory", raw_text or "not_applicable"),
            "does_path_signal_affect_target_criterion_judgment": assessment.get("does_path_signal_affect_target_criterion_judgment", False),
            "reason": assessment.get("reason", default_reason),
        }

        for optional_key in ("supporting_path_evidence", "raw_value"):
            if optional_key in assessment:
                normalized[optional_key] = assessment[optional_key]
        if coerced:
            normalized["raw_value"] = raw_text
            normalized["coerced_from_non_object"] = True

        for text_key in ("regime", "selected_recent_window", "path_signal_for_this_theory", "reason"):
            value = normalized.get(text_key)
            if value is None:
                normalized[text_key] = "not_applicable"
            elif not isinstance(value, str):
                normalized[text_key] = str(value)
            elif not value.strip():
                normalized[text_key] = "not_applicable"

        direction_value = normalized.get("does_path_signal_affect_target_criterion_judgment")
        if isinstance(direction_value, bool):
            normalized["does_path_signal_affect_target_criterion_judgment"] = direction_value
        elif isinstance(direction_value, str):
            normalized["does_path_signal_affect_target_criterion_judgment"] = direction_value.strip().lower() in {"true", "yes", "1"}
        else:
            normalized["does_path_signal_affect_target_criterion_judgment"] = bool(direction_value)

        return normalized

    def parse_theory_agent_output(
        raw_output: str | dict[str, Any],
        context: TheoryPromptContext,
    ) -> TheoryAgentOutput:
        if not isinstance(context, TheoryPromptContext):
            raise TypeError("context must be a TheoryPromptContext instance.")

        payload = _theory_coerce_mapping(raw_output)
        _theory_assert_exact_field_set(payload)

        probability_distribution = ProbabilityDistribution(
            probabilities=payload["probability_distribution"]
        )
        parsed_output = TheoryAgentOutput(
            recommendation=payload["recommendation"],
            probability_distribution=probability_distribution,
            confidence=payload["confidence"],
            primary_policy_reason=payload["primary_policy_reason"],
            secondary_policy_reason=payload["secondary_policy_reason"],
            key_variables_used=_theory_normalize_key_variables_used(_theory_coerce_key_variables_used(payload["key_variables_used"]), context),
            theory_explanation=payload["theory_explanation"],
            applicability_note=payload["applicability_note"],
            risk_note=payload["risk_note"],
            policy_path_regime_assessment=_theory_coerce_policy_path_regime_assessment(
                payload["policy_path_regime_assessment"],
                context,
            ),
        )
        _theory_validate_key_variables_used(parsed_output.key_variables_used, context)
        _theory_validate_confidence_consistency(parsed_output)
        return parsed_output


In [ ]:
# 文件：theory_agents/prompt_renderer.py
# 产出：产出理论层 prompt 渲染逻辑。
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    import json
    from typing import Any

    def _render_scalar(value: Any) -> str:
        if isinstance(value, (dict, list, tuple, str, int, float, bool)) or value is None:
            if isinstance(value, (dict, list, tuple)):
                return json.dumps(value, ensure_ascii=False, indent=2)
            return json.dumps(value, ensure_ascii=False)
        return repr(value)

    def _ordered_block_lines(block_name: str, payload: dict[str, Any]) -> list[str]:
        lines = [f"{block_name}:"]
        if not payload:
            lines.append("  - <空>")
            return lines

        for key in sorted(payload):
            lines.append(f"  - {key}: {_render_scalar(payload[key])}")
        return lines

    def _render_visible_sample(sample: TheoryVisibleSample) -> str:
        lines: list[str] = [
            f"国家: {sample.country}",
            f"决议日期: {sample.decision_date.isoformat()}",
        ]
        lines.extend(_ordered_block_lines("当前宏观快照", sample.visible_current_macro_snapshot))
        lines.extend(_ordered_block_lines("市场预期字段", sample.visible_market_expectations))
        lines.extend(_ordered_block_lines("政策相关输入", sample.visible_policy_inputs))
        lines.extend(_ordered_block_lines("文本输入字段", sample.visible_textual_inputs))
        return "\n".join(lines)

    def _render_output_contract() -> str:
        ordered_labels = [label.value for label in LABEL_ORDER]
        distribution_template = {label: 0.0 for label in ordered_labels}
        contract = {
            "recommendation": "7个固定标签中的唯一一个标签",
            "probability_distribution": distribution_template,
            "confidence": "0到100之间的整数",
            "primary_policy_reason": "简短文本",
            "secondary_policy_reason": "简短文本",
            "key_variables_used": ["已授权变量名"],
            "theory_explanation": "文本",
            "applicability_note": "文本",
            "risk_note": "文本；必须明确说明 failure_risk，并指出本理论最可能系统性误判的机制。",
            "policy_path_regime_assessment": {
                "applicability_score": "0-100 integer",
                "regime": "not_applicable or one of liftoff_waiting/liftoff_start/gradual_hiking_cycle/late_hiking_cycle/terminal_rate_plateau/insurance_cut_cycle/crisis_cut/easing_cycle_continuation/normal_regime",
                "selected_recent_window": "not_applicable or window_name",
                "path_signal_for_this_theory": "text",
                "does_path_signal_affect_target_criterion_judgment": False,
                "reason": "text",
            },
        }
        return json.dumps(contract, ensure_ascii=False, indent=2)

    def render_theory_agent_prompt(context: TheoryPromptContext) -> RenderedTheoryPrompt:
        if not isinstance(context, TheoryPromptContext):
            raise TypeError("context must be a TheoryPromptContext instance.")

        ordered_labels = ", ".join(label.value for label in LABEL_ORDER)
        allowed_variables = ", ".join(context.theory_spec.allowed_variables)
        forbidden_logic = "\n".join(f"- {item}" for item in context.forbidden_logic_crossovers)
        anchor_keys = "\n".join(f"- {key}" for key in context.theory_anchor_keys)

        system_prompt = "\n".join(
            [
                f"你当前扮演的理论智能体编号是 `{context.theory_id}`。",
                context.shared_prompt_part,
            ]
        )

        user_prompt = "\n".join(
            [
                "[理论身份]",
                context.theory_identity,
                "",
                "[理论逻辑]",
                context.policy_logic,
                "",
                "[理论锚点]",
                anchor_keys,
                "",
                "[适用性、失败风险、路径边界]",
                "必须在applicability_note中明确给出适用性判断，并在risk_note中明确给出failure_risk。applicability_score 是当前样本下本理论的适用性评分，不是置信度。",
                "Only theories authorized to read policy-rate history may identify policy-path regimes; other theories must write not_applicable.",
                "If a static target-rate signal may overstate the actual meeting-by-meeting move, state that risk explicitly.",
                "Authorized path-sensitive theories must classify the current policy stage when relevant: liftoff_waiting, liftoff_start, gradual_hiking_cycle, late_hiking_cycle, terminal_rate_plateau, insurance_cut_cycle, crisis_cut, easing_cycle_continuation, or normal_regime.",
                "interest_rate_smoothing and new_keynesian_optimal_rule must distinguish first hike/liftoff, late hiking, plateau, first cut or easing continuation, and crisis cut; this is a general stage diagnosis, not a date answer.",
                "In easing_cycle_continuation or insurance_cut_cycle, high inflation or high one-year inflation expectations usually constrain cut magnitude or support a pause; do not convert them into a current-meeting hike unless broad reacceleration evidence is overwhelming within the theory boundary.",
                "",
                "[执行协议]",
                context.execution_protocol,
                "",
                "[禁止越界逻辑]",
                forbidden_logic,
                "",
                "[授权变量]",
                allowed_variables,
                "",
                "[固定标签空间]",
                ordered_labels,
                "",
                "[当前可见样本]",
                _render_visible_sample(context.visible_sample),
                "",
                "[输出要求]",
                context.output_constraints,
                _render_output_contract(),
                "",
                "[自检规则]",
                context.self_check_rules,
            ]
        )

        return RenderedTheoryPrompt(
            theory_id=context.theory_id,
            system_prompt=system_prompt,
            user_prompt=user_prompt,
        )


In [ ]:
# 文件：theory_agents/runner.py
# 产出：产出单理论运行器与六理论批量运行逻辑。
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from dataclasses import dataclass
    import json

    @dataclass(slots=True)
    class SingleTheoryAgentRunner(TheoryAgentRunner):
        backend: TheoryModelBackend

        def __post_init__(self) -> None:
            if not isinstance(self.backend, TheoryModelBackend):
                raise TypeError("backend must be a TheoryModelBackend instance.")

        def render_prompt(self, context: TheoryPromptContext) -> RenderedTheoryPrompt:
            if not isinstance(context, TheoryPromptContext):
                raise TypeError("context must be a TheoryPromptContext instance.")
            return render_theory_agent_prompt(context)

        def invoke_backend(
            self,
            prompt: RenderedTheoryPrompt,
            context: TheoryPromptContext,
        ) -> str:
            if not isinstance(prompt, RenderedTheoryPrompt):
                raise TypeError("prompt must be a RenderedTheoryPrompt instance.")
            if not isinstance(context, TheoryPromptContext):
                raise TypeError("context must be a TheoryPromptContext instance.")
            if prompt.theory_id != context.theory_id:
                raise ValueError("prompt.theory_id must match context.theory_id.")
            return self.backend.generate(prompt=prompt, context=context)

        def parse_output(
            self,
            raw_output: str,
            context: TheoryPromptContext,
        ) -> TheoryAgentOutput:
            if not isinstance(raw_output, str):
                raise TypeError("raw_output must be a string.")
            if not isinstance(context, TheoryPromptContext):
                raise TypeError("context must be a TheoryPromptContext instance.")
            return parse_theory_agent_output(raw_output=raw_output, context=context)

        @staticmethod
        def _build_repair_prompt(
            prompt: RenderedTheoryPrompt,
            raw_output: str,
            error_message: str,
            context: TheoryPromptContext,
        ) -> RenderedTheoryPrompt:
            if not isinstance(prompt, RenderedTheoryPrompt):
                raise TypeError("prompt must be a RenderedTheoryPrompt instance.")
            if not isinstance(raw_output, str) or not raw_output.strip():
                raise ValueError("raw_output must be a non-empty string.")
            if not isinstance(error_message, str) or not error_message.strip():
                raise ValueError("error_message must be a non-empty string.")
            if not isinstance(context, TheoryPromptContext):
                raise TypeError("context must be a TheoryPromptContext instance.")

            visible_variables = (
                set(context.visible_sample.visible_current_macro_snapshot)
                | set(context.visible_sample.visible_market_expectations)
                | set(context.visible_sample.visible_policy_inputs)
                | set(context.visible_sample.visible_textual_inputs)
            )
            allowed_visible_variables = [name for name in context.theory_spec.allowed_variables if name in visible_variables]
            labels = [label.value for label in LABEL_ORDER]
            repair_template = {
                "recommendation": "0bp",
                "probability_distribution": {label: 0.0 for label in labels},
                "confidence": 50,
                "primary_policy_reason": "用当前样本中该理论允许变量解释主要政策信号。",
                "secondary_policy_reason": "用当前样本中该理论允许变量解释次要政策信号。",
                "key_variables_used": allowed_visible_variables[:2] or list(context.theory_spec.allowed_variables[:1]),
                "theory_explanation": "说明该理论如何从允许变量推导本次recommendation。",
                "applicability_note": "说明该理论在当前样本上的适用性。",
                "risk_note": "说明该理论在当前样本上的误判风险。",
                "policy_path_regime_assessment": {
                    "regime": "not_applicable",
                    "selected_recent_window": "not_applicable",
                    "path_signal_for_this_theory": "not_applicable",
                    "does_path_override_gap_signal": False,
                    "rationale": "not_applicable",
                },
            }
            repair_user_prompt = "\n".join(
                [
                    "[TASK]",
                    "Repair the invalid theory-agent response by returning one fresh valid JSON object.",
                    "Do not copy malformed field names from the invalid response.",
                    "",
                    "[VALIDATION_ERROR]",
                    error_message.strip(),
                    "",
                    "[INVALID_PREVIOUS_RESPONSE_FOR_REFERENCE_ONLY]",
                    raw_output.strip(),
                    "",
                    "[CURRENT_THEORY_ID]",
                    context.theory_id,
                    "",
                    "[ALLOWED_KEY_VARIABLES_USED_VALUES]",
                    json.dumps(allowed_visible_variables, ensure_ascii=False),
                    "",
                    "[FIXED_LABEL_SPACE]",
                    json.dumps(labels, ensure_ascii=False),
                    "",
                    "[MANDATORY_JSON_TEMPLATE_SHAPE]",
                    json.dumps(repair_template, ensure_ascii=False, indent=2),
                    "",
                    "[HARD_REPAIR_RULES]",
                    "- Return exactly one JSON object. No Markdown, no code fence, no explanation outside JSON.",
                    "- The object must contain exactly the 10 top-level fields shown in MANDATORY_JSON_TEMPLATE_SHAPE.",
                    "- The field name must be recommendation. Do not use recommend, recomm, label, final_recommendation, or any synonym.",
                    "- recommendation must be exactly one value from FIXED_LABEL_SPACE.",
                    "- probability_distribution must be an object with exactly all 7 labels from FIXED_LABEL_SPACE as keys and numeric probabilities summing to 1.",
                    "- confidence must be an integer from 0 to 100. Do not use decimals such as 0.7.",
                    "- key_variables_used must be a JSON array using only strings from ALLOWED_KEY_VARIABLES_USED_VALUES. Do not use aliases like inflation gap or output gap.",
                    "- policy_path_regime_assessment must be a JSON object, even for non-path theories; use not_applicable fields when path logic is not relevant.",
                    "- This is a theory-agent answer only. Do not output final decision fields, theory weights, contribution shares, memory fields, or training feedback fields.",
                    "- Base the repaired answer on the original theory prompt and current sample, but preserve the exact JSON field names and shape above.",
                ]
            )
            return RenderedTheoryPrompt(
                theory_id=prompt.theory_id,
                system_prompt=prompt.system_prompt,
                user_prompt=repair_user_prompt,
            )

        @staticmethod
        def _debug_excerpt(text: str, limit: int = 900) -> str:
            normalized = " ".join(str(text).split())
            if len(normalized) <= limit:
                return normalized
            return normalized[:limit] + "...<truncated>"

        def run(self, context: TheoryPromptContext) -> TheoryAgentOutput:
            if not isinstance(context, TheoryPromptContext):
                raise TypeError("context must be a TheoryPromptContext instance.")

            prompt = self.render_prompt(context)
            raw_output = self.invoke_backend(prompt=prompt, context=context)
            try:
                return self.parse_output(raw_output=raw_output, context=context)
            except (TypeError, ValueError) as first_exc:
                repair_prompt = self._build_repair_prompt(
                    prompt=prompt,
                    raw_output=raw_output,
                    error_message=str(first_exc),
                    context=context,
                )
                repaired_output = self.invoke_backend(prompt=repair_prompt, context=context)
                try:
                    return self.parse_output(raw_output=repaired_output, context=context)
                except (TypeError, ValueError) as second_exc:
                    raise ValueError(
                        "Theory agent output remained invalid after repair. "
                        f"theory_id={context.theory_id}; "
                        f"decision_date={context.visible_sample.decision_date.isoformat()}; "
                        f"first_error={str(first_exc)}; "
                        f"repair_error={str(second_exc)}; "
                        f"first_raw_excerpt={self._debug_excerpt(raw_output)}; "
                        f"repair_raw_excerpt={self._debug_excerpt(repaired_output)}"
                    ) from second_exc

    def run_all_theory_agents(
        contexts: dict[str, TheoryPromptContext],
        runner: SingleTheoryAgentRunner,
    ) -> dict[str, TheoryAgentOutput]:
        if not isinstance(contexts, dict):
            raise TypeError("contexts must be a dict keyed by theory_id.")
        if not isinstance(runner, SingleTheoryAgentRunner):
            raise TypeError("runner must be a SingleTheoryAgentRunner instance.")

        outputs: dict[str, TheoryAgentOutput] = {}
        for theory_id in THEORY_IDS:
            if theory_id not in contexts:
                raise ValueError(f"Missing TheoryPromptContext for {theory_id}.")
            context = contexts[theory_id]
            if not isinstance(context, TheoryPromptContext):
                raise TypeError(f"contexts[{theory_id}] must be a TheoryPromptContext instance.")
            if context.theory_id != theory_id:
                raise ValueError(f"contexts[{theory_id}].theory_id must equal the dict key.")
            outputs[theory_id] = runner.run(context)
        return outputs


In [ ]:
# 文件：theory_agents/__init__.py
# 产出：导出理论层接口与工具。

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    __all__ = [
        "REQUIRED_OUTPUT_FIELDS",
        "RenderedTheoryPrompt",
        "SingleTheoryAgentRunner",
        "TheoryAgentRunner",
        "TheoryModelBackend",
        "parse_theory_agent_output",
        "render_theory_agent_prompt",
        "run_all_theory_agents",
    ]


## Analysis / Handover

In [ ]:
# 文件：analysis_handover/builder.py
# 产出：产出 AnalysisPayload、DecisionHandoverPacket 和 AnalysisState 的构建逻辑。
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    import json
    from abc import ABC, abstractmethod
    from collections import Counter
    from dataclasses import dataclass
    from typing import Any

    def _validate_analysis_theory_outputs(theory_outputs: dict[str, TheoryAgentOutput]) -> dict[str, TheoryAgentOutput]:
        if not isinstance(theory_outputs, dict):
            raise TypeError("theory_outputs must be a dict.")

        missing = [theory_id for theory_id in THEORY_IDS if theory_id not in theory_outputs]
        extras = sorted(set(theory_outputs) - set(THEORY_IDS))
        if missing or extras:
            raise ValueError(
                f"theory_outputs must cover all registered theories exactly. Missing: {missing}; extras: {extras}."
            )

        normalized: dict[str, TheoryAgentOutput] = {}
        for theory_id in THEORY_IDS:
            output = theory_outputs[theory_id]
            if not isinstance(output, TheoryAgentOutput):
                raise TypeError(f"theory_outputs[{theory_id}] must be a TheoryAgentOutput instance.")
            normalized[theory_id] = output
        return normalized

    def _analysis_confidence_band(confidence: int) -> str:
        if confidence <= 39:
            return "low"
        if confidence <= 59:
            return "lower_mid"
        if confidence <= 79:
            return "upper_mid"
        return "high"

    @dataclass(frozen=True, slots=True)
    class AnalysisHandoverContext:
        sample_context: SampleContextState
        theory_outputs: dict[str, TheoryAgentOutput]

        def __post_init__(self) -> None:
            if not isinstance(self.sample_context, SampleContextState):
                raise TypeError("sample_context must be a SampleContextState instance.")
            object.__setattr__(self, "theory_outputs", _validate_analysis_theory_outputs(self.theory_outputs))

    def build_analysis_handover_context(
        theory_layer_state: TheoryLayerState,
    ) -> AnalysisHandoverContext:
        if not isinstance(theory_layer_state, TheoryLayerState):
            raise TypeError("theory_layer_state must be a TheoryLayerState instance.")
        return AnalysisHandoverContext(
            sample_context=theory_layer_state.sample_context,
            theory_outputs=theory_layer_state.theory_outputs,
        )

    @dataclass(frozen=True, slots=True)
    class RenderedAnalysisHandoverPrompt:
        system_prompt: str
        user_prompt: str

        def __post_init__(self) -> None:
            if not isinstance(self.system_prompt, str) or not self.system_prompt.strip():
                raise ValueError("system_prompt must be a non-empty string.")
            if not isinstance(self.user_prompt, str) or not self.user_prompt.strip():
                raise ValueError("user_prompt must be a non-empty string.")

    class AnalysisHandoverModelBackend(ABC):
        @abstractmethod
        def generate(
            self,
            prompt: RenderedAnalysisHandoverPrompt,
            context: AnalysisHandoverContext,
        ) -> str:
            '''
            Execute one analysis-handover synthesis call and return the raw model response text.
            '''

    class AnalysisHandoverRunner(ABC):
        @abstractmethod
        def run(self, context: AnalysisHandoverContext) -> AnalysisSynthesisOutput:
            '''
            Run one analysis-handover synthesis agent from context to validated structured output.
            '''

    def _render_analysis_theory_outputs(context: AnalysisHandoverContext) -> str:
        payload: dict[str, object] = {}
        for theory_id, output in context.theory_outputs.items():
            payload[theory_id] = {
                "recommendation": output.recommendation.value,
                "probability_distribution": output.probability_distribution.as_ordered_dict(),
                "confidence": output.confidence,
                "primary_policy_reason": output.primary_policy_reason,
                "secondary_policy_reason": output.secondary_policy_reason,
                "key_variables_used": list(output.key_variables_used),
                "theory_explanation": output.theory_explanation,
                "applicability_note": output.applicability_note,
                "risk_note": output.risk_note,
                "policy_path_regime_assessment": output.policy_path_regime_assessment,
            }
        return json.dumps(payload, ensure_ascii=False, indent=2)

    def _render_analysis_output_contract() -> str:
        contract = {
            "reason_summary_by_rule": {
                theory_id: {
                    "primary_policy_reason": "text",
                    "secondary_policy_reason": "text",
                    "theory_explanation": "text",
                }
                for theory_id in THEORY_IDS
            },
            "applicability_summary_by_rule": {
                theory_id: {"applicability_note": "text"}
                for theory_id in THEORY_IDS
            },
            "risk_summary_by_rule": {
                theory_id: {"risk_note": "text", "failure_risk": "text", "shared_driver_warning": "text", "mechanical_middle_warning": "text", "policy_stage_assessment": "text"}
                for theory_id in THEORY_IDS
            },
            "main_disagreement_dimension": "text",
            "secondary_disagreement_dimension": "text or null",
            "high_conflict_flag": False,
            "minority_high_confidence_alert": False,
            "policy_phase_signal": {
                "phase": "near_zlb_waiting / liftoff_start / rapid_hiking_cycle / high_rate_plateau / easing_after_plateau / crisis_cut / normal / uncertain",
                "phase_confidence": "high / medium / low",
                "evidence": "current-sample evidence from theory outputs and authorized path assessments",
                "direction_constraint": "hold_bias / hike_bias / cut_bias / no_constraint / uncertain",
                "magnitude_constraint": "avoid_large / allow_large / prefer_small / prefer_hold / no_constraint / uncertain",
                "anti_overfit_warning": "why this is not a date answer key",
            },
        }
        return json.dumps(contract, ensure_ascii=False, indent=2)

    def render_analysis_handover_prompt(context: AnalysisHandoverContext) -> RenderedAnalysisHandoverPrompt:
        if not isinstance(context, AnalysisHandoverContext):
            raise TypeError("context must be an AnalysisHandoverContext instance.")

        system_prompt = "\n".join(
            [
                "你是层级式多智能体利率决策系统中的分析交接综合Agent。",
                "你的任务不是做最终利率决策。",
                "你的任务是将七个理论Agent的输出压缩、规范化并综合成清晰的交接材料。",
                "你必须保留每个理论自身的立场和理由，不得把它们改写成单一的最终判断。",
                "必须只返回一个JSON对象，不要额外文本。",
            ]
        )

        user_prompt = "\n".join(
            [
                "[任务]",
                "Read all theory-agent outputs for the current sample and produce a structured synthesis handover.",
                "",
                "[Current Sample]",
                f"country: {context.sample_context.country}",
                f"decision_date: {context.sample_context.decision_date.isoformat()}",
                f"sample_id: {context.sample_context.sample_id}",
                "",
                "[Theory Outputs]",
                _render_analysis_theory_outputs(context),
                "",
                "[What You Must Do]",
                "1. Compress each theory's reasons into concise but faithful summaries.",
                "2. Compress each theory's applicability boundary into one clean note.",
                "3. Compress each theory's risk note into one clean note.",
                "4. Identify the main disagreement dimension across theories.",
                "5. Identify a secondary disagreement dimension if one materially exists; otherwise return null.",
                "6. Set high_conflict_flag to true only when the theories show materially strong conflict.",
                "7. Set minority_high_confidence_alert to true only when a minority theory strongly disagrees with high confidence.",
                "8. In each risk_summary_by_rule entry, include policy_stage_assessment: summarize any authorized policy_path_regime_assessment such as liftoff_waiting, liftoff_start, gradual_hiking_cycle, late_hiking_cycle, terminal_rate_plateau, insurance_cut_cycle, crisis_cut, easing_cycle_continuation, or normal_regime. Treat this as diagnostic context, not an answer key.",
                "9. Fill policy_phase_signal as a current-sample diagnostic. It must classify the phase using only theory outputs, authorized policy_path_regime_assessment, and visible macro variables.",
                "10. Phase definitions: near_zlb_waiting = near-zero or lower-bound setting where Taylor pressure may indicate future tightening but not necessarily current action; liftoff_start = first actual tightening from lower bound; rapid_hiking_cycle = already-started tightening with broad inflation/demand support; high_rate_plateau = restrictive rate maintained while evidence does not yet require cuts; easing_after_plateau = high-rate plateau with current evidence supporting a move toward cuts; crisis_cut = acute financial/output stress where large cuts are plausible; normal = no special phase.",
                "11. Set direction_constraint and magnitude_constraint to guide, not decide, the final layer. Use hold_bias for near_zlb_waiting, hike_bias for rapid_hiking_cycle, cut_bias for easing_after_plateau/crisis_cut, no_constraint for normal.",
                "",
                "[硬性约束]",
                "- 不得输出最终推荐标签。",
                "- 不得为决策层计算最终概率。",
                "- 不得把七个理论改写成一个混合理论。",
                "- 每个理论key必须保持原样。",
                "- 严格返回下方JSON结构。",
                "",
                "[冲突诊断要求]",
                "Diagnose whether same-direction theories are driven by the same macro variable; write shared_driver_warning when needed.",
                "Separate direction conflict from magnitude conflict, and identify when static Taylor-type signals are constrained by path-sensitive theories.",
                "Warn the decision layer not to mechanically choose a middle magnitude when hike signals conflict with hold/cut path signals.",
                "When the pattern looks like a continuous interval error risk, say so in main_disagreement_dimension or risk_summary_by_rule: Taylor pressure during liftoff_waiting is usually a future-turn signal, while first_cut/easing_cycle_start means a high-rate plateau may have begun moving into an easing path.",
                "",
                "[输出契约]",
                _render_analysis_output_contract(),
            ]
        )

        return RenderedAnalysisHandoverPrompt(
            system_prompt=system_prompt,
            user_prompt=user_prompt,
        )

    def _require_analysis_dict(payload: Any, field_name: str) -> dict[str, Any]:
        if not isinstance(payload, dict):
            raise TypeError(f"{field_name} must be a JSON object.")
        return payload

    def _require_analysis_bool(payload: Any, field_name: str) -> bool:
        if not isinstance(payload, bool):
            raise TypeError(f"{field_name} must be a boolean.")
        return payload

    def parse_analysis_handover_output(
        raw_output: str,
        context: AnalysisHandoverContext,
    ) -> AnalysisSynthesisOutput:
        if not isinstance(raw_output, str) or not raw_output.strip():
            raise ValueError("raw_output must be a non-empty string.")
        if not isinstance(context, AnalysisHandoverContext):
            raise TypeError("context must be an AnalysisHandoverContext instance.")

        try:
            payload = json.loads(raw_output)
        except json.JSONDecodeError as exc:
            raise ValueError(f"Analysis handover output must be valid JSON: {exc}") from exc
        payload = _require_analysis_dict(payload, "analysis_handover_output")

        return AnalysisSynthesisOutput(
            reason_summary_by_rule=_require_analysis_dict(payload["reason_summary_by_rule"], "reason_summary_by_rule"),
            applicability_summary_by_rule=_require_analysis_dict(
                payload["applicability_summary_by_rule"],
                "applicability_summary_by_rule",
            ),
            risk_summary_by_rule=_require_analysis_dict(payload["risk_summary_by_rule"], "risk_summary_by_rule"),
            main_disagreement_dimension=str(payload["main_disagreement_dimension"]),
            secondary_disagreement_dimension=(
                None
                if payload.get("secondary_disagreement_dimension") is None
                else str(payload["secondary_disagreement_dimension"])
            ),
            high_conflict_flag=_require_analysis_bool(payload["high_conflict_flag"], "high_conflict_flag"),
            minority_high_confidence_alert=_require_analysis_bool(
                payload["minority_high_confidence_alert"],
                "minority_high_confidence_alert",
            ),
            policy_phase_signal=_require_analysis_dict(
                payload.get("policy_phase_signal", {}),
                "policy_phase_signal",
            ),
        )

    @dataclass(slots=True)
    class SingleAnalysisHandoverRunner(AnalysisHandoverRunner):
        backend: AnalysisHandoverModelBackend

        def __post_init__(self) -> None:
            if not isinstance(self.backend, AnalysisHandoverModelBackend):
                raise TypeError("backend must be an AnalysisHandoverModelBackend instance.")

        def render_prompt(self, context: AnalysisHandoverContext) -> RenderedAnalysisHandoverPrompt:
            if not isinstance(context, AnalysisHandoverContext):
                raise TypeError("context must be an AnalysisHandoverContext instance.")
            return render_analysis_handover_prompt(context)

        def invoke_backend(
            self,
            prompt: RenderedAnalysisHandoverPrompt,
            context: AnalysisHandoverContext,
        ) -> str:
            if not isinstance(prompt, RenderedAnalysisHandoverPrompt):
                raise TypeError("prompt must be a RenderedAnalysisHandoverPrompt instance.")
            if not isinstance(context, AnalysisHandoverContext):
                raise TypeError("context must be an AnalysisHandoverContext instance.")
            return self.backend.generate(prompt=prompt, context=context)

        def parse_output(
            self,
            raw_output: str,
            context: AnalysisHandoverContext,
        ) -> AnalysisSynthesisOutput:
            if not isinstance(raw_output, str):
                raise TypeError("raw_output must be a string.")
            if not isinstance(context, AnalysisHandoverContext):
                raise TypeError("context must be an AnalysisHandoverContext instance.")
            return parse_analysis_handover_output(raw_output=raw_output, context=context)

        def run(self, context: AnalysisHandoverContext) -> AnalysisSynthesisOutput:
            if not isinstance(context, AnalysisHandoverContext):
                raise TypeError("context must be an AnalysisHandoverContext instance.")
            prompt = self.render_prompt(context)
            raw_output = self.invoke_backend(prompt=prompt, context=context)
            return self.parse_output(raw_output=raw_output, context=context)

    def _build_analysis_label_support_table(theory_outputs: dict[str, TheoryAgentOutput]) -> dict[str, dict[str, object]]:
        table: dict[str, dict[str, object]] = {}
        for label in LABEL_ORDER:
            supporting_theories = [
                theory_id
                for theory_id, output in theory_outputs.items()
                if output.recommendation.value == label.value
            ]
            probability_values = [
                output.probability_distribution.probability_for(label)
                for output in theory_outputs.values()
            ]
            table[label.value] = {
                "supporting_theories": supporting_theories,
                "recommendation_count": len(supporting_theories),
                "average_probability": sum(probability_values) / len(probability_values),
                "max_probability": max(probability_values),
                "probability_support_sum": sum(probability_values),
            }
        return table

    def _build_analysis_agent_position_table(theory_outputs: dict[str, TheoryAgentOutput]) -> dict[str, dict[str, object]]:
        return {
            theory_id: {
                "recommendation": output.recommendation.value,
                "primary_policy_reason": output.primary_policy_reason,
                "secondary_policy_reason": output.secondary_policy_reason,
                "key_variables_used": list(output.key_variables_used),
            }
            for theory_id, output in theory_outputs.items()
        }

    def _build_analysis_agent_distribution_table(theory_outputs: dict[str, TheoryAgentOutput]) -> dict[str, dict[str, float]]:
        return {
            theory_id: output.probability_distribution.as_ordered_dict()
            for theory_id, output in theory_outputs.items()
        }

    def _build_analysis_agent_confidence_table(theory_outputs: dict[str, TheoryAgentOutput]) -> dict[str, dict[str, object]]:
        return {
            theory_id: {
                "confidence": output.confidence,
                "confidence_band": _analysis_confidence_band(output.confidence),
            }
            for theory_id, output in theory_outputs.items()
        }

    def _analysis_fallback_synthesis_output(theory_outputs: dict[str, TheoryAgentOutput]) -> AnalysisSynthesisOutput:
        majority_label = Counter(output.recommendation.value for output in theory_outputs.values()).most_common(1)[0][0]
        minority_high_confidence_alert = any(
            output.recommendation.value != majority_label and output.confidence >= 80
            for output in theory_outputs.values()
        )
        return AnalysisSynthesisOutput(
            reason_summary_by_rule={
                theory_id: {
                    "primary_policy_reason": output.primary_policy_reason,
                    "secondary_policy_reason": output.secondary_policy_reason,
                    "theory_explanation": output.theory_explanation,
                }
                for theory_id, output in theory_outputs.items()
            },
            applicability_summary_by_rule={
                theory_id: {"applicability_note": output.applicability_note}
                for theory_id, output in theory_outputs.items()
            },
            risk_summary_by_rule={
                theory_id: {"risk_note": output.risk_note}
                for theory_id, output in theory_outputs.items()
            },
            main_disagreement_dimension="label_direction_or_magnitude",
            secondary_disagreement_dimension=None,
            high_conflict_flag=False,
            minority_high_confidence_alert=minority_high_confidence_alert,
            policy_phase_signal={
                "phase": "uncertain",
                "phase_confidence": "low",
                "evidence": "Fallback synthesis did not call the analysis model.",
                "direction_constraint": "uncertain",
                "magnitude_constraint": "uncertain",
                "anti_overfit_warning": "Fallback signal must not be treated as an answer key.",
            },
        )

    @dataclass(slots=True)
    class AnalysisHandoverStateBuilder:
        def build_payload(
            self,
            theory_outputs: dict[str, TheoryAgentOutput],
            synthesis_output: AnalysisSynthesisOutput | None,
        ) -> AnalysisPayload:
            normalized_outputs = _validate_analysis_theory_outputs(theory_outputs)
            resolved_synthesis_output = (
                _analysis_fallback_synthesis_output(normalized_outputs)
                if synthesis_output is None
                else synthesis_output
            )
            if not isinstance(resolved_synthesis_output, AnalysisSynthesisOutput):
                raise TypeError("synthesis_output must be an AnalysisSynthesisOutput instance or None.")

            return AnalysisPayload(
                label_support_table=_build_analysis_label_support_table(normalized_outputs),
                agent_position_table=_build_analysis_agent_position_table(normalized_outputs),
                agent_distribution_table=_build_analysis_agent_distribution_table(normalized_outputs),
                agent_confidence_table=_build_analysis_agent_confidence_table(normalized_outputs),
                reason_summary_by_rule=resolved_synthesis_output.reason_summary_by_rule,
                applicability_summary_by_rule=resolved_synthesis_output.applicability_summary_by_rule,
                risk_summary_by_rule=resolved_synthesis_output.risk_summary_by_rule,
                main_disagreement_dimension=resolved_synthesis_output.main_disagreement_dimension,
                secondary_disagreement_dimension=resolved_synthesis_output.secondary_disagreement_dimension,
                high_conflict_flag=resolved_synthesis_output.high_conflict_flag,
                minority_high_confidence_alert=resolved_synthesis_output.minority_high_confidence_alert,
                policy_phase_signal=resolved_synthesis_output.policy_phase_signal,
            )

        def build_decision_handover_packet(
            self,
            payload: AnalysisPayload,
        ) -> DecisionHandoverPacket:
            if not isinstance(payload, AnalysisPayload):
                raise TypeError("payload must be an AnalysisPayload instance.")

            return DecisionHandoverPacket(
                label_support_table=payload.label_support_table,
                agent_position_table=payload.agent_position_table,
                agent_distribution_table=payload.agent_distribution_table,
                agent_confidence_table=payload.agent_confidence_table,
                reason_summary_by_rule=payload.reason_summary_by_rule,
                applicability_summary_by_rule=payload.applicability_summary_by_rule,
                risk_summary_by_rule=payload.risk_summary_by_rule,
                main_disagreement_dimension=payload.main_disagreement_dimension,
                secondary_disagreement_dimension=payload.secondary_disagreement_dimension,
                high_conflict_flag=payload.high_conflict_flag,
                minority_high_confidence_alert=payload.minority_high_confidence_alert,
                policy_phase_signal=payload.policy_phase_signal,
            )

        def build_analysis_state(
            self,
            sample_context: SampleContextState,
            theory_outputs: dict[str, TheoryAgentOutput],
            synthesis_output: AnalysisSynthesisOutput | None,
        ) -> AnalysisState:
            if not isinstance(sample_context, SampleContextState):
                raise TypeError("sample_context must be a SampleContextState instance.")
            normalized_outputs = _validate_analysis_theory_outputs(theory_outputs)
            payload = self.build_payload(normalized_outputs, synthesis_output=synthesis_output)
            handover_packet = self.build_decision_handover_packet(payload)
            return AnalysisState(
                sample_context=sample_context,
                theory_outputs=normalized_outputs,
                analysis_payload=payload,
                decision_handover_packet=handover_packet,
            )


In [ ]:
# 文件：analysis_handover/__init__.py
# 产出：导出分析交接层构建器接口。
if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    __all__ = [
        "AnalysisHandoverContext",
        "AnalysisHandoverModelBackend",
        "AnalysisHandoverRunner",
        "AnalysisHandoverStateBuilder",
        "RenderedAnalysisHandoverPrompt",
        "SingleAnalysisHandoverRunner",
        "build_analysis_handover_context",
        "parse_analysis_handover_output",
        "render_analysis_handover_prompt",
    ]


## Decision Agent: Training-Set Textual Memory Informed


In [ ]:
# file: decision_agent/context.py
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from dataclasses import dataclass
    from datetime import date
    from typing import Any

    DEFAULT_TEXTUAL_THEORY_MEMORY: dict[str, Any] = {
        "schema_version": "system_c_empty_textual_memory_v1",
        "memory_status": "empty_before_training",
        "global_lessons": [],
        "normal_regime_lessons": [],
        "special_regime_lessons": [],
        "theory_specific_applicability": {theory_id: [] for theory_id in THEORY_IDS},
        "conflict_resolution_lessons": [],
        "conditional_applicability_rules": [],
        "uncertainty_lessons": [],
        "do_not_use_rules": ["不要把文字记忆当成固定数值权重。", "预测阶段不得使用真实标签。"],
    }
    CURRENT_TEXTUAL_THEORY_MEMORY: dict[str, Any] = dict(DEFAULT_TEXTUAL_THEORY_MEMORY)

    def validate_textual_theory_memory(field_name: str, value: dict[str, Any]) -> dict[str, Any]:
        if not isinstance(value, dict):
            raise TypeError(f"{field_name} must be a dict.")
        normalized = dict(value)
        normalized.pop("source_evidence_summary", None)
        normalized.setdefault("global_lessons", [])
        normalized.setdefault("normal_regime_lessons", [])
        normalized.setdefault("special_regime_lessons", [])
        normalized.setdefault("theory_specific_applicability", {theory_id: [] for theory_id in THEORY_IDS})
        normalized.setdefault("conflict_resolution_lessons", [])
        normalized.setdefault("conditional_applicability_rules", [])
        normalized.setdefault("uncertainty_lessons", [])
        normalized.setdefault("do_not_use_rules", [])
        theory_memory = normalized.get("theory_specific_applicability")
        if not isinstance(theory_memory, dict):
            raise TypeError(f"{field_name}.theory_specific_applicability must be a dict.")
        for theory_id in THEORY_IDS:
            theory_memory.setdefault(theory_id, [])
            if not isinstance(theory_memory[theory_id], list):
                raise TypeError(f"{field_name}.theory_specific_applicability[{theory_id}] must be a list.")
        normalized["theory_specific_applicability"] = {theory_id: list(theory_memory[theory_id]) for theory_id in THEORY_IDS}
        conditional_rules = normalized.get("conditional_applicability_rules")
        if not isinstance(conditional_rules, list):
            raise TypeError(f"{field_name}.conditional_applicability_rules must be a list.")
        normalized_rules: list[dict[str, Any]] = []
        allowed_directions = {"cut", "hold", "hike", "uncertain"}
        allowed_reliability = {"high", "medium", "low"}
        for index, raw_rule in enumerate(conditional_rules, 1):
            if not isinstance(raw_rule, dict):
                raise TypeError(f"{field_name}.conditional_applicability_rules[{index}] must be a dict.")
            rule = dict(raw_rule)
            rule.setdefault("rule_id", f"R{index:03d}")
            rule.setdefault("conflict_pattern", "")
            rule.setdefault("observable_conditions", [])
            rule.setdefault("preferred_direction", "uncertain")
            rule.setdefault("preferred_theory_family", [])
            rule.setdefault("downweighted_theory_family", [])
            rule.setdefault("magnitude_guidance", "")
            rule.setdefault("boundary_conditions", [])
            rule.setdefault("training_evidence_summary", "")
            rule.setdefault("reliability_level", "medium")
            if str(rule["preferred_direction"]) not in allowed_directions:
                rule["preferred_direction"] = "uncertain"
            if str(rule["reliability_level"]) not in allowed_reliability:
                rule["reliability_level"] = "medium"
            for list_field in ["observable_conditions", "preferred_theory_family", "downweighted_theory_family", "boundary_conditions"]:
                value = rule.get(list_field, [])
                if isinstance(value, str):
                    value = [value]
                if not isinstance(value, list):
                    value = []
                rule[list_field] = [str(item).strip() for item in value if str(item).strip()]
            normalized_rules.append(rule)
        normalized["conditional_applicability_rules"] = normalized_rules
        return normalized

    @dataclass(frozen=True, slots=True)
    class DecisionPromptContext:
        country: str
        decision_date: date
        theory_outputs_full: dict[str, TheoryAgentOutput]
        decision_handover_packet: DecisionHandoverPacket
        textual_theory_applicability_memory: dict[str, Any]

        def __post_init__(self) -> None:
            if not isinstance(self.country, str) or not self.country.strip():
                raise ValueError("country must be a non-empty string.")
            if not isinstance(self.decision_date, date):
                raise TypeError("decision_date must be a datetime.date instance.")
            if set(self.theory_outputs_full) != set(THEORY_IDS):
                raise ValueError("theory_outputs_full must cover all registered theories exactly.")
            for theory_id, output in self.theory_outputs_full.items():
                if not isinstance(output, TheoryAgentOutput):
                    raise TypeError(f"theory_outputs_full[{theory_id}] must be a TheoryAgentOutput instance.")
            if not isinstance(self.decision_handover_packet, DecisionHandoverPacket):
                raise TypeError("decision_handover_packet must not be None.")
            object.__setattr__(self, "textual_theory_applicability_memory", validate_textual_theory_memory("textual_theory_applicability_memory", self.textual_theory_applicability_memory))

    def set_current_textual_theory_memory(memory: dict[str, Any]) -> dict[str, Any]:
        global CURRENT_TEXTUAL_THEORY_MEMORY
        CURRENT_TEXTUAL_THEORY_MEMORY = validate_textual_theory_memory("CURRENT_TEXTUAL_THEORY_MEMORY", memory)
        return CURRENT_TEXTUAL_THEORY_MEMORY

    def get_current_textual_theory_memory() -> dict[str, Any]:
        return validate_textual_theory_memory("CURRENT_TEXTUAL_THEORY_MEMORY", CURRENT_TEXTUAL_THEORY_MEMORY)

    def build_decision_prompt_context(analysis_state: AnalysisState) -> DecisionPromptContext:
        if not isinstance(analysis_state, AnalysisState):
            raise TypeError("analysis_state must be an AnalysisState instance.")
        if analysis_state.decision_handover_packet is None:
            raise ValueError("analysis_state.decision_handover_packet must not be None.")
        return DecisionPromptContext(
            country=analysis_state.sample_context.country,
            decision_date=analysis_state.sample_context.decision_date,
            theory_outputs_full=analysis_state.theory_outputs,
            decision_handover_packet=analysis_state.decision_handover_packet,
            textual_theory_applicability_memory=get_current_textual_theory_memory(),
        )


In [ ]:
# 文件：decision_agent/base.py
# 产出：产出决策层抽象基类与渲染后 prompt 结构。
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from abc import ABC, abstractmethod
    from dataclasses import dataclass

    @dataclass(frozen=True, slots=True)
    class RenderedDecisionPrompt:
        system_prompt: str
        user_prompt: str

        def __post_init__(self) -> None:
            if not isinstance(self.system_prompt, str) or not self.system_prompt.strip():
                raise ValueError("system_prompt must be a non-empty string.")
            if not isinstance(self.user_prompt, str) or not self.user_prompt.strip():
                raise ValueError("user_prompt must be a non-empty string.")

    class DecisionModelBackend(ABC):
        @abstractmethod
        def generate(
            self,
            prompt: RenderedDecisionPrompt,
            context: DecisionPromptContext,
        ) -> str:
            """
            Execute one decision-agent call and return the raw model response text.
            """

    class DecisionAgentRunner(ABC):
        @abstractmethod
        def run(self, context: DecisionPromptContext) -> DecisionAgentOutput:
            """
            Run one decision agent from decision context to validated structured output.
            """


In [ ]:
# file: decision_agent/output_parser.py
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    import ast
    import json
    import re
    from typing import Any

    REQUIRED_OUTPUT_FIELDS = (
        "final_recommendation",
        "final_probability_distribution",
        "final_confidence",
        "final_explanation",
        "dominant_rule_family",
        "supporting_rule_families",
        "theory_weight_profile",
        "theory_weight_rationale",
        "rule_contribution_shares",
        "memory_usage_summary",
        "matched_memory_lessons",
        "matched_conditional_rules",
        "rule_application_summary",
        "direction_decision",
        "magnitude_decision_rationale",
        "first_contributor_rule",
        "current_sample_reasoning",
        "remaining_uncertainty",
        "decision_path",
    )

    RULE_FAMILY_ALIASES: dict[str, str] = {
        "status_quo": "interest_rate_smoothing", "status quo": "interest_rate_smoothing",
        "interest rate smoothing": "interest_rate_smoothing", "interest-rate smoothing": "interest_rate_smoothing",
        "interest rate smoothing rule": "interest_rate_smoothing", "hold": "interest_rate_smoothing",
        "hold_rule": "interest_rate_smoothing", "smoothing": "interest_rate_smoothing",
        "open_economy": "open_economy_rule", "open economy": "open_economy_rule",
        "open economy rule": "open_economy_rule", "external_balance": "open_economy_rule",
        "financial_stability": "financial_stability_rule", "financial stability": "financial_stability_rule",
        "financial stability rule": "financial_stability_rule", "new_keynesian": "new_keynesian_optimal_rule",
        "new keynesian": "new_keynesian_optimal_rule", "new keynesian optimal rule": "new_keynesian_optimal_rule",
        "forward_looking": "forward_looking_taylor", "forward-looking": "forward_looking_taylor",
        "forward looking": "forward_looking_taylor", "forward looking taylor": "forward_looking_taylor",
        "forward looking taylor rule": "forward_looking_taylor", "standard": "standard_taylor",
        "standard taylor": "standard_taylor", "standard taylor rule": "standard_taylor",
    }

    def _strip_markdown_code_fence(text: str) -> str:
        stripped = text.strip()
        if stripped.startswith("```") and stripped.endswith("```"):
            lines = stripped.splitlines()
            if len(lines) >= 3:
                return "\n".join(lines[1:-1]).strip()
        return stripped

    def _extract_json_object_candidate(text: str) -> str:
        start = text.find("{")
        if start < 0:
            return text
        depth = 0
        in_string = False
        escaped = False
        for index in range(start, len(text)):
            char = text[index]
            if in_string:
                if escaped:
                    escaped = False
                elif char == "\\":
                    escaped = True
                elif char == '"':
                    in_string = False
                continue
            if char == '"':
                in_string = True
            elif char == "{":
                depth += 1
            elif char == "}":
                depth -= 1
                if depth == 0:
                    return text[start : index + 1]
        return text[start:].strip()

    def _parse_json_object(raw_output: str) -> dict[str, Any]:
        candidate = _extract_json_object_candidate(_strip_markdown_code_fence(raw_output))
        try:
            parsed = json.loads(candidate)
        except json.JSONDecodeError:
            parsed = ast.literal_eval(candidate)
        if not isinstance(parsed, dict):
            raise ValueError("Decision output must be a JSON object.")
        return parsed

    def _normalize_rule_family(value: Any) -> str:
        if not isinstance(value, str):
            raise TypeError("rule family must be a string.")
        normalized = value.strip()
        if normalized in THEORY_IDS:
            return normalized
        key = re.sub(r"\s+", " ", normalized.replace("_", " ").lower())
        if key in RULE_FAMILY_ALIASES:
            return RULE_FAMILY_ALIASES[key]
        compact_key = normalized.strip().lower()
        if compact_key in RULE_FAMILY_ALIASES:
            return RULE_FAMILY_ALIASES[compact_key]
        raise ValueError(f"Unknown rule family {value!r}.")

    def _normalize_supporting(value: Any, dominant_rule_family: str) -> list[str]:
        if value is None:
            return []
        if isinstance(value, str):
            raw_items = [item.strip() for item in re.split(r"[,;]", value) if item.strip()]
        elif isinstance(value, (list, tuple)):
            raw_items = list(value)
        else:
            raise TypeError("supporting_rule_families must be a list or string.")
        result: list[str] = []
        for item in raw_items:
            theory_id = _normalize_rule_family(item)
            if theory_id != dominant_rule_family and theory_id not in result:
                result.append(theory_id)
        return result

    def _normalize_text_list(value: Any) -> list[str]:
        if value is None:
            return []
        if isinstance(value, str):
            return [value.strip()] if value.strip() else []
        if isinstance(value, (list, tuple)):
            return [str(item).strip() for item in value if str(item).strip()]
        raise TypeError("matched_memory_lessons must be a list or string.")

    def _normalize_payload(payload: dict[str, Any], context: DecisionPromptContext) -> dict[str, Any]:
        normalized = dict(payload)
        missing = [field for field in REQUIRED_OUTPUT_FIELDS if field not in normalized]
        if missing:
            raise ValueError(f"Decision output is missing required fields: {missing}.")
        extra = sorted(set(normalized) - set(REQUIRED_OUTPUT_FIELDS))
        if extra:
            raise ValueError(f"Decision output contains unexpected fields: {extra}.")
        normalized["dominant_rule_family"] = _normalize_rule_family(normalized["dominant_rule_family"])
        normalized["supporting_rule_families"] = _normalize_supporting(normalized["supporting_rule_families"], dominant_rule_family=normalized["dominant_rule_family"])
        normalized["theory_weight_profile"] = validate_theory_weight_profile("theory_weight_profile", normalized["theory_weight_profile"])
        normalized["rule_contribution_shares"] = validate_theory_weight_profile("rule_contribution_shares", normalized["rule_contribution_shares"])
        normalized["matched_memory_lessons"] = _normalize_text_list(normalized["matched_memory_lessons"])
        normalized["matched_conditional_rules"] = _normalize_text_list(normalized["matched_conditional_rules"])
        normalized["rule_application_summary"] = str(normalized["rule_application_summary"]).strip()
        if not normalized["rule_application_summary"]:
            raise ValueError("rule_application_summary must not be empty.")
        normalized["direction_decision"] = str(normalized["direction_decision"]).strip()
        if normalized["direction_decision"] not in {"cut", "hold", "hike", "uncertain"}:
            raise ValueError("direction_decision must be one of cut, hold, hike, uncertain.")
        normalized["magnitude_decision_rationale"] = str(normalized["magnitude_decision_rationale"]).strip()
        if not normalized["magnitude_decision_rationale"]:
            raise ValueError("magnitude_decision_rationale must not be empty.")
        normalized["first_contributor_rule"] = _normalize_rule_family(normalized["first_contributor_rule"])
        top_share = max(normalized["rule_contribution_shares"].values())
        top_theories = {theory_id for theory_id, share in normalized["rule_contribution_shares"].items() if abs(share - top_share) <= 1e-9}
        if normalized["first_contributor_rule"] not in top_theories:
            raise ValueError("first_contributor_rule must be one of the highest-share theories in rule_contribution_shares.")
        normalized["decision_path"] = "training_textual_memory_informed_aggregation"
        return normalized

    def parse_decision_agent_output(raw_output: str, context: DecisionPromptContext) -> DecisionAgentOutput:
        if not isinstance(context, DecisionPromptContext):
            raise TypeError("context must be a DecisionPromptContext instance.")
        payload = _normalize_payload(_parse_json_object(raw_output), context=context)
        probability_distribution = ProbabilityDistribution(probabilities=payload["final_probability_distribution"])
        return DecisionAgentOutput(
            final_recommendation=payload["final_recommendation"],
            final_probability_distribution=probability_distribution,
            final_confidence=int(payload["final_confidence"]),
            final_explanation=str(payload["final_explanation"]),
            dominant_rule_family=payload["dominant_rule_family"],
            supporting_rule_families=payload["supporting_rule_families"],
            theory_weight_profile=payload["theory_weight_profile"],
            theory_weight_rationale=str(payload["theory_weight_rationale"]),
            rule_contribution_shares=payload["rule_contribution_shares"],
            memory_usage_summary=str(payload["memory_usage_summary"]),
            matched_memory_lessons=payload["matched_memory_lessons"],
            matched_conditional_rules=payload["matched_conditional_rules"],
            rule_application_summary=str(payload["rule_application_summary"]),
            direction_decision=str(payload["direction_decision"]),
            magnitude_decision_rationale=str(payload["magnitude_decision_rationale"]),
            first_contributor_rule=payload["first_contributor_rule"],
            current_sample_reasoning=str(payload["current_sample_reasoning"]),
            remaining_uncertainty=str(payload["remaining_uncertainty"]),
            decision_path=payload["decision_path"],
        )


In [ ]:
# file: decision_agent/prompt_renderer.py
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    import json
    from typing import Any

    def _render_scalar(value: Any) -> str:
        if isinstance(value, (dict, list, tuple, str, int, float, bool)) or value is None:
            if isinstance(value, (dict, list, tuple)):
                return json.dumps(value, ensure_ascii=False, indent=2)
            return json.dumps(value, ensure_ascii=False)
        return repr(value)

    def _ordered_block_lines(block_name: str, payload: dict[str, Any]) -> list[str]:
        lines = [f"{block_name}:"]
        if not payload:
            lines.append("  - <empty>")
            return lines
        for key in sorted(payload):
            lines.append(f"  - {key}: {_render_scalar(payload[key])}")
        return lines

    def _render_theory_outputs(context: DecisionPromptContext) -> str:
        payload: dict[str, object] = {}
        for theory_id, output in context.theory_outputs_full.items():
            payload[theory_id] = {
                "recommendation": output.recommendation.value,
                "probability_distribution": output.probability_distribution.as_ordered_dict(),
                "confidence": output.confidence,
                "primary_policy_reason": output.primary_policy_reason,
                "secondary_policy_reason": output.secondary_policy_reason,
                "key_variables_used": list(output.key_variables_used),
                "theory_explanation": output.theory_explanation,
                "applicability_note": output.applicability_note,
                "risk_note": output.risk_note,
                "policy_path_regime_assessment": output.policy_path_regime_assessment,
            }
        return json.dumps(payload, ensure_ascii=False, indent=2)

    def _render_handover_packet(context: DecisionPromptContext) -> str:
        packet = context.decision_handover_packet
        lines: list[str] = [f"country: {context.country}", f"decision_date: {context.decision_date.isoformat()}"]
        lines.extend(_ordered_block_lines("label_support_table", packet.label_support_table))
        lines.extend(_ordered_block_lines("agent_position_table", packet.agent_position_table))
        lines.extend(_ordered_block_lines("agent_distribution_table", packet.agent_distribution_table))
        lines.extend(_ordered_block_lines("agent_confidence_table", packet.agent_confidence_table))
        lines.extend(_ordered_block_lines("reason_summary_by_rule", packet.reason_summary_by_rule))
        lines.extend(_ordered_block_lines("applicability_summary_by_rule", packet.applicability_summary_by_rule))
        lines.extend(_ordered_block_lines("risk_summary_by_rule", packet.risk_summary_by_rule))
        lines.append(f"main_disagreement_dimension: {packet.main_disagreement_dimension}")
        lines.append(f"secondary_disagreement_dimension: {_render_scalar(packet.secondary_disagreement_dimension)}")
        lines.append(f"high_conflict_flag: {packet.high_conflict_flag}")
        lines.append(f"minority_high_confidence_alert: {packet.minority_high_confidence_alert}")
        lines.append(f"policy_phase_signal: {_render_scalar(packet.policy_phase_signal)}")
        return "\n".join(lines)

    def _render_decision_output_contract(context: DecisionPromptContext) -> str:
        distribution_template = {label.value: 0.0 for label in LABEL_ORDER}
        contract = {
            "final_recommendation": "固定7类标签中的一个",
            "final_probability_distribution": distribution_template,
            "final_confidence": "0到100之间的整数",
            "final_explanation": "text",
            "dominant_rule_family": "七个理论ID之一",
            "supporting_rule_families": ["zero or more of the valid theory ids, excluding dominant_rule_family"],
            "theory_weight_profile": {theory_id: 0.0 for theory_id in THEORY_IDS},
            "theory_weight_rationale": "用中文说明当前样本为何更强调某些理论；必须引用结构化条件规则的适用或不适用理由。",
            "rule_contribution_shares": {theory_id: 0.0 for theory_id in THEORY_IDS},
            "memory_usage_summary": "用中文说明冻结文字记忆如何被使用，或为何当前样本不适合由记忆主导。",
            "matched_memory_lessons": ["匹配到的一般记忆经验条目的简短中文引用；如无匹配则返回空列表"],
            "matched_conditional_rules": ["匹配到的conditional_applicability_rules.rule_id；如无匹配则返回空列表"],
            "rule_application_summary": "逐条说明匹配到的结构化条件规则如何影响方向和幅度；若规则不适用，说明触发了哪些boundary_conditions。",
            "direction_decision": "cut / hold / hike / uncertain",
            "magnitude_decision_rationale": "先确定方向后再确定7类幅度的中文理由；必须说明是否服从policy_phase_signal的direction_constraint和magnitude_constraint。",
            "first_contributor_rule": "七个理论ID之一 with the highest contribution share",
            "current_sample_reasoning": "用中文说明当前证据、理论冲突和结构化记忆规则引导下的推理过程。",
            "remaining_uncertainty": "text",
            "decision_path": "training_textual_memory_informed_aggregation",
        }
        return json.dumps(contract, ensure_ascii=False, indent=2)

    def render_decision_agent_prompt(context: DecisionPromptContext) -> RenderedDecisionPrompt:
        if not isinstance(context, DecisionPromptContext):
            raise TypeError("context must be a DecisionPromptContext instance.")
        ordered_labels = ", ".join(label.value for label in LABEL_ORDER)
        theory_ids = ", ".join(THEORY_IDS)
        memory = json.dumps(context.textual_theory_applicability_memory, ensure_ascii=False, indent=2)
        system_prompt = "\n".join([
            "你是层级式多智能体利率决议预测框架中的最终决策Agent。",
            "这是系统C：训练集文字化理论适用性记忆。",
            "七个理论Agent和Analysis层已经给出当前样本证据；你只负责最终聚合。",
            "冻结文字记忆只能来自训练集反思，不能被当成测试期答案库、日期模板或固定数值权重。",
            "预测阶段不得更新文字记忆，也不得使用真实标签。",
            "只返回一个JSON对象，不要额外文字。",
        ])
        user_prompt = "\n".join([
            "[任务]",
            "为当前样本输出最终利率变动标签预测。",
            "你必须同时使用当前样本的七个理论输出、Analysis层交接信息和冻结文字理论适用性记忆。",
            "冻结记忆中的conditional_applicability_rules是训练集抽象出的条件适用规则；它们用于判断‘在当前理论冲突结构下应更信哪些理论’，不是答案库。",
            "不得按年份、测试期阶段标签或历史真实结果推断答案。",
            "",
            "[决策流程]",
            "1. 先识别当前样本的理论冲突结构：哪些理论支持cut/hold/hike，哪些理论只是共享同一驱动变量。",
            "2. 从conditional_applicability_rules中匹配可观察条件和冲突模式；只允许匹配当前理论输出、Analysis交接信息和可见宏观变量能支持的规则。",
            "3. 对每条匹配规则检查boundary_conditions；边界条件触发时不得机械套用该规则。",
            "4. 先确定direction_decision：cut、hold、hike或uncertain。",
            "5. 再在方向确定后选择7类幅度标签；幅度选择应由路径敏感理论、金融稳定风险、就业/通胀冲突和结构化规则共同约束。",
            "6. 必须读取Analysis层的policy_phase_signal：先判断它是否由当前样本证据充分支持，再决定是否服从其direction_constraint和magnitude_constraint。",
            "7. 如果最终方向或幅度违背policy_phase_signal，必须在rule_application_summary和magnitude_decision_rationale中给出强证据理由。",
            "8. 如果结构化规则和当前样本强证据冲突，以当前样本证据为准，但必须在rule_application_summary中解释。",
            "",
            "[重要原则]",
            "- 文字记忆不是固定数值权重；theory_weight_profile是当前样本下的理论侧重结构。",
            "- rule_contribution_shares是最终标签的结果侧归因，不是训练得到的长期权重。",
            "- Same-direction theories are not automatically independent evidence; downweight shared-driver clusters.",
            "- When static Taylor-type rules conflict with path-sensitive rules, do not mechanically choose a middle magnitude; judge theory applicability first.",
            "- The decision layer may only cite path judgments produced by authorized theory agents or Analysis handover; it must not reconstruct raw policy-rate history itself.",
            "- policy_phase_signal is a current-sample diagnostic constraint, not a date template. Use it to prevent repeated phase errors: near_zlb_waiting should resist premature hikes; rapid_hiking_cycle should allow large hikes; easing_after_plateau should resist inertial 0bp when cut evidence is strong.",
            "- Employment-risk, inflation-pressure, financial-stability and policy-path arguments must be adjudicated as theory conflicts, not as date-based templates.",
            "",
            "[固定标签空间]", ordered_labels, "",
            "[合法理论ID]", theory_ids, "",
            "[冻结文字化理论适用性记忆]", memory, "",
            "[七个理论Agent原始输出]", _render_theory_outputs(context), "",
            "[Analysis层交接信息]", _render_handover_packet(context), "",
            "[输出契约]", _render_decision_output_contract(context), "",
            "[硬性约束]",
            "- final_recommendation must match the highest-probability label in final_probability_distribution.",
            "- final_probability_distribution must cover all 7 labels and sum to 1.",
            "- theory_weight_profile must cover all valid theory ids, every value must be >= 0, and the sum must be 1.",
            "- rule_contribution_shares must cover all valid theory ids, every value must be >= 0, and the sum must be 1.",
            "- matched_conditional_rules must contain only rule_id values from conditional_applicability_rules; return [] if no rule fits.",
            "- rule_application_summary must explicitly say how matched rules changed or did not change the direction/magnitude choice.",
            "- direction_decision must be one of cut, hold, hike, uncertain.",
            "- rule_application_summary must explicitly reference policy_phase_signal and explain whether it was followed or overridden.",
            "- Decide direction first, then magnitude; do not mechanically average to a middle magnitude.",
            "- Treat shared-driver same-direction theories as correlated evidence, not independent votes.",
            "- first_contributor_rule must be one of the highest-share theories in rule_contribution_shares.",
            "- dominant_rule_family and supporting_rule_families must use only the valid theory ids.",
            "- supporting_rule_families must not contain dominant_rule_family.",
            "- final_confidence must be an 0到100之间的整数.",
            "- decision_path must be training_textual_memory_informed_aggregation.",
        ])
        return RenderedDecisionPrompt(system_prompt=system_prompt, user_prompt=user_prompt)


In [ ]:
# file: decision_agent/runner.py
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from dataclasses import dataclass

    @dataclass(slots=True)
    class SingleDecisionAgentRunner(DecisionAgentRunner):
        backend: DecisionModelBackend

        def __post_init__(self) -> None:
            if not isinstance(self.backend, DecisionModelBackend):
                raise TypeError("backend must be a DecisionModelBackend instance.")

        def render_prompt(self, context: DecisionPromptContext) -> RenderedDecisionPrompt:
            if not isinstance(context, DecisionPromptContext):
                raise TypeError("context must be a DecisionPromptContext instance.")
            return render_decision_agent_prompt(context)

        def invoke_backend(
            self,
            prompt: RenderedDecisionPrompt,
            context: DecisionPromptContext,
        ) -> str:
            if not isinstance(prompt, RenderedDecisionPrompt):
                raise TypeError("prompt must be a RenderedDecisionPrompt instance.")
            if not isinstance(context, DecisionPromptContext):
                raise TypeError("context must be a DecisionPromptContext instance.")
            return self.backend.generate(prompt=prompt, context=context)

        def _phase_repair_reasons(
            self,
            output: DecisionAgentOutput,
            context: DecisionPromptContext,
        ) -> list[str]:
            packet = context.decision_handover_packet
            signal = packet.policy_phase_signal if isinstance(packet.policy_phase_signal, dict) else {}
            phase = str(signal.get("phase", "uncertain"))
            direction_constraint = str(signal.get("direction_constraint", "uncertain"))
            magnitude_constraint = str(signal.get("magnitude_constraint", "uncertain"))
            final_label = output.final_recommendation.value
            reasons: list[str] = []

            if phase == "near_zlb_waiting" and direction_constraint == "hold_bias" and final_label in {"(0,25]bp", "(25,50]bp", ">50bp"}:
                reasons.append(
                    "policy_phase_signal=near_zlb_waiting/hold_bias，但最终输出为加息。该阶段下Taylor类压力通常应先解释为未来转向信号；除非非Taylor路径敏感理论也强烈支持当前会议加息，否则必须重新比较0bp与加息。"
                )
            if phase == "liftoff_start" and magnitude_constraint == "prefer_small" and final_label in {"(25,50]bp", ">50bp"}:
                reasons.append(
                    "policy_phase_signal=liftoff_start/prefer_small，但最终幅度超过小幅加息。首次启动阶段必须重新比较0bp、(0,25]bp与(25,50]bp，不得仅因Taylor类大幅压力直接选择较大幅度。"
                )
            if phase == "rapid_hiking_cycle" and magnitude_constraint == "allow_large" and final_label == "(25,50]bp":
                hike_support = sum(
                    1
                    for item in context.theory_outputs_full.values()
                    if item.recommendation.value in {"(0,25]bp", "(25,50]bp", ">50bp"}
                )
                large_support = sum(
                    1
                    for item in context.theory_outputs_full.values()
                    if item.recommendation.value == ">50bp"
                )
                if hike_support >= 6 or large_support >= 2:
                    reasons.append(
                        "policy_phase_signal=rapid_hiking_cycle/allow_large，且多数理论支持加息。当前输出为(25,50]bp时，必须重新比较(25,50]bp与>50bp；若通胀、需求、就业或金融压力多源一致，不得被利率平滑机械压低。"
                    )
            if phase == "easing_after_plateau" and final_label == "0bp":
                reasons.append(
                    "policy_phase_signal=easing_after_plateau，但最终输出为0bp。高位平台向宽松转换时，必须重新比较0bp与[-25,0)bp；若没有强通胀再加速证据，不能只用路径惯性压回维持。"
                )
            if phase == "high_rate_plateau" and final_label == "0bp":
                evidence = str(signal.get("evidence", ""))
                if any(token in evidence for token in ["5.5", "高位", "限制性", "维持8次", "维持9次"]):
                    reasons.append(
                        "policy_phase_signal=high_rate_plateau且利率处于高位平台，最终输出为0bp。必须额外比较0bp与小幅降息，说明为何当前不是平台后转向宽松；不能仅因平滑规则偏维持就结束判断。"
                    )
            return reasons

        def _repair_decision_output(
            self,
            *,
            raw_output: str,
            context: DecisionPromptContext,
            repair_reasons: list[str],
        ) -> DecisionAgentOutput:
            base_prompt = self.render_prompt(context)
            repair_prompt = RenderedDecisionPrompt(
                system_prompt=(
                    "You repair the System C decision agent output under phase-constraint validation. "
                    "只返回一个JSON对象，不要额外文字。"
                ),
                user_prompt="\n".join(
                    [
                        base_prompt.user_prompt,
                        "",
                        "[阶段约束校验失败]",
                        *[f"- {reason}" for reason in repair_reasons],
                        "",
                        "[修复要求]",
                        "重新输出完整决策JSON，必须包含输出契约中的所有字段。",
                        "不要添加额外顶层字段。",
                        "final_recommendation必须等于概率最高的标签。",
                        "rule_application_summary必须明确说明是否服从policy_phase_signal；若不服从，必须给出强证据。",
                        "magnitude_decision_rationale必须重新比较被校验指出的候选标签。",
                        "",
                        "[上一次输出]",
                        raw_output,
                    ]
                ),
            )
            repaired_raw = self.invoke_backend(prompt=repair_prompt, context=context)
            repaired = parse_decision_agent_output(raw_output=repaired_raw, context=context)
            remaining_reasons = self._phase_repair_reasons(repaired, context)
            if remaining_reasons:
                repaired.remaining_uncertainty = (
                    str(repaired.remaining_uncertainty)
                    + " phase_constraint_remaining_warning="
                    + " | ".join(remaining_reasons)
                )
            return repaired

        def _direction_from_label(self, label: str) -> str:
            if label in {"<-50bp", "[-50,-25)bp", "[-25,0)bp"}:
                return "cut"
            if label == "0bp":
                return "hold"
            if label in {"(0,25]bp", "(25,50]bp", ">50bp"}:
                return "hike"
            return "uncertain"

        def _force_output_label(
            self,
            output: DecisionAgentOutput,
            *,
            forced_label: str,
            reason: str,
        ) -> DecisionAgentOutput:
            old_probs = output.final_probability_distribution.as_ordered_dict()
            forced_probs: dict[str, float] = {}
            other_total = sum(prob for label, prob in old_probs.items() if label != forced_label)
            for label, prob in old_probs.items():
                if label == forced_label:
                    forced_probs[label] = 0.64
                else:
                    forced_probs[label] = (prob / other_total * 0.36) if other_total > 0 else (0.36 / 6.0)
            forced_distribution = ProbabilityDistribution(probabilities=forced_probs)
            forced_direction = self._direction_from_label(forced_label)
            return DecisionAgentOutput(
                final_recommendation=forced_label,
                final_probability_distribution=forced_distribution,
                final_confidence=max(50, min(int(output.final_confidence), 78)),
                final_explanation=str(output.final_explanation) + " deterministic_phase_postprocess: " + reason,
                dominant_rule_family=output.dominant_rule_family,
                supporting_rule_families=list(output.supporting_rule_families),
                theory_weight_profile=dict(output.theory_weight_profile),
                theory_weight_rationale=str(output.theory_weight_rationale),
                rule_contribution_shares=dict(output.rule_contribution_shares),
                memory_usage_summary=str(output.memory_usage_summary),
                matched_memory_lessons=list(output.matched_memory_lessons),
                matched_conditional_rules=list(output.matched_conditional_rules),
                rule_application_summary=str(output.rule_application_summary) + " deterministic_phase_postprocess: " + reason,
                direction_decision=forced_direction,
                magnitude_decision_rationale=str(output.magnitude_decision_rationale) + " deterministic_phase_postprocess: " + reason,
                first_contributor_rule=output.first_contributor_rule,
                current_sample_reasoning=str(output.current_sample_reasoning),
                remaining_uncertainty=str(output.remaining_uncertainty) + " deterministic_phase_postprocess_applied=" + reason,
                decision_path=output.decision_path,
            )

        def _apply_deterministic_phase_postprocess(
            self,
            output: DecisionAgentOutput,
            context: DecisionPromptContext,
        ) -> DecisionAgentOutput:
            packet = context.decision_handover_packet
            signal = packet.policy_phase_signal if isinstance(packet.policy_phase_signal, dict) else {}
            phase = str(signal.get("phase", "uncertain"))
            direction_constraint = str(signal.get("direction_constraint", "uncertain"))
            magnitude_constraint = str(signal.get("magnitude_constraint", "uncertain"))
            final_label = output.final_recommendation.value

            # Rule C+1: lower-bound waiting/liftoff waiting should not convert Taylor pressure
            # into a current hike when the path-sensitive New Keynesian rule still recommends hold.
            new_keynesian_label = context.theory_outputs_full["new_keynesian_optimal_rule"].recommendation.value
            if phase in {"near_zlb_waiting", "liftoff_start"} and magnitude_constraint == "prefer_small":
                if final_label == "(0,25]bp" and new_keynesian_label == "0bp":
                    return self._force_output_label(
                        output,
                        forced_label="0bp",
                        reason=(
                            "C+1 near_zlb/liftoff waiting hard constraint: new_keynesian_optimal_rule仍为0bp，"
                            "将小幅加息视为未来转向压力而非当前动作。"
                        ),
                    )
                if final_label in {"(25,50]bp", ">50bp"}:
                    return self._force_output_label(
                        output,
                        forced_label="(0,25]bp",
                        reason=(
                            "C+1 liftoff_start hard constraint: 首次启动阶段优先小幅，"
                            "禁止在prefer_small信号下直接给出较大加息。"
                        ),
                    )

            # Rule C+2: once rapid hiking is diagnosed and large moves are allowed,
            # do not let smoothing mechanically cap the decision at 25-50bp.
            if phase == "rapid_hiking_cycle" and magnitude_constraint == "allow_large" and final_label == "(25,50]bp":
                hike_support = sum(
                    1
                    for item in context.theory_outputs_full.values()
                    if item.recommendation.value in {"(0,25]bp", "(25,50]bp", ">50bp"}
                )
                if hike_support >= 6:
                    return self._force_output_label(
                        output,
                        forced_label=">50bp",
                        reason=(
                            "C+2 rapid_hiking_cycle hard constraint: 多数理论同向支持加息且Analysis允许大幅，"
                            "将(25,50]bp上调为>50bp以避免低估快速加息周期。"
                        ),
                    )

            # Rule C+3: when Analysis explicitly diagnoses easing_after_plateau with cut_bias,
            # the deterministic postprocessor prevents inertial 0bp.
            if phase == "easing_after_plateau" and direction_constraint == "cut_bias" and final_label == "0bp":
                return self._force_output_label(
                    output,
                    forced_label="[-25,0)bp",
                    reason=(
                        "C+3 easing_after_plateau hard constraint: Analysis给出cut_bias，"
                        "将高位平台后的0bp惯性修正为小幅降息。"
                    ),
                )

            return output

        def parse_output(
            self,
            raw_output: str,
            context: DecisionPromptContext,
        ) -> DecisionAgentOutput:
            if not isinstance(raw_output, str):
                raise TypeError("raw_output must be a string.")
            if not isinstance(context, DecisionPromptContext):
                raise TypeError("context must be a DecisionPromptContext instance.")
            try:
                output = parse_decision_agent_output(raw_output=raw_output, context=context)
            except Exception as parse_error:
                repair_prompt = RenderedDecisionPrompt(
                    system_prompt=(
                        "You repair malformed JSON from the System C decision agent. "
                        "只返回一个JSON对象，不要额外文字。"
                    ),
                    user_prompt="\n".join(
                        [
                            "The previous decision-agent output could not be parsed.",
                            f"Parser error: {type(parse_error).__name__}: {parse_error}",
                            "",
                            "只返回一个JSON对象，且只能包含输出契约中的19个顶层字段：",
                            "- final_recommendation",
                            "- final_probability_distribution",
                            "- final_confidence",
                            "- final_explanation",
                            "- dominant_rule_family",
                            "- supporting_rule_families",
                            "- theory_weight_profile",
                            "- theory_weight_rationale",
                            "- rule_contribution_shares",
                            "- memory_usage_summary",
                            "- matched_memory_lessons",
                            "- matched_conditional_rules",
                            "- rule_application_summary",
                            "- direction_decision",
                            "- magnitude_decision_rationale",
                            "- first_contributor_rule",
                            "- current_sample_reasoning",
                            "- remaining_uncertainty",
                            "- decision_path",
                            "",
                            "保持decision_path等于training_textual_memory_informed_aggregation。",
                            "theory_weight_profile必须是当前样本的局部理论侧重，不得把冻结文字记忆当成固定数值权重。",
                            "rule_application_summary必须说明结构化记忆和policy_phase_signal如何影响方向与幅度。",
                            "",
                            "格式错误的输出：",
                            raw_output,
                        ]
                    ),
                )
                repaired_raw = self.invoke_backend(prompt=repair_prompt, context=context)
                output = parse_decision_agent_output(raw_output=repaired_raw, context=context)
                raw_output = repaired_raw

            repair_reasons = self._phase_repair_reasons(output, context)
            if repair_reasons:
                output = self._repair_decision_output(
                    raw_output=raw_output,
                    context=context,
                    repair_reasons=repair_reasons,
                )
            return self._apply_deterministic_phase_postprocess(output, context)

        def run(self, context: DecisionPromptContext) -> DecisionAgentOutput:
            if not isinstance(context, DecisionPromptContext):
                raise TypeError("context must be a DecisionPromptContext instance.")
            prompt = self.render_prompt(context)
            raw_output = self.invoke_backend(prompt=prompt, context=context)
            return self.parse_output(raw_output=raw_output, context=context)


In [ ]:
# 文件：decision_agent/__init__.py
# 产出：导出决策层接口与工具。

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    __all__ = [
        "DecisionAgentRunner",
        "DecisionModelBackend",
        "DecisionPromptContext",
        "REQUIRED_OUTPUT_FIELDS",
        "RenderedDecisionPrompt",
        "SingleDecisionAgentRunner",
        "build_decision_prompt_context",
        "parse_decision_agent_output",
        "render_decision_agent_prompt",
    ]


## Runtime State and Pipeline

In [ ]:
# file: state_models/pipeline.py
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from dataclasses import dataclass
    from datetime import date

    @dataclass(slots=True)
    class SampleContextState:
        sample: BaseDecisionSample
        sample_id: str
        country: str
        decision_date: date
        is_training: bool

        def __post_init__(self) -> None:
            if not isinstance(self.sample, BaseDecisionSample):
                raise TypeError("sample must be a BaseDecisionSample instance.")
            if not isinstance(self.sample_id, str) or not self.sample_id.strip():
                raise ValueError("sample_id must be a non-empty string.")
            if self.country != self.sample.country:
                raise ValueError("country must match sample.country.")
            if self.decision_date != self.sample.decision_date:
                raise ValueError("decision_date must match sample.decision_date.")
            if not isinstance(self.is_training, bool):
                raise TypeError("is_training must be a bool.")
            if self.is_training and not isinstance(self.sample, TrainingDecisionSample):
                raise ValueError("Training state must carry a TrainingDecisionSample.")
            if (not self.is_training) and not isinstance(self.sample, PredictionDecisionSample):
                raise ValueError("Prediction state must carry a prediction-compatible sample.")

    @dataclass(slots=True)
    class TheoryLayerState:
        sample_context: SampleContextState
        theory_outputs: dict[str, TheoryAgentOutput]

        def __post_init__(self) -> None:
            if not isinstance(self.sample_context, SampleContextState):
                raise TypeError("sample_context must be a SampleContextState instance.")
            if set(self.theory_outputs) != set(THEORY_IDS):
                raise ValueError("theory_outputs must cover all registered theories exactly.")
            for theory_id, output in self.theory_outputs.items():
                if not isinstance(output, TheoryAgentOutput):
                    raise TypeError(f"theory_outputs[{theory_id}] must be a TheoryAgentOutput instance.")

    @dataclass(slots=True)
    class AnalysisState:
        sample_context: SampleContextState
        theory_outputs: dict[str, TheoryAgentOutput]
        analysis_payload: AnalysisPayload | None = None
        decision_handover_packet: DecisionHandoverPacket | None = None

        def __post_init__(self) -> None:
            if not isinstance(self.sample_context, SampleContextState):
                raise TypeError("sample_context must be a SampleContextState instance.")
            if set(self.theory_outputs) != set(THEORY_IDS):
                raise ValueError("theory_outputs must cover all registered theories exactly.")
            if self.analysis_payload is not None and not isinstance(self.analysis_payload, AnalysisPayload):
                raise TypeError("analysis_payload must be an AnalysisPayload instance or None.")
            if self.decision_handover_packet is not None and not isinstance(
                self.decision_handover_packet, DecisionHandoverPacket
            ):
                raise TypeError("decision_handover_packet must be a DecisionHandoverPacket instance or None.")

    @dataclass(slots=True)
    class DecisionState:
        sample_context: SampleContextState
        analysis_state: AnalysisState
        decision_output: DecisionAgentOutput

        def __post_init__(self) -> None:
            if not isinstance(self.sample_context, SampleContextState):
                raise TypeError("sample_context must be a SampleContextState instance.")
            if not isinstance(self.analysis_state, AnalysisState):
                raise TypeError("analysis_state must be an AnalysisState instance.")
            if not isinstance(self.decision_output, DecisionAgentOutput):
                raise TypeError("decision_output must be a DecisionAgentOutput instance.")


In [ ]:
# file: state_models/__init__.py
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    __all__ = ["AnalysisState", "DecisionState", "SampleContextState", "TheoryLayerState"]


In [ ]:

from __future__ import annotations

import csv
import json
from dataclasses import fields, is_dataclass
from datetime import date, datetime, timezone
from enum import Enum
from pathlib import Path

CURRENT_RUNTIME_TRACE_TITLE = None
CURRENT_RUNTIME_TRACE_EVENTS: list[dict[str, object]] = []
CURRENT_RUNTIME_TRACE_LIVE_PATH: Path | None = None
RUNTIME_PRINT_SUMMARY = True
RUNTIME_VERBOSE_PRINT = False

def to_jsonable(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, Enum):
        return value.value
    if isinstance(value, (date, datetime)):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    if hasattr(value, "as_ordered_dict") and callable(value.as_ordered_dict):
        return to_jsonable(value.as_ordered_dict())
    if is_dataclass(value):
        return {field.name: to_jsonable(getattr(value, field.name)) for field in fields(value)}
    if isinstance(value, dict):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_jsonable(item) for item in value]
    if hasattr(value, "__dict__"):
        return {str(key): to_jsonable(item) for key, item in vars(value).items()}
    return repr(value)

def _label_value(value):
    return getattr(value, "value", value)
def reset_runtime_trace(title: str) -> None:
    global CURRENT_RUNTIME_TRACE_TITLE, CURRENT_RUNTIME_TRACE_EVENTS
    CURRENT_RUNTIME_TRACE_TITLE = title
    CURRENT_RUNTIME_TRACE_EVENTS = []

def set_runtime_live_trace_path(path: Path | None) -> None:
    global CURRENT_RUNTIME_TRACE_LIVE_PATH
    CURRENT_RUNTIME_TRACE_LIVE_PATH = path

def append_runtime_trace(stage: str, payload: dict[str, object]) -> None:
    CURRENT_RUNTIME_TRACE_EVENTS.append({"stage": stage, "timestamp": datetime.now().isoformat(timespec="seconds"), "payload": to_jsonable(payload)})

def _compact_stage_name(stage: str) -> str:
    return {
        "theory_layer_output": "theory",
        "analysis_layer_output": "analysis",
        "decision_layer_output": "decision",
        "inference_started": "predict_started",
        "inference_sample_started": "predict_start",
        "inference_sample_skipped": "predict_skip",
        "inference_sample_failed": "predict_failed",
        "inference_sample_completed": "predict_done",
        "inference_finished": "predict_finished",
    }.get(stage, stage)

def _build_runtime_summary_line(stage: str, payload: dict[str, object]) -> str:
    parts = [f"stage={_compact_stage_name(stage)}"]
    sample_id = payload.get("sample_id")
    if sample_id:
        parts.append(f"sample={sample_id}")
    decision_date = payload.get("decision_date")
    if decision_date:
        parts.append(f"date={decision_date}")
    country = payload.get("country")
    if country:
        parts.append(f"country={country}")
    if stage == "inference_sample_completed":
        parts.append(f"actual={payload.get('actual_label')}")
        parts.append(f"pred={payload.get('final_recommendation')}")
        parts.append(f"pred_conf={payload.get('final_confidence')}")
    return " | ".join(str(item) for item in parts if item is not None)

def emit_runtime_stage(stage: str, payload: dict[str, object]) -> None:
    normalized_payload = to_jsonable(payload)
    append_runtime_trace(stage, normalized_payload)
    if RUNTIME_PRINT_SUMMARY:
        print(_build_runtime_summary_line(stage, normalized_payload), flush=True)
    if RUNTIME_VERBOSE_PRINT:
        print(json.dumps(normalized_payload, ensure_ascii=False, indent=2), flush=True)
    if CURRENT_RUNTIME_TRACE_LIVE_PATH is not None:
        try:
            _json_write(CURRENT_RUNTIME_TRACE_LIVE_PATH, snapshot_runtime_trace())
        except Exception:
            pass

def snapshot_runtime_trace() -> dict[str, object]:
    return {"title": CURRENT_RUNTIME_TRACE_TITLE, "created_at": datetime.now().isoformat(timespec="seconds"), "events": CURRENT_RUNTIME_TRACE_EVENTS}

def _json_write(path: Path, payload: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(to_jsonable(payload), ensure_ascii=False, indent=2), encoding="utf-8")

def _write_csv(path: Path, rows: list[dict[str, object]], fieldnames: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow({key: row.get(key) for key in fieldnames})

def _write_jsonl(path: Path, rows: list[dict[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(to_jsonable(row), ensure_ascii=False) + "\n")

def save_live_runtime_trace(title: str, base_dir: Path) -> Path:
    output_path = base_dir / f"{title}_trace_live.json"
    _json_write(output_path, snapshot_runtime_trace())
    return output_path

def save_runtime_trace(title: str, base_dir: Path, run_id: str | None = None) -> Path:
    token = run_id or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    output_path = base_dir / f"{title}_trace_{token}.json"
    _json_write(output_path, snapshot_runtime_trace())
    return output_path



In [ ]:
# file: prediction_pipeline/orchestrator.py
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from dataclasses import dataclass

    @dataclass(frozen=True, slots=True)
    class SingleSamplePipelineResult:
        sample_context: SampleContextState
        theory_layer_state: TheoryLayerState
        analysis_state: AnalysisState
        decision_state: DecisionState

    def build_sample_context_state(sample: BaseDecisionSample, sample_id: str) -> SampleContextState:
        is_training = isinstance(sample, TrainingDecisionSample)
        return SampleContextState(sample=sample, sample_id=sample_id, country=sample.country, decision_date=sample.decision_date, is_training=is_training)

    @dataclass(slots=True)
    class HierarchicalPredictionPipeline:
        theory_runner: SingleTheoryAgentRunner
        analysis_runner: SingleAnalysisHandoverRunner
        analysis_state_builder: AnalysisHandoverStateBuilder
        decision_runner: SingleDecisionAgentRunner

        def run_theory_layer(self, sample_context: SampleContextState) -> TheoryLayerState:
            contexts = build_all_theory_prompt_contexts(sample_context.sample)
            theory_outputs = run_all_theory_agents(contexts=contexts, runner=self.theory_runner)
            return TheoryLayerState(sample_context=sample_context, theory_outputs=theory_outputs)

        def run_analysis_layer(self, theory_layer_state: TheoryLayerState) -> AnalysisState:
            analysis_context = build_analysis_handover_context(theory_layer_state)
            synthesis_output = self.analysis_runner.run(analysis_context)
            return self.analysis_state_builder.build_analysis_state(sample_context=theory_layer_state.sample_context, theory_outputs=theory_layer_state.theory_outputs, synthesis_output=synthesis_output)

        def run_decision_layer(self, analysis_state: AnalysisState) -> DecisionState:
            decision_context = build_decision_prompt_context(analysis_state=analysis_state)
            decision_output = self.decision_runner.run(decision_context)
            return DecisionState(sample_context=analysis_state.sample_context, analysis_state=analysis_state, decision_output=decision_output)

        def run_single_sample(self, sample: BaseDecisionSample, sample_id: str) -> SingleSamplePipelineResult:
            sample_context = build_sample_context_state(sample=sample, sample_id=sample_id)
            theory_layer_state = self.run_theory_layer(sample_context)
            emit_runtime_stage("theory_layer_output", {"sample_id": sample_context.sample_id, "decision_date": sample_context.decision_date.isoformat(), "country": sample_context.country, "theory_outputs": to_jsonable(theory_layer_state.theory_outputs)})
            analysis_state = self.run_analysis_layer(theory_layer_state)
            emit_runtime_stage("analysis_layer_output", {"sample_id": sample_context.sample_id, "decision_date": sample_context.decision_date.isoformat(), "country": sample_context.country, "analysis_payload": to_jsonable(analysis_state.analysis_payload), "decision_handover_packet": to_jsonable(analysis_state.decision_handover_packet)})
            decision_state = self.run_decision_layer(analysis_state=analysis_state)
            emit_runtime_stage("decision_layer_output", {"sample_id": sample_context.sample_id, "decision_date": sample_context.decision_date.isoformat(), "country": sample_context.country, "decision_output": to_jsonable(decision_state.decision_output)})
            return SingleSamplePipelineResult(sample_context=sample_context, theory_layer_state=theory_layer_state, analysis_state=analysis_state, decision_state=decision_state)


In [ ]:
# file: prediction_pipeline/__init__.py
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    __all__ = ["HierarchicalPredictionPipeline", "SingleSamplePipelineResult", "build_sample_context_state"]


## Direct Inference

In [ ]:
# file: inference_loop/driver.py
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from dataclasses import dataclass
    from typing import Any, Callable

    @dataclass(frozen=True, slots=True)
    class DirectInferenceItem:
        sample_ref: ScheduledSampleRef
        pipeline_result: SingleSamplePipelineResult
        actual_label: RateMoveLabel

    @dataclass(frozen=True, slots=True)
    class DirectInferenceRunResult:
        test_schedule: tuple[ScheduledSampleRef, ...]
        items: tuple[DirectInferenceItem, ...]

    @dataclass(slots=True)
    class DirectInferenceDriver:
        scheduler: USSampleScheduler
        prediction_pipeline: HierarchicalPredictionPipeline
        max_attempts_per_sample: int = 2

        def run(self, progress_callback: Callable[[dict[str, Any]], None] | None = None, skip_sample_ids: set[str] | None = None, completed_item_callback: Callable[[DirectInferenceItem], None] | None = None) -> DirectInferenceRunResult:
            if skip_sample_ids is None:
                skip_sample_ids = set()
            test_schedule = self.scheduler.test_schedule()
            items: list[DirectInferenceItem] = []
            total_samples = len(test_schedule)
            skipped_samples = 0
            if progress_callback is not None:
                progress_callback({"event": "inference_started", "completed_samples": 0, "total_samples": total_samples})
            for sample_index, sample_ref in enumerate(test_schedule):
                if sample_ref.sample_id in skip_sample_ids:
                    skipped_samples += 1
                    if progress_callback is not None:
                        progress_callback({"event": "inference_sample_skipped", "sample_index": sample_index, "sample_id": sample_ref.sample_id, "decision_date": sample_ref.decision_date.isoformat(), "skip_reason": "already_successful_prediction_step", "completed_samples": len(items) + skipped_samples, "total_samples": total_samples})
                    continue
                if progress_callback is not None:
                    progress_callback({"event": "inference_sample_started", "sample_index": sample_index, "sample_id": sample_ref.sample_id, "decision_date": sample_ref.decision_date.isoformat(), "completed_samples": len(items) + skipped_samples, "total_samples": total_samples})
                sample, _ = self.scheduler.build_test_sample(sample_index)
                pipeline_result = None
                for attempt_count in range(1, self.max_attempts_per_sample + 1):
                    try:
                        pipeline_result = self.prediction_pipeline.run_single_sample(sample=sample, sample_id=sample_ref.sample_id)
                        break
                    except Exception as exc:
                        if progress_callback is not None:
                            progress_callback({"event": "inference_sample_failed", "sample_index": sample_index, "sample_id": sample_ref.sample_id, "decision_date": sample_ref.decision_date.isoformat(), "attempt_count": attempt_count, "max_attempts": self.max_attempts_per_sample, "error_type": f"{type(exc).__module__}.{type(exc).__qualname__}", "error_message": str(exc), "completed_samples": len(items) + skipped_samples, "total_samples": total_samples, "will_retry": attempt_count < self.max_attempts_per_sample})
                        if attempt_count >= self.max_attempts_per_sample:
                            raise
                actual_label = self.scheduler.sample_builder.build_actual_label(sample_ref.decision_date)
                item = DirectInferenceItem(sample_ref=sample_ref, pipeline_result=pipeline_result, actual_label=actual_label)
                items.append(item)
                if completed_item_callback is not None:
                    completed_item_callback(item)
                decision_output = pipeline_result.decision_state.decision_output
                if progress_callback is not None:
                    progress_callback({"event": "inference_sample_completed", "sample_index": sample_index, "sample_id": sample_ref.sample_id, "decision_date": sample_ref.decision_date.isoformat(), "actual_label": _label_value(actual_label), "final_recommendation": _label_value(decision_output.final_recommendation), "final_confidence": decision_output.final_confidence, "completed_samples": len(items) + skipped_samples, "total_samples": total_samples})
            if progress_callback is not None:
                progress_callback({"event": "inference_finished", "completed_samples": len(items) + skipped_samples, "total_samples": total_samples, "skipped_samples": skipped_samples})
            return DirectInferenceRunResult(test_schedule=test_schedule, items=tuple(items))


In [ ]:
# file: inference_loop/__init__.py
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    __all__ = ["DirectInferenceDriver", "DirectInferenceItem", "DirectInferenceRunResult"]


## Model Backend

In [ ]:
# 文件：model_backends/openai_compat.py
# 产出：产出统一的大模型 API 调用后端。
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    import json
    import os
    import random
    import requests
    import time
    from dataclasses import dataclass



    OPENAI_COMPAT_DEFAULT_BASE_URL = "https://api.deepseek.com/v1"
    OPENAI_COMPAT_DEFAULT_MODEL = "deepseek-v4-flash"
    OPENAI_COMPAT_DEFAULT_API_KEY_ENV = "DEEPSEEK_API_KEY"
    DEFAULT_TEMPERATURE = 0.2
    DEFAULT_MAX_RETRIES = 3
    DEFAULT_INITIAL_BACKOFF_SECONDS = 1.0
    DEFAULT_BACKOFF_MULTIPLIER = 2.0


    def first_nonempty(*keys: str, default: str | None = None) -> str | None:
        for key in keys:
            value = os.getenv(key, "").strip()
            if value:
                return value
        return default


    def normalize_openai_compat_base_url(base_url: str | None) -> str:
        candidate = str(base_url or "").strip()
        if not candidate:
            return OPENAI_COMPAT_DEFAULT_BASE_URL
        candidate = candidate.rstrip("/")
        if candidate.endswith("/chat/completions"):
            candidate = candidate[: -len("/chat/completions")]
        return candidate


    @dataclass(slots=True)
    class OpenAICompatBackend(
        TheoryModelBackend,
        AnalysisHandoverModelBackend,
        DecisionModelBackend,
    ):
        model_name: str | None = None
        api_base: str | None = None
        api_key_env: str = OPENAI_COMPAT_DEFAULT_API_KEY_ENV
        timeout_seconds: int = 120
        temperature: float = DEFAULT_TEMPERATURE
        max_output_tokens: int | None = None
        max_retries: int = DEFAULT_MAX_RETRIES
        initial_backoff_seconds: float = DEFAULT_INITIAL_BACKOFF_SECONDS
        backoff_multiplier: float = DEFAULT_BACKOFF_MULTIPLIER
        use_json_schema: bool = False
        fallback_to_json_object: bool = True
        include_response_format: bool = True

        def __post_init__(self) -> None:
            if self.model_name is not None and (
                not isinstance(self.model_name, str) or not self.model_name.strip()
            ):
                raise ValueError("model_name must be a non-empty string or None.")
            if self.api_base is not None and not isinstance(self.api_base, str):
                raise TypeError("api_base must be a string or None.")
            if not isinstance(self.api_key_env, str) or not self.api_key_env.strip():
                raise ValueError("api_key_env must be a non-empty string.")
            if not isinstance(self.timeout_seconds, int) or self.timeout_seconds <= 0:
                raise ValueError("timeout_seconds must be a positive integer.")
            if not isinstance(self.temperature, (int, float)):
                raise TypeError("temperature must be numeric.")
            self.temperature = float(self.temperature)
            if not 0.0 <= self.temperature <= 2.0:
                raise ValueError("temperature must lie within [0, 2].")
            if self.max_output_tokens is not None:
                if not isinstance(self.max_output_tokens, int) or self.max_output_tokens <= 0:
                    raise ValueError("max_output_tokens must be a positive integer or None.")
            if not isinstance(self.max_retries, int) or self.max_retries < 0:
                raise ValueError("max_retries must be a non-negative integer.")
            if not isinstance(self.initial_backoff_seconds, (int, float)):
                raise TypeError("initial_backoff_seconds must be numeric.")
            self.initial_backoff_seconds = float(self.initial_backoff_seconds)
            if self.initial_backoff_seconds <= 0.0:
                raise ValueError("initial_backoff_seconds must be strictly positive.")
            if not isinstance(self.backoff_multiplier, (int, float)):
                raise TypeError("backoff_multiplier must be numeric.")
            self.backoff_multiplier = float(self.backoff_multiplier)
            if self.backoff_multiplier < 1.0:
                raise ValueError("backoff_multiplier must be at least 1.0.")
            if not isinstance(self.use_json_schema, bool):
                raise TypeError("use_json_schema must be a bool.")
            if not isinstance(self.fallback_to_json_object, bool):
                raise TypeError("fallback_to_json_object must be a bool.")
            if not isinstance(self.include_response_format, bool):
                raise TypeError("include_response_format must be a bool.")

            explicit_api_base = isinstance(self.api_base, str) and bool(self.api_base.strip())
            explicit_model_name = isinstance(self.model_name, str) and bool(self.model_name.strip())

            if explicit_api_base:
                resolved_base = self.api_base
            else:
                resolved_base = first_nonempty(
                    "OPENAI_COMPAT_BASE_URL",
                    "DEEPSEEK_API_BASE",
                    "DEEPSEEK_BASE_URL",
                    "OPENAI_BASE_URL",
                    "MKEAI_BASE_URL",
                    default=OPENAI_COMPAT_DEFAULT_BASE_URL,
                )

            if explicit_model_name:
                resolved_model = self.model_name
            else:
                resolved_model = first_nonempty(
                    "OPENAI_COMPAT_MODEL",
                    "DEEPSEEK_MODEL",
                    "OPENAI_MODEL",
                    "MKEAI_MODEL",
                    default=OPENAI_COMPAT_DEFAULT_MODEL,
                )
            self.api_base = normalize_openai_compat_base_url(resolved_base)
            self.model_name = str(resolved_model or OPENAI_COMPAT_DEFAULT_MODEL).strip()
            include_response_format = first_nonempty(
                "OPENAI_COMPAT_INCLUDE_RESPONSE_FORMAT",
                "OPENAI_INCLUDE_RESPONSE_FORMAT",
                default=None,
            )
            if include_response_format is not None:
                self.include_response_format = include_response_format.strip().lower() in {"1", "true", "yes", "on"}

        def resolved_config(self) -> dict[str, object]:
            return {
                "api_base": self.api_base,
                "model_name": self.model_name,
                "api_key_env": self.api_key_env,
                "temperature": self.temperature,
                "max_output_tokens": self.max_output_tokens,
                "use_json_schema": self.use_json_schema,
                "fallback_to_json_object": self.fallback_to_json_object,
                "include_response_format": self.include_response_format,
            }

        def generate(
            self,
            prompt: (
                RenderedTheoryPrompt
                | RenderedAnalysisHandoverPrompt
                | RenderedDecisionPrompt

            ),
            context: (
                TheoryPromptContext
                | AnalysisHandoverContext
                | DecisionPromptContext

            ),
        ) -> str:
            if isinstance(prompt, RenderedTheoryPrompt):
                if not isinstance(context, TheoryPromptContext):
                    raise TypeError("Theory prompt must pair with TheoryPromptContext.")
            elif isinstance(prompt, RenderedAnalysisHandoverPrompt):
                if not isinstance(context, AnalysisHandoverContext):
                    raise TypeError("Analysis handover prompt must pair with AnalysisHandoverContext.")
            elif isinstance(prompt, RenderedDecisionPrompt):
                if not isinstance(context, DecisionPromptContext):
                    raise TypeError("Decision prompt must pair with DecisionPromptContext.")
            else:
                raise TypeError(
                    "prompt must be a RenderedTheoryPrompt, RenderedAnalysisHandoverPrompt, RenderedDecisionPrompt, "
                    "RenderedDecisionPrompt instance."
                )

            return self._chat_completion(
                system_prompt=prompt.system_prompt,
                user_prompt=prompt.user_prompt,
                response_format=self._build_response_format(prompt),
            )

        def _chat_completion(
            self,
            system_prompt: str,
            user_prompt: str,
            response_format: dict[str, object],
        ) -> str:
            api_key = first_nonempty(
                self.api_key_env,
                "DEEPSEEK_API_KEY",
                "SUBLYX_API_KEY",
                "OPENAI_API_KEY",
                "OPENAI_COMPAT_API_KEY",
                "MKEAI_API_KEY",
                default="",
            ) or ""
            if not api_key:
                raise RuntimeError(f"Missing API key in environment variable: {self.api_key_env}")

            payload = {
                "model": self.model_name,
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                "temperature": self.temperature,
            }
            if self.include_response_format:
                payload["response_format"] = response_format
            if self.max_output_tokens is not None:
                payload["max_tokens"] = self.max_output_tokens
            request_url = f"{str(self.api_base).rstrip('/')}/chat/completions"
            request_headers = {
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json",
                "User-Agent": "Mozilla/5.0",
            }
            try:
                body = self._execute_request_with_retry(
                    url=request_url,
                    payload=payload,
                    headers=request_headers,
                )
            except RuntimeError as exc:
                error_text = str(exc)
                if (
                    self.use_json_schema
                    and self.fallback_to_json_object
                    and self.include_response_format
                    and response_format.get("type") == "json_schema"
                    and self._looks_like_json_schema_unsupported(error_text)
                ):
                    fallback_payload = dict(payload)
                    fallback_payload["response_format"] = {"type": "json_object"}
                    body = self._execute_request_with_retry(
                        url=request_url,
                        payload=fallback_payload,
                        headers=request_headers,
                    )
                elif (
                    self.fallback_to_json_object
                    and self.include_response_format
                    and response_format.get("type") == "json_object"
                    and self._looks_like_response_format_transport_failure(error_text)
                ):
                    fallback_payload = dict(payload)
                    fallback_payload.pop("response_format", None)
                    body = self._execute_request_with_retry(
                        url=request_url,
                        payload=fallback_payload,
                        headers=request_headers,
                    )
                else:
                    raise

            try:
                return self._extract_message_content(body)
            except (KeyError, IndexError, TypeError) as exc:
                response_excerpt = self._response_debug_excerpt(body)
                raise RuntimeError(
                    "OpenAI-compatible response is missing choices[0].message.content. "
                    f"response_excerpt={response_excerpt}"
                ) from exc

        def _execute_request_with_retry(
            self,
            url: str,
            payload: dict[str, object],
            headers: dict[str, str],
        ) -> dict[str, object]:
            last_error: RuntimeError | None = None
            for attempt in range(self.max_retries + 1):
                try:
                    response = requests.post(
                        url,
                        headers=headers,
                        json=payload,
                        timeout=self.timeout_seconds,
                    )
                    if response.ok:
                        return response.json()
                    error_body = response.text
                    if len(error_body) > 1200:
                        error_body = error_body[:1200] + "...<truncated>"
                    last_error = RuntimeError(f"OpenAI-compatible HTTP {response.status_code}: {error_body}")
                    if not self._should_retry_http(response.status_code) or attempt >= self.max_retries:
                        raise last_error
                except requests.RequestException as exc:
                    last_error = RuntimeError(f"OpenAI-compatible request failed: {exc}")
                    if attempt >= self.max_retries:
                        raise last_error from exc

                sleep_seconds = self.initial_backoff_seconds * (self.backoff_multiplier**attempt)
                sleep_seconds += random.uniform(0.0, 0.25)
                time.sleep(sleep_seconds)

            if last_error is not None:
                raise last_error
            raise RuntimeError("OpenAI-compatible request failed without a captured error.")

        @staticmethod
        def _should_retry_http(status_code: int) -> bool:
            return status_code in {429, 500, 502, 503, 504}

        @staticmethod
        def _looks_like_json_schema_unsupported(error_text: str) -> bool:
            lowered = error_text.lower()
            unsupported_markers = (
                "json_schema is not supported",
                "response_format type json_schema is not supported",
                "this response_format type is unavailable now",
                "response_format type is unavailable now",
                "unsupported response_format",
                "unsupported value: 'json_schema'",
                'unsupported value: "json_schema"',
                "invalid value for response_format",
                "invalid value: 'json_schema'",
                'invalid value: "json_schema"',
                "response_format must be one of",
                "does not support json_schema",
            )
            return any(marker in lowered for marker in unsupported_markers)

        @staticmethod
        def _looks_like_response_format_transport_failure(error_text: str) -> bool:
            lowered = error_text.lower()
            if "response_format" in lowered:
                return True
            return "http 400" in lowered and "upstream_error" in lowered

        @staticmethod
        def _extract_message_content(body: dict[str, object]) -> str:
            choices = body["choices"]
            if not isinstance(choices, list) or not choices:
                raise RuntimeError("OpenAI-compatible response is missing choices.")
            first_choice = choices[0]
            if not isinstance(first_choice, dict):
                raise RuntimeError("OpenAI-compatible response choices[0] is invalid.")
            message = first_choice["message"]
            if not isinstance(message, dict):
                raise RuntimeError("OpenAI-compatible response choices[0].message is invalid.")
            content = message["content"]
            normalized = OpenAICompatBackend._normalize_message_content(content)
            if normalized is not None:
                return normalized
            for fallback_key in ("reasoning_content", "output_text", "text"):
                normalized_fallback = OpenAICompatBackend._normalize_message_content(message.get(fallback_key))
                if normalized_fallback is not None:
                    return normalized_fallback
            refusal = message.get("refusal")
            if isinstance(refusal, str) and refusal.strip():
                return refusal.strip()
            tool_calls = message.get("tool_calls")
            if isinstance(tool_calls, list) and tool_calls:
                serialized = json.dumps(tool_calls, ensure_ascii=False)
                if serialized.strip():
                    return serialized
            raise RuntimeError(
                "OpenAI-compatible response choices[0].message.content is empty. "
                f"response_excerpt={OpenAICompatBackend._response_debug_excerpt(body)}"
            )

        @staticmethod
        def _normalize_message_content(content: object) -> str | None:
            if isinstance(content, str):
                stripped = content.strip()
                return stripped or None
            if isinstance(content, list):
                parts: list[str] = []
                for item in content:
                    normalized = OpenAICompatBackend._normalize_content_part(item)
                    if normalized:
                        parts.append(normalized)
                merged = "\n".join(part for part in parts if part.strip()).strip()
                return merged or None
            return None

        @staticmethod
        def _normalize_content_part(item: object) -> str:
            if isinstance(item, str):
                return item
            if isinstance(item, dict):
                item_type = item.get("type")
                if item_type == "text":
                    text_value = item.get("text")
                    if isinstance(text_value, str):
                        return text_value
                    if isinstance(text_value, dict):
                        nested_text = text_value.get("value")
                        if isinstance(nested_text, str):
                            return nested_text
                for key in ("text", "content", "value", "reasoning_content", "output_text"):
                    value = item.get(key)
                    if isinstance(value, str) and value.strip():
                        return value
                    if isinstance(value, list):
                        nested_parts = [
                            OpenAICompatBackend._normalize_content_part(part)
                            for part in value
                        ]
                        merged = "\n".join(part for part in nested_parts if part.strip()).strip()
                        if merged:
                            return merged
                try:
                    return json.dumps(item, ensure_ascii=False)
                except TypeError:
                    return repr(item)
            return str(item)

        @staticmethod
        def _response_debug_excerpt(body: dict[str, object], max_length: int = 1200) -> str:
            try:
                serialized = json.dumps(body, ensure_ascii=False)
            except TypeError:
                serialized = repr(body)
            if len(serialized) <= max_length:
                return serialized
            return serialized[:max_length] + "...<truncated>"

        def _build_response_format(
            self,
            prompt: (
                RenderedTheoryPrompt
                | RenderedAnalysisHandoverPrompt
                | RenderedDecisionPrompt

            ),
        ) -> dict[str, object]:
            if not self.use_json_schema:
                return {"type": "json_object"}
            if isinstance(prompt, RenderedTheoryPrompt):
                return self._json_schema_response_format("theory_agent_output", self._theory_output_schema())
            if isinstance(prompt, RenderedAnalysisHandoverPrompt):
                return self._json_schema_response_format(
                    "analysis_handover_output",
                    self._analysis_handover_output_schema(),
                )
            if isinstance(prompt, RenderedDecisionPrompt):
                return self._json_schema_response_format("decision_agent_output", self._decision_output_schema())
            return {"type": "json_object"}

        @staticmethod
        def _json_schema_response_format(name: str, schema: dict[str, object]) -> dict[str, object]:
            return {
                "type": "json_schema",
                "json_schema": {
                    "name": name,
                    "strict": True,
                    "schema": schema,
                },
            }

        @staticmethod
        def _fixed_label_probability_schema() -> dict[str, object]:
            labels = (
                "<-50bp",
                "[-50,-25)bp",
                "[-25,0)bp",
                "0bp",
                "(0,25]bp",
                "(25,50]bp",
                ">50bp",
            )
            return {
                "type": "object",
                "properties": {
                    label: {"type": "number"}
                    for label in labels
                },
                "required": list(labels),
                "additionalProperties": False,
            }

        @staticmethod
        def _fixed_theory_weight_schema() -> dict[str, object]:
            theory_ids = (
                "standard_taylor",
                "forward_looking_taylor",
                "interest_rate_smoothing",
                "open_economy_rule",
                "financial_stability_rule",
                "new_keynesian_optimal_rule",
            )
            return {
                "type": "object",
                "properties": {theory_id: {"type": "number"} for theory_id in theory_ids},
                "required": list(theory_ids),
                "additionalProperties": False,
            }

        @staticmethod
        def _analysis_reason_summary_schema() -> dict[str, object]:
            return {
                "type": "object",
                "properties": {
                    theory_id: {
                        "type": "object",
                        "properties": {
                            "primary_policy_reason": {"type": "string"},
                            "secondary_policy_reason": {"type": "string"},
                            "theory_explanation": {"type": "string"},
                        },
                        "required": [
                            "primary_policy_reason",
                            "secondary_policy_reason",
                            "theory_explanation",
                        ],
                        "additionalProperties": False,
                    }
                    for theory_id in (
                        "standard_taylor",
                        "forward_looking_taylor",
                        "interest_rate_smoothing",
                        "open_economy_rule",
                        "financial_stability_rule",
                        "new_keynesian_optimal_rule",
                    )
                },
                "required": [
                    "standard_taylor",
                    "forward_looking_taylor",
                    "interest_rate_smoothing",
                    "open_economy_rule",
                    "financial_stability_rule",
                    "new_keynesian_optimal_rule",
                ],
                "additionalProperties": False,
            }

        @staticmethod
        def _analysis_single_text_note_schema(field_name: str) -> dict[str, object]:
            theory_ids = (
                "standard_taylor",
                "forward_looking_taylor",
                "interest_rate_smoothing",
                "open_economy_rule",
                "financial_stability_rule",
                "new_keynesian_optimal_rule",
            )
            return {
                "type": "object",
                "properties": {
                    theory_id: {
                        "type": "object",
                        "properties": {
                            field_name: {"type": "string"},
                        },
                        "required": [field_name],
                        "additionalProperties": False,
                    }
                    for theory_id in theory_ids
                },
                "required": list(theory_ids),
                "additionalProperties": False,
            }

        def _analysis_handover_output_schema(self) -> dict[str, object]:
            return {
                "type": "object",
                "properties": {
                    "reason_summary_by_rule": self._analysis_reason_summary_schema(),
                    "applicability_summary_by_rule": self._analysis_single_text_note_schema("applicability_note"),
                    "risk_summary_by_rule": self._analysis_single_text_note_schema("risk_note"),
                    "main_disagreement_dimension": {"type": "string"},
                    "secondary_disagreement_dimension": {"type": ["string", "null"]},
                    "high_conflict_flag": {"type": "boolean"},
                    "minority_high_confidence_alert": {"type": "boolean"},
                },
                "required": [
                    "reason_summary_by_rule",
                    "applicability_summary_by_rule",
                    "risk_summary_by_rule",
                    "main_disagreement_dimension",
                    "secondary_disagreement_dimension",
                    "high_conflict_flag",
                    "minority_high_confidence_alert",
                ],
                "additionalProperties": False,
            }

        def _theory_output_schema(self) -> dict[str, object]:
            labels = list(self._fixed_label_probability_schema()["required"])
            return {
                "type": "object",
                "properties": {
                    "recommendation": {"type": "string", "enum": labels},
                    "probability_distribution": self._fixed_label_probability_schema(),
                    "confidence": {"type": "integer", "minimum": 0, "maximum": 100},
                    "primary_policy_reason": {"type": "string"},
                    "secondary_policy_reason": {"type": "string"},
                    "key_variables_used": {"type": "array", "items": {"type": "string"}},
                    "theory_explanation": {"type": "string"},
                    "applicability_note": {"type": "string"},
                    "risk_note": {"type": "string"},
                },
                "required": [
                    "recommendation",
                    "probability_distribution",
                    "confidence",
                    "primary_policy_reason",
                    "secondary_policy_reason",
                    "key_variables_used",
                    "theory_explanation",
                    "applicability_note",
                    "risk_note",
                ],
                "additionalProperties": False,
            }

        def _decision_output_schema(self) -> dict[str, object]:
            labels = list(self._fixed_label_probability_schema()["required"])
            theory_ids = list(self._fixed_theory_weight_schema()["required"])
            return {
                "type": "object",
                "properties": {
                    "final_recommendation": {"type": "string", "enum": labels},
                    "final_probability_distribution": self._fixed_label_probability_schema(),
                    "final_confidence": {"type": "integer", "minimum": 0, "maximum": 100},
                    "final_explanation": {"type": "string"},
                    "dominant_rule_family": {"type": "string", "enum": theory_ids},
                    "supporting_rule_families": {"type": "array", "items": {"type": "string", "enum": theory_ids}},
                    "theory_weight_profile": self._fixed_theory_weight_schema(),
                    "theory_weight_rationale": {"type": "string"},
                    "rule_contribution_shares": self._fixed_theory_weight_schema(),
                    "first_contributor_rule": {"type": "string", "enum": theory_ids},
                    "current_sample_reasoning": {"type": "string"},
                    "remaining_uncertainty": {"type": "string"},
                    "decision_path": {"type": "string", "enum": ["training_textual_memory_informed_aggregation"]},
                },
                "required": [
                    "final_recommendation",
                    "final_probability_distribution",
                    "final_confidence",
                    "final_explanation",
                    "dominant_rule_family",
                    "supporting_rule_families",
                    "theory_weight_profile",
                    "theory_weight_rationale",
                    "rule_contribution_shares",
                    "first_contributor_rule",
                    "current_sample_reasoning",
                    "remaining_uncertainty",
                    "decision_path",
                ],
                "additionalProperties": False,
            }


In [ ]:
# 文件：model_backends/__init__.py
# 产出：导出 OpenAI-compatible 后端接口。

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    __all__ = [
        "DEFAULT_BACKOFF_MULTIPLIER",
        "DEFAULT_INITIAL_BACKOFF_SECONDS",
        "DEFAULT_MAX_RETRIES",
        "DEFAULT_TEMPERATURE",
        "OPENAI_COMPAT_DEFAULT_API_KEY_ENV",
        "OPENAI_COMPAT_DEFAULT_BASE_URL",
        "OPENAI_COMPAT_DEFAULT_MODEL",
        "OpenAICompatBackend",
        "first_nonempty",
        "normalize_openai_compat_base_url",
    ]


## Runtime Assembly and Application

In [ ]:
# file: runtime_assembly/factory.py
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from dataclasses import dataclass

    @dataclass(frozen=True, slots=True)
    class OpenAICompatRuntimeConfig:
        model_name: str = OPENAI_COMPAT_DEFAULT_MODEL
        api_base: str | None = OPENAI_COMPAT_DEFAULT_BASE_URL
        api_key_env: str = OPENAI_COMPAT_DEFAULT_API_KEY_ENV
        timeout_seconds: int = 120
        temperature: float = DEFAULT_TEMPERATURE
        max_output_tokens: int | None = None
        max_retries: int = DEFAULT_MAX_RETRIES
        initial_backoff_seconds: float = DEFAULT_INITIAL_BACKOFF_SECONDS
        backoff_multiplier: float = DEFAULT_BACKOFF_MULTIPLIER
        use_json_schema: bool = False
        fallback_to_json_object: bool = True
        include_response_format: bool = True

        def build_backend(self) -> OpenAICompatBackend:
            return OpenAICompatBackend(model_name=self.model_name, api_base=self.api_base, api_key_env=self.api_key_env, timeout_seconds=self.timeout_seconds, temperature=float(self.temperature), max_output_tokens=self.max_output_tokens, max_retries=self.max_retries, initial_backoff_seconds=float(self.initial_backoff_seconds), backoff_multiplier=float(self.backoff_multiplier), use_json_schema=self.use_json_schema, fallback_to_json_object=self.fallback_to_json_object, include_response_format=self.include_response_format)

    @dataclass(frozen=True, slots=True)
    class RuntimeBackendBundle:
        theory_backend: TheoryModelBackend
        analysis_backend: AnalysisHandoverModelBackend
        decision_backend: DecisionModelBackend

    def build_shared_backend_bundle(shared_backend: object) -> RuntimeBackendBundle:
        return RuntimeBackendBundle(theory_backend=shared_backend, analysis_backend=shared_backend, decision_backend=shared_backend)

    @dataclass(frozen=True, slots=True)
    class RuntimeAssembly:
        scheduler: USSampleScheduler
        backend_bundle: RuntimeBackendBundle
        theory_runner: SingleTheoryAgentRunner
        analysis_runner: SingleAnalysisHandoverRunner
        analysis_state_builder: AnalysisHandoverStateBuilder
        decision_runner: SingleDecisionAgentRunner
        prediction_pipeline: HierarchicalPredictionPipeline
        inference_driver: DirectInferenceDriver

    def build_runtime_assembly(scheduler: USSampleScheduler, model_config: OpenAICompatRuntimeConfig) -> RuntimeAssembly:
        shared_backend = model_config.build_backend()
        backend_bundle = build_shared_backend_bundle(shared_backend)
        theory_runner = SingleTheoryAgentRunner(backend=backend_bundle.theory_backend)
        analysis_runner = SingleAnalysisHandoverRunner(backend=backend_bundle.analysis_backend)
        analysis_state_builder = AnalysisHandoverStateBuilder()
        decision_runner = SingleDecisionAgentRunner(backend=backend_bundle.decision_backend)
        prediction_pipeline = HierarchicalPredictionPipeline(theory_runner=theory_runner, analysis_runner=analysis_runner, analysis_state_builder=analysis_state_builder, decision_runner=decision_runner)
        inference_driver = DirectInferenceDriver(scheduler=scheduler, prediction_pipeline=prediction_pipeline)
        return RuntimeAssembly(scheduler=scheduler, backend_bundle=backend_bundle, theory_runner=theory_runner, analysis_runner=analysis_runner, analysis_state_builder=analysis_state_builder, decision_runner=decision_runner, prediction_pipeline=prediction_pipeline, inference_driver=inference_driver)


In [ ]:
# file: runtime_assembly/__init__.py
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    __all__ = [
        "OpenAICompatRuntimeConfig",
        "RuntimeAssembly",
        "RuntimeBackendBundle",
        "build_runtime_assembly",
        "build_shared_backend_bundle",
    ]


In [ ]:
# file: application_entry/service.py
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    from dataclasses import dataclass
    from datetime import date
    from pathlib import Path
    from typing import Any, Callable

    @dataclass(frozen=True, slots=True)
    class USRuntimeApplicationConfig:
        project_root: Path | None = None
        hp_lambda: float = 129600.0
        decision_cutoff_date: date | None = None
        model_name: str = OPENAI_COMPAT_DEFAULT_MODEL
        api_base: str | None = OPENAI_COMPAT_DEFAULT_BASE_URL
        api_key_env: str = OPENAI_COMPAT_DEFAULT_API_KEY_ENV
        timeout_seconds: int = 120
        temperature: float = 0.2
        max_output_tokens: int | None = None
        max_retries: int = 3
        initial_backoff_seconds: float = 1.0
        backoff_multiplier: float = 2.0
        use_json_schema: bool = False
        fallback_to_json_object: bool = True
        include_response_format: bool = True

        def build_model_config(self) -> OpenAICompatRuntimeConfig:
            return OpenAICompatRuntimeConfig(model_name=self.model_name, api_base=self.api_base, api_key_env=self.api_key_env, timeout_seconds=self.timeout_seconds, temperature=float(self.temperature), max_output_tokens=self.max_output_tokens, max_retries=self.max_retries, initial_backoff_seconds=float(self.initial_backoff_seconds), backoff_multiplier=float(self.backoff_multiplier), use_json_schema=self.use_json_schema, fallback_to_json_object=self.fallback_to_json_object, include_response_format=self.include_response_format)

    @dataclass(slots=True)
    class USRuntimeApplication:
        config: USRuntimeApplicationConfig
        sample_builder: USSampleBuilder
        scheduler: USSampleScheduler
        runtime_assembly: RuntimeAssembly

        def run_direct_inference(self, progress_callback: Callable[[dict[str, Any]], None] | None = None, skip_sample_ids: set[str] | None = None, completed_item_callback: Callable[[DirectInferenceItem], None] | None = None) -> DirectInferenceRunResult:
            return self.runtime_assembly.inference_driver.run(progress_callback=progress_callback, skip_sample_ids=skip_sample_ids, completed_item_callback=completed_item_callback)

        def save_prediction_output(self, output_path: Path | str, result: DirectInferenceRunResult) -> Path:
            output_path = Path(output_path)
            _json_write(output_path, to_jsonable(result))
            return output_path

    def build_us_runtime_application(config: USRuntimeApplicationConfig | None = None) -> USRuntimeApplication:
        effective_config = USRuntimeApplicationConfig() if config is None else config
        sample_builder = USSampleBuilder(project_root=effective_config.project_root, hp_lambda=float(effective_config.hp_lambda), decision_cutoff_date=effective_config.decision_cutoff_date)
        scheduler = USSampleScheduler(sample_builder=sample_builder)
        runtime_assembly = build_runtime_assembly(scheduler=scheduler, model_config=effective_config.build_model_config())
        return USRuntimeApplication(config=effective_config, sample_builder=sample_builder, scheduler=scheduler, runtime_assembly=runtime_assembly)


In [ ]:
# file: application_entry/__init__.py
from __future__ import annotations

if globals().get("NOTEBOOK_SKIP_ARCHIVE_SOURCE_EXECUTION", True):
    pass
else:
    __all__ = ["USRuntimeApplication", "USRuntimeApplicationConfig", "build_us_runtime_application"]


In [ ]:
# file: system_c_training_textual_memory.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Callable

SYSTEM_C_TRAINING_STEP_SCHEMA = "system_c_textual_memory_training_step_v1"
SYSTEM_C_TRAINING_MANIFEST_SCHEMA = "system_c_textual_memory_training_manifest_v1"
SYSTEM_C_TRAINING_RUN_SCHEMA = "system_c_textual_memory_training_run_v1"
SYSTEM_C_FROZEN_SNAPSHOT_SCHEMA = "system_c_textual_memory_frozen_snapshot_v1"
SYSTEM_C_PREDICTION_STEP_SCHEMA = "system_c_textual_memory_prediction_step_v1"
SYSTEM_C_PREDICTION_MANIFEST_SCHEMA = "system_c_textual_memory_prediction_manifest_v1"
SYSTEM_C_TEXTUAL_MEMORY_SCHEMA = "system_c_textual_theory_applicability_memory_v1"
SYSTEM_C_REFLECTION_NOTE_SCHEMA = "system_c_case_reflection_note_v1"


@dataclass(frozen=True, slots=True)
class TextualMemoryTrainingRunResult:
    training_schedule: tuple[ScheduledSampleRef, ...]
    new_step_records: tuple[dict[str, object], ...]
    textual_theory_applicability_memory_train_final: dict[str, object]
    memory_path: Path


def _parse_json_object_from_text(raw_output: str) -> dict[str, Any]:
    text = str(raw_output or "").strip()
    if text.startswith("```") and text.endswith("```"):
        lines = text.splitlines()
        if len(lines) >= 3:
            text = "\n".join(lines[1:-1]).strip()
    start = text.find("{")
    if start >= 0:
        depth = 0
        in_string = False
        escaped = False
        for index in range(start, len(text)):
            char = text[index]
            if in_string:
                if escaped:
                    escaped = False
                elif char == "\\":
                    escaped = True
                elif char == '"':
                    in_string = False
                continue
            if char == '"':
                in_string = True
            elif char == "{":
                depth += 1
            elif char == "}":
                depth -= 1
                if depth == 0:
                    text = text[start:index + 1]
                    break
    parsed = json.loads(text)
    if not isinstance(parsed, dict):
        raise ValueError("Memory agent output must be a JSON object.")
    return parsed


def _call_json_memory_agent(backend: object, *, system_prompt: str, user_prompt: str) -> dict[str, Any]:
    if not hasattr(backend, "_chat_completion"):
        raise TypeError("Memory agent requires an OpenAI-compatible backend with _chat_completion.")
    raw = backend._chat_completion(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        response_format={"type": "json_object"},
    )
    return _parse_json_object_from_text(raw)


def _safe_text(value: Any, max_chars: int = 900) -> str:
    text = str(value or "").strip()
    if len(text) > max_chars:
        return text[:max_chars].rstrip() + "..."
    return text


def _extract_true_label_probability_support(pipeline_result: SingleSamplePipelineResult, actual_label: RateMoveLabel | str) -> dict[str, float]:
    label = validate_label(actual_label)
    return {
        theory_id: float(pipeline_result.theory_layer_state.theory_outputs[theory_id].probability_distribution.probability_for(label))
        for theory_id in THEORY_IDS
    }


def _build_case_record(*, sample_ref: ScheduledSampleRef, pipeline_result: SingleSamplePipelineResult, actual_label: RateMoveLabel | str) -> dict[str, object]:
    decision_output = pipeline_result.decision_state.decision_output
    predicted_label = _label_value(decision_output.final_recommendation)
    actual_value = _label_value(actual_label)
    support = _extract_true_label_probability_support(pipeline_result, actual_label)
    theory_records: dict[str, object] = {}
    for theory_id, output in pipeline_result.theory_layer_state.theory_outputs.items():
        theory_records[theory_id] = {
            "recommendation": output.recommendation.value,
            "probability_distribution": output.probability_distribution.as_ordered_dict(),
            "probability_on_true_label": support[theory_id],
            "confidence": output.confidence,
            "primary_policy_reason": _safe_text(output.primary_policy_reason),
            "secondary_policy_reason": _safe_text(output.secondary_policy_reason),
            "theory_explanation": _safe_text(output.theory_explanation),
            "applicability_note": _safe_text(output.applicability_note),
            "risk_note": _safe_text(output.risk_note),
            "policy_path_regime_assessment": dict(output.policy_path_regime_assessment),
        }
    handover = pipeline_result.analysis_state.decision_handover_packet
    analysis_summary = {
        "main_disagreement_dimension": getattr(handover, "main_disagreement_dimension", None),
        "secondary_disagreement_dimension": getattr(handover, "secondary_disagreement_dimension", None),
        "high_conflict_flag": getattr(handover, "high_conflict_flag", None),
        "minority_high_confidence_alert": getattr(handover, "minority_high_confidence_alert", None),
        "label_support_table": getattr(handover, "label_support_table", None),
        "applicability_summary_by_rule": getattr(handover, "applicability_summary_by_rule", None),
        "risk_summary_by_rule": getattr(handover, "risk_summary_by_rule", None),
    }
    return {
        "sample_id": sample_ref.sample_id,
        "split": sample_ref.split,
        "decision_date": sample_ref.decision_date.isoformat(),
        "actual_label": actual_value,
        "decision_prediction": predicted_label,
        "decision_correct": predicted_label == actual_value,
        "decision_layer": to_jsonable(decision_output),
        "decision_reasoning": {
            "final_explanation": _safe_text(decision_output.final_explanation),
            "current_sample_reasoning": _safe_text(decision_output.current_sample_reasoning),
            "theory_weight_rationale": _safe_text(decision_output.theory_weight_rationale),
            "remaining_uncertainty": _safe_text(decision_output.remaining_uncertainty),
        },
        "analysis_summary": to_jsonable(analysis_summary),
        "theory_records": theory_records,
        "true_label_probability_support": support,
    }


def _reflect_on_case(*, backend: object, case_record: dict[str, object]) -> dict[str, object]:
    compact_case = {
        "sample_id": case_record.get("sample_id"),
        "decision_date": case_record.get("decision_date"),
        "actual_label": case_record.get("actual_label"),
        "decision_prediction": case_record.get("decision_prediction"),
        "decision_correct": case_record.get("decision_correct"),
        "decision_reasoning": case_record.get("decision_reasoning"),
        "analysis_summary": case_record.get("analysis_summary"),
        "theory_records": case_record.get("theory_records"),
    }
    system_prompt = "\n".join([
        "你是系统C的案例反思Agent。",
        "你的任务是为一个训练集决议点写一条简短中文案例反思，供后续记忆综合使用。",
        "训练集真实标签只能用来评估这个训练案例。",
        "不得修改理论Agent提示词，不得生成长期数值权重，不得写基于日期的未来答案。",
        "可以总结政策阶段相关的可迁移经验，例如liftoff_waiting、late_hiking_cycle、terminal_rate_plateau、insurance_cut_cycle、crisis_cut、easing_cycle_continuation下哪些理论容易误判，但必须写成一般规则。",
        "只返回一个JSON对象。",
    ])
    user_prompt = "\n".join([
        "[训练案例记录]",
        json.dumps(compact_case, ensure_ascii=False, indent=2),
        "",
        "[必须输出的JSON字段]",
        json.dumps({
            "schema_version": SYSTEM_C_REFLECTION_NOTE_SCHEMA,
            "case_quality_summary": "一段简短中文总结",
            "helpful_theory_signals": {theory_id: "该理论为何有帮助；没有则留空" for theory_id in THEORY_IDS},
            "misleading_theory_signals": {theory_id: "该理论为何产生误导；没有则留空" for theory_id in THEORY_IDS},
            "special_condition_if_any": "正常状态/无特殊情况，或用简短中文描述特殊情况",
            "candidate_general_lesson": "一条可迁移的中文经验，不能是日期答案",
            "do_not_overgeneralize": "这个单一案例不应被过度泛化的内容",
            "policy_stage_lesson": "若本案例涉及政策阶段，写一条一般化阶段经验；否则留空",
        }, ensure_ascii=False, indent=2),
    ])
    payload = _call_json_memory_agent(backend, system_prompt=system_prompt, user_prompt=user_prompt)
    payload["schema_version"] = SYSTEM_C_REFLECTION_NOTE_SCHEMA
    payload["sample_id"] = case_record.get("sample_id")
    payload["decision_date"] = case_record.get("decision_date")
    payload["actual_label"] = case_record.get("actual_label")
    payload["decision_prediction"] = case_record.get("decision_prediction")
    payload["decision_correct"] = case_record.get("decision_correct")
    return payload


def _training_step_to_record(*, sample_ref: ScheduledSampleRef, pipeline_result: SingleSamplePipelineResult, actual_label: RateMoveLabel | str, reflection_note: dict[str, object], case_record: dict[str, object]) -> dict[str, object]:
    return {
        "schema_version": SYSTEM_C_TRAINING_STEP_SCHEMA,
        "notebook_runtime_revision": globals().get("NOTEBOOK_RUNTIME_REVISION"),
        "theory_ids": list(THEORY_IDS),
        "sample_id": sample_ref.sample_id,
        "split": sample_ref.split,
        "decision_date": sample_ref.decision_date.isoformat(),
        "actual_label": _label_value(actual_label),
        "case_record": case_record,
        "reflection_note": reflection_note,
        "decision_layer": to_jsonable(pipeline_result.decision_state.decision_output),
        "theory_layer": to_jsonable(pipeline_result.theory_layer_state.theory_outputs),
        "analysis_layer": to_jsonable(pipeline_result.analysis_state),
        "status": "success",
    }


def _read_training_manifest(training_root: Path) -> dict[str, object]:
    manifest_path = training_root / "manifest.json"
    fallback = {"schema_version": SYSTEM_C_TRAINING_MANIFEST_SCHEMA, "steps": {}}
    if not manifest_path.exists():
        return fallback
    try:
        payload = json.loads(manifest_path.read_text(encoding="utf-8"))
    except Exception:
        return fallback
    if not isinstance(payload, dict):
        return fallback
    if not isinstance(payload.get("steps"), dict):
        payload["steps"] = {}
    payload.setdefault("schema_version", SYSTEM_C_TRAINING_MANIFEST_SCHEMA)
    return payload


def _step_record_matches_current_runtime(record: dict[str, object]) -> bool:
    if not isinstance(record, dict):
        return False
    if record.get("status") != "success":
        return False
    return tuple(record.get("theory_ids") or ()) == tuple(THEORY_IDS)

def successful_training_sample_ids(artifact_root: Path) -> set[str]:
    training_root = artifact_root / "training_history"
    manifest = _read_training_manifest(training_root)
    steps = manifest.get("steps") if isinstance(manifest, dict) else None
    if not isinstance(steps, dict):
        return set()
    successful: set[str] = set()
    for sample_id, item in steps.items():
        if not isinstance(item, dict) or item.get("status") != "success":
            continue
        raw_path = item.get("path")
        if not isinstance(raw_path, str) or not raw_path.strip():
            continue
        try:
            record = json.loads((training_root / raw_path).read_text(encoding="utf-8"))
        except Exception:
            continue
        if _step_record_matches_current_runtime(record):
            successful.add(str(sample_id))
    return successful

def load_successful_training_step_records(artifact_root: Path) -> list[dict[str, object]]:
    training_root = artifact_root / "training_history"
    manifest = _read_training_manifest(training_root)
    records: list[dict[str, object]] = []
    steps = manifest.get("steps", {})
    if not isinstance(steps, dict):
        return records
    for item in steps.values():
        if not isinstance(item, dict) or item.get("status") != "success":
            continue
        relative_path = item.get("path")
        if not isinstance(relative_path, str) or not relative_path.strip():
            continue
        try:
            record = json.loads((training_root / relative_path).read_text(encoding="utf-8"))
        except Exception:
            continue
        if _step_record_matches_current_runtime(record):
            records.append(record)
    records.sort(key=lambda record: str(record.get("decision_date", "")))
    return records

def persist_training_step_record(*, record: dict[str, object], artifact_root: Path) -> Path:
    training_root = artifact_root / "training_history"
    steps_dir = training_root / "steps"
    reflections_dir = training_root / "reflection_notes"
    steps_dir.mkdir(parents=True, exist_ok=True)
    reflections_dir.mkdir(parents=True, exist_ok=True)
    sample_id = str(record["sample_id"])
    step_path = steps_dir / f"{sample_id}.json"
    _json_write(step_path, record)
    reflection_note = record.get("reflection_note")
    if isinstance(reflection_note, dict):
        _json_write(reflections_dir / f"{sample_id}.json", reflection_note)
    manifest = _read_training_manifest(training_root)
    manifest.setdefault("steps", {})
    manifest["steps"][sample_id] = {"status": "success", "decision_date": record.get("decision_date"), "path": str(Path("steps") / f"{sample_id}.json")}
    _json_write(training_root / "manifest.json", manifest)
    write_training_summaries_from_records(records=load_successful_training_step_records(artifact_root), summaries_dir=training_root / "summaries", runs_dir=training_root / "runs")
    return step_path


def persist_training_run_record(*, artifact_root: Path, run_id: str, title: str, status: str, step_ids: list[str] | None = None, error_message: str | None = None, memory_path: str | None = None) -> Path:
    training_root = artifact_root / "training_history"
    runs_dir = training_root / "runs"
    runs_dir.mkdir(parents=True, exist_ok=True)
    run_record_path = runs_dir / f"run_{run_id}_{title}.json"
    created_at = datetime.now(timezone.utc).isoformat()
    if run_record_path.exists():
        try:
            existing = json.loads(run_record_path.read_text(encoding="utf-8"))
            if isinstance(existing, dict) and isinstance(existing.get("created_at"), str):
                created_at = existing["created_at"]
        except Exception:
            pass
    normalized_step_ids = list(step_ids or [])
    run_record = {
        "schema_version": SYSTEM_C_TRAINING_RUN_SCHEMA,
        "run_id": run_id,
        "title": title,
        "status": status,
        "notebook_runtime_revision": globals().get("NOTEBOOK_RUNTIME_REVISION"),
        "system_variant": globals().get("NOTEBOOK_SYSTEM_VARIANT"),
        "step_ids": normalized_step_ids,
        "completed_step_count": len(normalized_step_ids),
        "memory_path": memory_path,
        "created_at": created_at,
        "updated_at": datetime.now(timezone.utc).isoformat(),
    }
    if error_message:
        run_record["error_message"] = error_message
    _json_write(run_record_path, run_record)
    return run_record_path


def write_training_summaries_from_records(*, records: list[dict[str, object]], summaries_dir: Path, runs_dir: Path) -> None:
    summaries_dir.mkdir(parents=True, exist_ok=True)
    runs_dir.mkdir(parents=True, exist_ok=True)
    case_rows, reflection_rows, reflection_theory_rows, decision_rows, support_rows = [], [], [], [], []
    for record in records:
        base = {"sample_id": record.get("sample_id"), "decision_date": record.get("decision_date"), "actual_label": record.get("actual_label")}
        case_record = record.get("case_record") if isinstance(record.get("case_record"), dict) else {}
        decision_output = record.get("decision_layer") if isinstance(record.get("decision_layer"), dict) else {}
        reflection = record.get("reflection_note") if isinstance(record.get("reflection_note"), dict) else {}
        support = case_record.get("true_label_probability_support") if isinstance(case_record, dict) else {}
        support = support if isinstance(support, dict) else {}
        case_rows.append({**base, "decision_prediction": case_record.get("decision_prediction"), "decision_correct": case_record.get("decision_correct")})
        helpful = reflection.get("helpful_theory_signals") if isinstance(reflection.get("helpful_theory_signals"), dict) else {}
        misleading = reflection.get("misleading_theory_signals") if isinstance(reflection.get("misleading_theory_signals"), dict) else {}
        reflection_rows.append({**base, "decision_correct": case_record.get("decision_correct"), "case_quality_summary": reflection.get("case_quality_summary"), "special_condition_if_any": reflection.get("special_condition_if_any"), "candidate_general_lesson": reflection.get("candidate_general_lesson"), "do_not_overgeneralize": reflection.get("do_not_overgeneralize"), "helpful_theory_signals": json.dumps(helpful, ensure_ascii=False), "misleading_theory_signals": json.dumps(misleading, ensure_ascii=False)})
        for theory_id in THEORY_IDS:
            reflection_theory_rows.append({**base, "decision_correct": case_record.get("decision_correct"), "theory_id": theory_id, "helpful_signal": helpful.get(theory_id), "misleading_signal": misleading.get(theory_id), "probability_on_true_label": support.get(theory_id)})
        decision_rows.append({**base, "final_recommendation": decision_output.get("final_recommendation"), "final_confidence": decision_output.get("final_confidence"), "dominant_rule_family": decision_output.get("dominant_rule_family"), "decision_path": decision_output.get("decision_path")})
        support_rows.append({**base, **{theory_id: support.get(theory_id) for theory_id in THEORY_IDS}})
    _write_csv(summaries_dir / "case_records_all.csv", case_rows, ["sample_id", "decision_date", "actual_label", "decision_prediction", "decision_correct"])
    _write_csv(summaries_dir / "reflection_notes_all.csv", reflection_rows, ["sample_id", "decision_date", "actual_label", "decision_correct", "case_quality_summary", "special_condition_if_any", "candidate_general_lesson", "do_not_overgeneralize", "helpful_theory_signals", "misleading_theory_signals"])
    _write_csv(summaries_dir / "reflection_theory_signals_all.csv", reflection_theory_rows, ["sample_id", "decision_date", "actual_label", "decision_correct", "theory_id", "probability_on_true_label", "helpful_signal", "misleading_signal"])
    _write_csv(summaries_dir / "decision_layer_all.csv", decision_rows, ["sample_id", "decision_date", "actual_label", "final_recommendation", "final_confidence", "dominant_rule_family", "decision_path"])
    _write_csv(summaries_dir / "true_label_probability_support_all.csv", support_rows, ["sample_id", "decision_date", "actual_label", *THEORY_IDS])
    _write_jsonl(summaries_dir / "case_records_all.jsonl", [{"sample_id": r.get("sample_id"), "case_record": r.get("case_record")} for r in records])
    _write_jsonl(summaries_dir / "reflection_notes_all.jsonl", [{"sample_id": r.get("sample_id"), "reflection_note": r.get("reflection_note")} for r in records])
    run_rows = []
    for run_path in sorted(runs_dir.glob("run_*.json")):
        try:
            payload = json.loads(run_path.read_text(encoding="utf-8"))
        except Exception:
            continue
        run_rows.append({"run_id": payload.get("run_id"), "title": payload.get("title"), "status": payload.get("status"), "completed_step_count": payload.get("completed_step_count"), "notebook_runtime_revision": payload.get("notebook_runtime_revision"), "system_variant": payload.get("system_variant"), "created_at": payload.get("created_at"), "updated_at": payload.get("updated_at"), "error_message": payload.get("error_message"), "memory_path": payload.get("memory_path")})
    _write_csv(summaries_dir / "run_revision_log.csv", run_rows, ["run_id", "title", "status", "completed_step_count", "notebook_runtime_revision", "system_variant", "created_at", "updated_at", "error_message", "memory_path"])



def _build_compact_training_evidence(records: list[dict[str, object]]) -> dict[str, object]:
    total = len(records)
    correct_count = 0
    theory_support_sum = {theory_id: 0.0 for theory_id in THEORY_IDS}
    theory_top_true_count = {theory_id: 0 for theory_id in THEORY_IDS}
    high_confidence_error_count = {theory_id: 0 for theory_id in THEORY_IDS}
    normal_cases, special_cases, error_cases = [], [], []
    for record in records:
        case_record = record.get("case_record") if isinstance(record.get("case_record"), dict) else {}
        reflection = record.get("reflection_note") if isinstance(record.get("reflection_note"), dict) else {}
        if case_record.get("decision_correct") is True:
            correct_count += 1
        else:
            error_cases.append({"sample_id": record.get("sample_id"), "decision_date": record.get("decision_date"), "actual_label": record.get("actual_label"), "decision_prediction": case_record.get("decision_prediction"), "reflection": reflection.get("candidate_general_lesson")})
        support = case_record.get("true_label_probability_support") if isinstance(case_record, dict) else {}
        support = support if isinstance(support, dict) else {}
        for theory_id in THEORY_IDS:
            value = support.get(theory_id, 0.0)
            if isinstance(value, (int, float)):
                theory_support_sum[theory_id] += float(value)
                if float(value) == max([float(support.get(t, 0.0) or 0.0) for t in THEORY_IDS]):
                    theory_top_true_count[theory_id] += 1
        theory_records = case_record.get("theory_records") if isinstance(case_record, dict) else {}
        if isinstance(theory_records, dict):
            for theory_id, theory_record in theory_records.items():
                if not isinstance(theory_record, dict):
                    continue
                if theory_record.get("recommendation") != case_record.get("actual_label") and isinstance(theory_record.get("confidence"), (int, float)) and theory_record.get("confidence") >= 80:
                    high_confidence_error_count[theory_id] = high_confidence_error_count.get(theory_id, 0) + 1
        condition = str(reflection.get("special_condition_if_any") or "").lower()
        case_summary = {"sample_id": record.get("sample_id"), "decision_date": record.get("decision_date"), "actual_label": record.get("actual_label"), "decision_prediction": case_record.get("decision_prediction"), "decision_correct": case_record.get("decision_correct"), "candidate_general_lesson": reflection.get("candidate_general_lesson"), "special_condition_if_any": reflection.get("special_condition_if_any"), "do_not_overgeneralize": reflection.get("do_not_overgeneralize")}
        if condition and "normal" not in condition and "no special" not in condition:
            special_cases.append(case_summary)
        else:
            normal_cases.append(case_summary)
    avg_support = {theory_id: (theory_support_sum[theory_id] / total if total else 0.0) for theory_id in THEORY_IDS}
    return {
        "schema_version": "system_c_compact_training_evidence_v1",
        "training_sample_count": total,
        "decision_correct_count": correct_count,
        "decision_accuracy_in_training": (correct_count / total if total else None),
        "average_true_label_probability_by_theory": avg_support,
        "top_true_label_support_count_by_theory": theory_top_true_count,
        "high_confidence_error_count_by_theory": high_confidence_error_count,
        "normal_case_reflection_samples": normal_cases[:40],
        "special_case_reflection_samples": special_cases[:40],
        "decision_error_samples": error_cases[:40],
        "all_reflection_lessons": [
            {"sample_id": r.get("sample_id"), "decision_date": r.get("decision_date"), "lesson": (r.get("reflection_note") or {}).get("candidate_general_lesson"), "do_not_overgeneralize": (r.get("reflection_note") or {}).get("do_not_overgeneralize")}
            for r in records
            if isinstance(r.get("reflection_note"), dict)
        ],
    }


def _synthesize_final_textual_memory(*, backend: object, records: list[dict[str, object]]) -> dict[str, object]:
    evidence = _build_compact_training_evidence(records)
    training_dates = sorted(str(record.get("decision_date")) for record in records if record.get("decision_date"))
    training_window = {
        "start": training_dates[0] if training_dates else "YYYY-MM-DD",
        "end": training_dates[-1] if training_dates else "YYYY-MM-DD",
        "sample_count": len(records),
    }
    reflection_notes = []
    for record in records:
        note = record.get("reflection_note")
        if isinstance(note, dict):
            reflection_notes.append({
                "decision_date": record.get("decision_date"),
                "actual_label": record.get("actual_label"),
                "case_quality_summary": note.get("case_quality_summary"),
                "helpful_theory_signals": note.get("helpful_theory_signals"),
                "misleading_theory_signals": note.get("misleading_theory_signals"),
                "special_condition_if_any": note.get("special_condition_if_any"),
                "candidate_general_lesson": note.get("candidate_general_lesson"),
                "do_not_overgeneralize": note.get("do_not_overgeneralize"),
                "policy_stage_lesson": note.get("policy_stage_lesson"),
                "theory_level_lessons": note.get("theory_level_lessons"),
            })
    system_prompt = "\n".join([
        "你是系统C的最终训练记忆合成Agent。",
        "你的任务是把训练集反思记录压缩为可被预测期决策层使用的结构化理论适用性记忆。",
        "只能依据训练集样本、训练集真实标签和训练期reflection notes。",
        "不得使用测试集结果、测试期错误区间、测试期年份、测试期阶段标签或任何事后答案模板。",
        "不得写成按年份、按月份或按具体测试期事件的预测规则。",
        "必须把经验抽象为可观察的理论冲突结构、理论适用边界和方向/幅度判断原则。",
        "结构化条件规则必须服务于：在某类理论冲突结构下，当前样本更应该信哪些理论、下调哪些理论，以及幅度如何约束。",
        "每条规则必须包含boundary_conditions，防止过度泛化。",
        "如果训练证据不足，reliability_level必须标为low，不能夸大。",
        "只返回一个JSON对象，不要额外文字。",
    ])
    required_contract = {
        "schema_version": SYSTEM_C_TEXTUAL_MEMORY_SCHEMA,
        "memory_scope": "training-set reflection notes converted into structured conditional theory-applicability memory",
        "training_window": training_window,
        "global_lessons": [{
            "lesson": "训练集支持的总体理论适用性经验，不能包含测试期年份或答案模板",
            "evidence_strength": "strong/medium/weak",
            "helps_with": "direction/magnitude/zero_change/conflict_resolution/confidence",
            "overgeneralization_risk": "该经验何时可能误导",
        }],
        "normal_regime_lessons": [{
            "lesson": "常规政策环境中的理论适用经验",
            "evidence_strength": "strong/medium/weak",
            "helps_with": "direction/magnitude/zero_change/conflict_resolution/confidence",
        }],
        "special_regime_lessons": [{
            "lesson": "训练集中可观察到的非常规或高冲突情形下的经验，不得引用测试期事件",
            "evidence_strength": "strong/medium/weak",
            "helps_with": "direction/magnitude/zero_change/conflict_resolution/confidence",
            "extrapolation_warning": "迁移到测试期时的边界",
        }],
        "conditional_applicability_rules": [{
            "rule_id": "R001",
            "conflict_pattern": "例如：Taylor类因通胀压力支持加息，但路径敏感理论支持维持",
            "observable_conditions": ["只能写预测期可由理论输出、Analysis交接或可见宏观变量观察到的条件"],
            "preferred_direction": "cut/hold/hike/uncertain",
            "preferred_theory_family": ["七个理论ID中的一个或多个"],
            "downweighted_theory_family": ["七个理论ID中的一个或多个"],
            "magnitude_guidance": "方向确定后的幅度约束，例如小幅、维持、避免大幅等",
            "boundary_conditions": ["该规则不适用或应弱化的条件"],
            "training_evidence_summary": "训练集依据摘要，只能来自训练反思",
            "reliability_level": "high/medium/low",
        }],
        "theory_specific_applicability": {
            theory_id: [{
                "useful_when": "该理论在训练集中更可靠的可观察条件",
                "misleading_when": "该理论在训练集中容易误导的可观察条件",
                "decision_layer_use": "预测期决策层如何使用该理论",
                "evidence_strength": "strong/medium/weak",
                "helps_with": "direction/magnitude/zero_change/conflict_resolution/confidence",
            }] for theory_id in THEORY_IDS
        },
        "conflict_resolution_lessons": [{
            "lesson": "理论冲突时的处理经验",
            "evidence_strength": "strong/medium/weak",
            "helps_with": "direction/magnitude/conflict_resolution/confidence",
        }],
        "uncertainty_lessons": [{
            "lesson": "训练集中高不确定样本的处理经验",
            "evidence_strength": "strong/medium/weak",
        }],
        "do_not_use_rules": [
            "不得把文字记忆当成固定数值权重。",
            "不得写测试期年份、测试期阶段标签或测试期答案模板。",
            "不得因为单个训练案例就泛化为强规则。",
            "不得替代七个理论Agent的当前样本判断。",
            "不得使用预测期真实标签。",
        ],
        "memory_quality_warning": "说明训练记忆的主要局限。",
    }
    user_prompt = "\n".join([
        "[训练集紧凑证据]",
        json.dumps(evidence, ensure_ascii=False, indent=2),
        "",
        "[训练集逐样本反思记录]",
        json.dumps(reflection_notes, ensure_ascii=False, indent=2),
        "",
        "[必须遵守的输出契约]",
        json.dumps(required_contract, ensure_ascii=False, indent=2),
    ])
    payload = _call_json_memory_agent(backend, system_prompt=system_prompt, user_prompt=user_prompt)
    payload["schema_version"] = SYSTEM_C_TEXTUAL_MEMORY_SCHEMA
    payload.setdefault("training_window", training_window)
    payload["source_record_count"] = len(records)
    return validate_textual_theory_memory("textual_theory_applicability_memory_train_final", payload)

def _stringify_lesson(value: object) -> str:
    if isinstance(value, str):
        return value.strip()
    return json.dumps(value, ensure_ascii=False)


def _flatten_memory_lessons(memory: dict[str, object]) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    section_names = ["global_lessons", "normal_regime_lessons", "special_regime_lessons", "conditional_applicability_rules", "conflict_resolution_lessons", "uncertainty_lessons", "do_not_use_rules"]
    for section in section_names:
        values = memory.get(section, [])
        if isinstance(values, list):
            for index, value in enumerate(values, 1):
                rows.append({"section": section, "theory_id": "", "lesson_index": index, "lesson": _stringify_lesson(value)})
    theory_memory = memory.get("theory_specific_applicability", {})
    if isinstance(theory_memory, dict):
        for theory_id in THEORY_IDS:
            values = theory_memory.get(theory_id, [])
            if isinstance(values, list):
                for index, value in enumerate(values, 1):
                    rows.append({"section": "theory_specific_applicability", "theory_id": theory_id, "lesson_index": index, "lesson": _stringify_lesson(value)})
    return rows


def _write_textual_memory_readable(*, memory: dict[str, object], path: Path) -> Path:
    lines: list[str] = []
    lines.append("# 系统C冻结文字化理论适用性记忆")
    lines.append("")
    lines.append(f"schema_version: {memory.get('schema_version')}")
    lines.append(f"source_record_count: {memory.get('source_record_count')}")
    training_window = memory.get("training_window", {})
    lines.append(f"training_window: {json.dumps(training_window, ensure_ascii=False)}")
    lines.append("")
    for section in ["global_lessons", "normal_regime_lessons", "special_regime_lessons", "conflict_resolution_lessons", "uncertainty_lessons", "do_not_use_rules"]:
        lines.append(f"## {section}")
        values = memory.get(section, [])
        if isinstance(values, list) and values:
            for value in values:
                lines.append(f"- {_stringify_lesson(value)}")
        else:
            lines.append("- <empty>")
        lines.append("")
    lines.append("## theory_specific_applicability")
    theory_memory = memory.get("theory_specific_applicability", {})
    if not isinstance(theory_memory, dict):
        theory_memory = {}
    for theory_id in THEORY_IDS:
        lines.append(f"### {theory_id}")
        values = theory_memory.get(theory_id, [])
        if isinstance(values, list) and values:
            for value in values:
                lines.append(f"- {_stringify_lesson(value)}")
        else:
            lines.append("- <empty>")
        lines.append("")
    if memory.get("memory_quality_warning"):
        lines.append("## memory_quality_warning")
        lines.append(str(memory.get("memory_quality_warning")))
        lines.append("")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(lines), encoding="utf-8")
    return path


def persist_compact_training_evidence(*, artifact_root: Path, records: list[dict[str, object]]) -> Path:
    evidence = _build_compact_training_evidence(records)
    path = artifact_root / "training_history" / "summaries" / "compact_training_evidence_summary.json"
    _json_write(path, evidence)
    return path

def persist_textual_theory_memory(*, artifact_root: Path, memory: dict[str, object], completed_step_count: int) -> Path:
    payload = dict(validate_textual_theory_memory("memory", memory))
    payload["completed_training_step_count"] = int(completed_step_count)
    payload["updated_at"] = datetime.now(timezone.utc).isoformat()
    training_root = artifact_root / "training_history"
    summaries_dir = training_root / "summaries"
    path = training_root / "textual_theory_applicability_memory_train_final.json"
    _json_write(path, payload)
    _write_textual_memory_readable(memory=payload, path=training_root / "textual_theory_applicability_memory_train_final.md")
    _write_textual_memory_readable(memory=payload, path=summaries_dir / "textual_theory_applicability_memory_train_final.md")
    _write_csv(summaries_dir / "textual_memory_lessons_all.csv", _flatten_memory_lessons(payload), ["section", "theory_id", "lesson_index", "lesson"])
    return path


def save_frozen_textual_memory_snapshot(*, artifact_root: Path, memory: dict[str, object], completed_step_count: int) -> Path:
    payload = {
        "schema_version": SYSTEM_C_FROZEN_SNAPSHOT_SCHEMA,
        "system_variant": globals().get("NOTEBOOK_SYSTEM_VARIANT"),
        "notebook_runtime_revision": globals().get("NOTEBOOK_RUNTIME_REVISION"),
        "train_test_split_date": TRAIN_TEST_SPLIT_DATE.isoformat(),
        "completed_training_step_count": int(completed_step_count),
        "textual_theory_applicability_memory_train_final": validate_textual_theory_memory("memory", memory),
        "frozen_at": datetime.now(timezone.utc).isoformat(),
    }
    path = artifact_root / "frozen_runtime_snapshot.json"
    _json_write(path, payload)
    return path


def load_frozen_textual_memory_snapshot(path: Path) -> dict[str, object]:
    if not path.exists():
        raise FileNotFoundError(f"Frozen textual memory snapshot does not exist: {path}")
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise ValueError("Frozen textual memory snapshot must be a JSON object.")
    payload["textual_theory_applicability_memory_train_final"] = validate_textual_theory_memory("textual_theory_applicability_memory_train_final", payload.get("textual_theory_applicability_memory_train_final"))
    return payload


def run_textual_memory_training(*, application: USRuntimeApplication, artifact_root: Path, run_id: str, title: str = "train", progress_callback: Callable[[dict[str, Any]], None] | None = None) -> TextualMemoryTrainingRunResult:
    training_schedule = application.scheduler.training_schedule()
    skip_sample_ids = successful_training_sample_ids(artifact_root)
    new_records: list[dict[str, object]] = []
    total_samples = len(training_schedule)
    if progress_callback is not None:
        progress_callback({"event": "training_started", "completed_samples": len(skip_sample_ids), "total_samples": total_samples})
    set_current_textual_theory_memory(DEFAULT_TEXTUAL_THEORY_MEMORY)
    backend = application.runtime_assembly.backend_bundle.decision_backend
    for sample_index, sample_ref in enumerate(training_schedule):
        if sample_ref.sample_id in skip_sample_ids:
            if progress_callback is not None:
                saved_records = load_successful_training_step_records(artifact_root)
                saved_record = next((record for record in saved_records if record.get("sample_id") == sample_ref.sample_id), {})
                progress_callback({"event": "training_sample_skipped", "sample_index": sample_index, "sample_id": sample_ref.sample_id, "decision_date": sample_ref.decision_date.isoformat(), "skip_reason": "already_successful_training_step", "decision_prediction": saved_record.get("case_record", {}).get("decision_prediction"), "actual_label": saved_record.get("actual_label"), "decision_correct": saved_record.get("case_record", {}).get("decision_correct"), "completed_samples": len(saved_records), "total_samples": total_samples})
            continue
        if progress_callback is not None:
            progress_callback({"event": "training_sample_started", "sample_index": sample_index, "sample_id": sample_ref.sample_id, "decision_date": sample_ref.decision_date.isoformat(), "completed_samples": len(load_successful_training_step_records(artifact_root)), "total_samples": total_samples})
        sample, _ = application.scheduler.build_training_sample(sample_index)
        pipeline_result = application.runtime_assembly.prediction_pipeline.run_single_sample(sample=sample, sample_id=sample_ref.sample_id)
        actual_label = sample.realized_label
        case_record = _build_case_record(sample_ref=sample_ref, pipeline_result=pipeline_result, actual_label=actual_label)
        reflection_note = _reflect_on_case(backend=backend, case_record=case_record)
        record = _training_step_to_record(sample_ref=sample_ref, pipeline_result=pipeline_result, actual_label=actual_label, reflection_note=reflection_note, case_record=case_record)
        step_path = persist_training_step_record(record=record, artifact_root=artifact_root)
        new_records.append(record)
        if progress_callback is not None:
            progress_callback({"event": "training_sample_completed", "sample_index": sample_index, "sample_id": sample_ref.sample_id, "decision_date": sample_ref.decision_date.isoformat(), "decision_prediction": case_record.get("decision_prediction"), "actual_label": _label_value(actual_label), "decision_correct": case_record.get("decision_correct"), "path": str(step_path), "completed_samples": len(load_successful_training_step_records(artifact_root)), "total_samples": total_samples})
    all_records = load_successful_training_step_records(artifact_root)
    persist_compact_training_evidence(artifact_root=artifact_root, records=all_records)
    if len(all_records) < total_samples:
        memory = get_current_textual_theory_memory()
        memory_path = persist_textual_theory_memory(artifact_root=artifact_root, memory=memory, completed_step_count=len(all_records))
    else:
        frozen_snapshot_path = artifact_root / "frozen_runtime_snapshot.json"
        final_memory_path = artifact_root / "training_history" / "textual_theory_applicability_memory_train_final.json"
        if not new_records and frozen_snapshot_path.exists():
            snapshot = load_frozen_textual_memory_snapshot(frozen_snapshot_path)
            memory = dict(snapshot["textual_theory_applicability_memory_train_final"])
            memory_path = final_memory_path
            if not final_memory_path.exists():
                memory_path = persist_textual_theory_memory(artifact_root=artifact_root, memory=memory, completed_step_count=len(all_records))
        elif not new_records and final_memory_path.exists():
            memory = validate_textual_theory_memory("textual_theory_applicability_memory_train_final", json.loads(final_memory_path.read_text(encoding="utf-8")))
            memory_path = final_memory_path
        else:
            memory = _synthesize_final_textual_memory(backend=backend, records=all_records)
            memory_path = persist_textual_theory_memory(artifact_root=artifact_root, memory=memory, completed_step_count=len(all_records))
            save_frozen_textual_memory_snapshot(artifact_root=artifact_root, memory=memory, completed_step_count=len(all_records))
    set_current_textual_theory_memory(memory)
    if progress_callback is not None:
        progress_callback({"event": "training_finished", "completed_samples": len(all_records), "total_samples": total_samples, "textual_memory_path": str(memory_path), "final_memory_ready": len(all_records) >= total_samples})
    return TextualMemoryTrainingRunResult(training_schedule=training_schedule, new_step_records=tuple(new_records), textual_theory_applicability_memory_train_final=memory, memory_path=memory_path)


def _prediction_item_to_step_record(item) -> dict[str, object]:
    decision_output = item.pipeline_result.decision_state.decision_output
    memory = get_current_textual_theory_memory()
    return {
        "schema_version": SYSTEM_C_PREDICTION_STEP_SCHEMA,
        "notebook_runtime_revision": globals().get("NOTEBOOK_RUNTIME_REVISION"),
        "theory_ids": list(THEORY_IDS),
        "sample_id": item.sample_ref.sample_id,
        "split": item.sample_ref.split,
        "decision_date": item.sample_ref.decision_date.isoformat(),
        "actual_label": getattr(item.actual_label, "value", item.actual_label),
        "frozen_textual_memory_schema": memory.get("schema_version"),
        "frozen_textual_memory_source_record_count": memory.get("source_record_count"),
        "decision_layer": to_jsonable(decision_output),
        "theory_layer": to_jsonable(item.pipeline_result.theory_layer_state.theory_outputs),
        "analysis_layer": to_jsonable(item.pipeline_result.analysis_state),
        "status": "success",
    }


def _read_prediction_manifest(prediction_root: Path) -> dict[str, object]:
    manifest_path = prediction_root / "manifest.json"
    fallback = {"schema_version": SYSTEM_C_PREDICTION_MANIFEST_SCHEMA, "steps": {}}
    if not manifest_path.exists():
        return fallback
    try:
        payload = json.loads(manifest_path.read_text(encoding="utf-8"))
    except Exception:
        return fallback
    if not isinstance(payload, dict):
        return fallback
    if not isinstance(payload.get("steps"), dict):
        payload["steps"] = {}
    payload.setdefault("schema_version", SYSTEM_C_PREDICTION_MANIFEST_SCHEMA)
    return payload


def successful_prediction_sample_ids(artifact_root: Path) -> set[str]:
    prediction_root = artifact_root / "prediction_history"
    manifest = _read_prediction_manifest(prediction_root)
    steps = manifest.get("steps") if isinstance(manifest, dict) else None
    if not isinstance(steps, dict):
        return set()
    successful: set[str] = set()
    for sample_id, item in steps.items():
        if not isinstance(item, dict) or item.get("status") != "success":
            continue
        raw_path = item.get("path")
        if not isinstance(raw_path, str) or not raw_path.strip():
            continue
        try:
            record = json.loads((prediction_root / raw_path).read_text(encoding="utf-8"))
        except Exception:
            continue
        if _step_record_matches_current_runtime(record):
            successful.add(str(sample_id))
    return successful

def persist_prediction_step_record(*, record: dict[str, object], artifact_root: Path) -> Path:
    prediction_root = artifact_root / "prediction_history"
    steps_dir = prediction_root / "steps"
    steps_dir.mkdir(parents=True, exist_ok=True)
    sample_id = str(record["sample_id"])
    step_path = steps_dir / f"{sample_id}.json"
    _json_write(step_path, record)
    manifest = _read_prediction_manifest(prediction_root)
    manifest.setdefault("steps", {})
    manifest["steps"][sample_id] = {"status": "success", "decision_date": record.get("decision_date"), "path": str(Path("steps") / f"{sample_id}.json")}
    _json_write(prediction_root / "manifest.json", manifest)
    write_prediction_summaries_from_records(records=load_all_successful_prediction_step_records(artifact_root), summaries_dir=prediction_root / "summaries", runs_dir=prediction_root / "runs")
    return step_path


def persist_prediction_step_item(*, item, artifact_root: Path) -> Path:
    return persist_prediction_step_record(record=_prediction_item_to_step_record(item), artifact_root=artifact_root)


def load_successful_prediction_step_records(artifact_root: Path) -> list[dict[str, object]]:
    prediction_root = artifact_root / "prediction_history"
    manifest = _read_prediction_manifest(prediction_root)
    records: list[dict[str, object]] = []
    steps = manifest.get("steps", {})
    if not isinstance(steps, dict):
        return records
    for item in steps.values():
        if not isinstance(item, dict) or item.get("status") != "success":
            continue
        relative_path = item.get("path")
        if not isinstance(relative_path, str) or not relative_path.strip():
            continue
        try:
            record = json.loads((prediction_root / relative_path).read_text(encoding="utf-8"))
        except Exception:
            continue
        if _step_record_matches_current_runtime(record):
            records.append(record)
    records.sort(key=lambda record: str(record.get("decision_date", "")))
    return records


def load_all_successful_prediction_step_records(artifact_root: Path) -> list[dict[str, object]]:
    prediction_root = artifact_root / "prediction_history"
    steps_dir = prediction_root / "steps"
    records: list[dict[str, object]] = []
    for step_path in sorted(steps_dir.glob("test_us_*.json")):
        try:
            record = json.loads(step_path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if record.get("status") == "success":
            records.append(record)
    records.sort(key=lambda record: str(record.get("decision_date", "")))
    return records

def persist_prediction_run_record(*, artifact_root: Path, run_id: str, title: str, status: str, step_ids: list[str] | None = None, error_message: str | None = None) -> Path:
    prediction_root = artifact_root / "prediction_history"
    runs_dir = prediction_root / "runs"
    runs_dir.mkdir(parents=True, exist_ok=True)
    run_record_path = runs_dir / f"run_{run_id}_{title}.json"
    created_at = datetime.now(timezone.utc).isoformat()
    if run_record_path.exists():
        try:
            existing = json.loads(run_record_path.read_text(encoding="utf-8"))
            if isinstance(existing, dict) and isinstance(existing.get("created_at"), str):
                created_at = existing["created_at"]
        except Exception:
            pass
    normalized_step_ids = list(step_ids or [])
    run_record = {"schema_version": SYSTEM_C_PREDICTION_MANIFEST_SCHEMA, "run_id": run_id, "title": title, "status": status, "notebook_runtime_revision": globals().get("NOTEBOOK_RUNTIME_REVISION"), "system_variant": globals().get("NOTEBOOK_SYSTEM_VARIANT"), "step_ids": normalized_step_ids, "completed_step_count": len(normalized_step_ids), "created_at": created_at, "updated_at": datetime.now(timezone.utc).isoformat()}
    if error_message:
        run_record["error_message"] = error_message
    _json_write(run_record_path, run_record)
    return run_record_path


def write_prediction_summaries_from_records(*, records: list[dict[str, object]], summaries_dir: Path, runs_dir: Path) -> None:
    summaries_dir.mkdir(parents=True, exist_ok=True)
    runs_dir.mkdir(parents=True, exist_ok=True)
    decision_rows, comparison_rows, probability_rows, weight_rows, contribution_rows = [], [], [], [], []
    error_rows, high_confidence_error_rows, theory_rows, analysis_rows = [], [], [], []
    for record in records:
        decision_output = record.get("decision_layer", {})
        if not isinstance(decision_output, dict):
            continue
        actual_label = record.get("actual_label")
        predicted_label = _label_value(decision_output.get("final_recommendation"))
        correct = predicted_label == actual_label
        supporting_rule_families = decision_output.get("supporting_rule_families", [])
        if not isinstance(supporting_rule_families, list):
            supporting_rule_families = []
        base = {"sample_id": record.get("sample_id"), "decision_date": record.get("decision_date"), "actual_label": actual_label, "final_recommendation": predicted_label, "is_correct": correct}
        decision_row = {**base, "final_confidence": decision_output.get("final_confidence"), "dominant_rule_family": decision_output.get("dominant_rule_family"), "supporting_rule_families": ";".join(str(value) for value in supporting_rule_families), "final_explanation": decision_output.get("final_explanation"), "theory_weight_rationale": decision_output.get("theory_weight_rationale"), "memory_usage_summary": decision_output.get("memory_usage_summary"), "matched_memory_lessons": json.dumps(decision_output.get("matched_memory_lessons", []), ensure_ascii=False), "first_contributor_rule": decision_output.get("first_contributor_rule"), "current_sample_reasoning": decision_output.get("current_sample_reasoning"), "remaining_uncertainty": decision_output.get("remaining_uncertainty"), "decision_path": decision_output.get("decision_path")}
        decision_rows.append(decision_row)
        comparison_rows.append({**base, "final_confidence": decision_output.get("final_confidence")})
        probability_distribution = decision_output.get("final_probability_distribution", {}) if isinstance(decision_output.get("final_probability_distribution", {}), dict) else {}
        probability_rows.append({**base, **{label.value: probability_distribution.get(label.value) for label in LABEL_ORDER}})
        weights = decision_output.get("theory_weight_profile", {}) if isinstance(decision_output.get("theory_weight_profile", {}), dict) else {}
        weight_rows.append({**base, **{theory_id: weights.get(theory_id) for theory_id in THEORY_IDS}})
        shares = decision_output.get("rule_contribution_shares", {}) if isinstance(decision_output.get("rule_contribution_shares", {}), dict) else {}
        contribution_rows.append({**base, **{theory_id: shares.get(theory_id) for theory_id in THEORY_IDS}, "first_contributor_rule": decision_output.get("first_contributor_rule")})
        theory_rows.append({**base, "theory_outputs": record.get("theory_layer")})
        analysis_rows.append({**base, "analysis_layer": record.get("analysis_layer")})
        if not correct:
            error_rows.append(decision_row)
            if isinstance(decision_output.get("final_confidence"), (int, float)) and decision_output.get("final_confidence") >= 80:
                high_confidence_error_rows.append(decision_row)
    decision_fields = ["sample_id", "decision_date", "actual_label", "final_recommendation", "is_correct", "final_confidence", "dominant_rule_family", "supporting_rule_families", "final_explanation", "theory_weight_rationale", "memory_usage_summary", "matched_memory_lessons", "first_contributor_rule", "current_sample_reasoning", "remaining_uncertainty", "decision_path"]
    _write_csv(summaries_dir / "decision_layer_all.csv", decision_rows, decision_fields)
    _write_csv(summaries_dir / "memory_usage_all.csv", [{key: row.get(key) for key in ["sample_id", "decision_date", "actual_label", "final_recommendation", "is_correct", "memory_usage_summary", "matched_memory_lessons", "theory_weight_rationale"]} for row in decision_rows], ["sample_id", "decision_date", "actual_label", "final_recommendation", "is_correct", "memory_usage_summary", "matched_memory_lessons", "theory_weight_rationale"])
    _write_jsonl(summaries_dir / "memory_usage_all.jsonl", [{"sample_id": row.get("sample_id"), "decision_date": row.get("decision_date"), "actual_label": row.get("actual_label"), "final_recommendation": row.get("final_recommendation"), "is_correct": row.get("is_correct"), "memory_usage_summary": row.get("memory_usage_summary"), "matched_memory_lessons": row.get("matched_memory_lessons"), "theory_weight_rationale": row.get("theory_weight_rationale")} for row in decision_rows])
    _write_csv(summaries_dir / "prediction_vs_actual.csv", comparison_rows, ["sample_id", "decision_date", "actual_label", "final_recommendation", "is_correct", "final_confidence"])
    _write_csv(summaries_dir / "decision_probabilities_all.csv", probability_rows, ["sample_id", "decision_date", "actual_label", "final_recommendation", "is_correct", *[label.value for label in LABEL_ORDER]])
    _write_csv(summaries_dir / "weight_history_all.csv", weight_rows, ["sample_id", "decision_date", "actual_label", "final_recommendation", "is_correct", *THEORY_IDS])
    _write_csv(summaries_dir / "rule_contribution_shares_all.csv", contribution_rows, ["sample_id", "decision_date", "actual_label", "final_recommendation", "is_correct", *THEORY_IDS, "first_contributor_rule"])
    _write_jsonl(summaries_dir / "theory_outputs_all.jsonl", theory_rows)
    _write_jsonl(summaries_dir / "analysis_layer_all.jsonl", analysis_rows)
    _write_csv(summaries_dir / "decision_errors_only.csv", error_rows, decision_fields)
    _write_csv(summaries_dir / "high_confidence_errors.csv", high_confidence_error_rows, decision_fields)


def persist_prediction_result(*, result: DirectInferenceRunResult, artifact_root: Path, run_id: str, title: str) -> dict[str, object]:
    records = [_prediction_item_to_step_record(item) for item in result.items]
    step_paths = [persist_prediction_step_record(record=record, artifact_root=artifact_root) for record in records]
    all_step_ids = sorted(successful_prediction_sample_ids(artifact_root))
    run_record = persist_prediction_run_record(artifact_root=artifact_root, run_id=run_id, title=title, status="completed", step_ids=all_step_ids)
    return {"step_paths": step_paths, "step_count": len(step_paths), "total_completed_step_count": len(all_step_ids), "run_record": run_record}



## LangGraph Direct Prediction Graph

In [ ]:
from typing import Any, Literal, TypedDict

from langgraph.graph import END, START, StateGraph

from application_entry import USRuntimeApplication, USRuntimeApplicationConfig, build_us_runtime_application
from inference_loop import DirectInferenceRunResult


In [ ]:
class SystemGraphState(TypedDict, total=False):
    mode: Literal["train", "predict", "train_then_predict"]
    runtime_title: str
    runtime_run_id: str
    app_config: USRuntimeApplicationConfig
    application: USRuntimeApplication
    prediction_output: Path
    frozen_snapshot_input: Path
    frozen_snapshot_output: Path
    training_result: TextualMemoryTrainingRunResult
    inference_result: DirectInferenceRunResult
    textual_theory_applicability_memory: dict[str, object]
    status: str
    message_log: list[str]
    runtime_diagnostics: dict[str, object]


In [ ]:
NOTEBOOK_RUNTIME_REVISION = "system_c_training_set_textual_memory_2026_06_15_v7_softened_continuous_language"


def append_message(state: SystemGraphState, message: str) -> list[str]:
    return [*state.get("message_log", []), message]


def runtime_diagnostics_summary(application: USRuntimeApplication) -> dict[str, object]:
    backend = application.runtime_assembly.backend_bundle.theory_backend
    return {
        "notebook_runtime_revision": NOTEBOOK_RUNTIME_REVISION,
        "system_variant": NOTEBOOK_SYSTEM_VARIANT,
        "backend_resolved_config": backend.resolved_config(),
        "training_enabled": True,
        "textual_memory_enabled": True,
        "memory_method": "case_reflection_notes_plus_final_memory_synthesis",
        "prompt_optimization_enabled": False,
    }


def build_application_node(state: SystemGraphState) -> SystemGraphState:
    application = build_us_runtime_application(state["app_config"])
    diagnostics = runtime_diagnostics_summary(application)
    messages = append_message(state, "System C training-set textual memory runtime application assembled")
    messages.extend([
        f"notebook_revision: {diagnostics['notebook_runtime_revision']}",
        f"system_variant: {diagnostics['system_variant']}",
        f"training_enabled: {diagnostics['training_enabled']}",
        f"textual_memory_enabled: {diagnostics['textual_memory_enabled']}",
        f"memory_method: {diagnostics['memory_method']}",
        f"prompt_optimization_enabled: {diagnostics['prompt_optimization_enabled']}",
        f"backend_config: {json.dumps(diagnostics['backend_resolved_config'], ensure_ascii=False)}",
    ])
    return {**state, "application": application, "runtime_diagnostics": diagnostics, "status": "application_built", "message_log": messages}


def run_training_node(state: SystemGraphState) -> SystemGraphState:
    artifact_root = framework_dir / "artifacts"
    run_id = str(state.get("runtime_run_id") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
    title = str(state.get("runtime_title") or "train")
    persist_training_run_record(artifact_root=artifact_root, run_id=run_id, title=title, status="running", step_ids=sorted(successful_training_sample_ids(artifact_root)))
    emit_runtime_stage("training_schedule_build_started", {"run_id": run_id, "skip_successful_samples": len(successful_training_sample_ids(artifact_root))})
    schedule = state["application"].scheduler.training_schedule()
    schedule_payload = {"run_id": run_id, "total_samples": len(schedule), "skip_successful_samples": len(successful_training_sample_ids(artifact_root))}
    if schedule:
        schedule_payload.update({"first_sample_id": schedule[0].sample_id, "first_decision_date": schedule[0].decision_date.isoformat(), "last_sample_id": schedule[-1].sample_id, "last_decision_date": schedule[-1].decision_date.isoformat()})
    emit_runtime_stage("training_schedule_ready", schedule_payload)

    def _progress_callback(event: dict[str, Any]) -> None:
        if str(event.get("event", "")) in {"training_started", "training_sample_started", "training_sample_skipped", "training_sample_completed", "training_finished"}:
            emit_runtime_stage(str(event["event"]), event)
            if event.get("event") in {"training_sample_completed", "training_sample_skipped"}:
                status_label = "completed" if event.get("event") == "training_sample_completed" else "skipped"
                print(f"[train {status_label}] sample={event.get('sample_id')} date={event.get('decision_date')} prediction={event.get('decision_prediction')} actual={event.get('actual_label')} correct={event.get('decision_correct')} completed={event.get('completed_samples')}/{event.get('total_samples')}")
            if event.get("event") == "training_sample_completed":
                persist_training_run_record(artifact_root=artifact_root, run_id=run_id, title=title, status="running", step_ids=sorted(successful_training_sample_ids(artifact_root)))

    result = run_textual_memory_training(application=state["application"], artifact_root=artifact_root, run_id=run_id, title=title, progress_callback=_progress_callback)
    frozen_path = artifact_root / "frozen_runtime_snapshot.json"
    if frozen_path.exists():
        snapshot_path = frozen_path
    else:
        snapshot_path = save_frozen_textual_memory_snapshot(artifact_root=artifact_root, memory=result.textual_theory_applicability_memory_train_final, completed_step_count=len(load_successful_training_step_records(artifact_root)))
    persist_training_run_record(artifact_root=artifact_root, run_id=run_id, title=title, status="completed", step_ids=sorted(successful_training_sample_ids(artifact_root)), memory_path=str(result.memory_path))
    return {**state, "training_result": result, "textual_theory_applicability_memory": result.textual_theory_applicability_memory_train_final, "frozen_snapshot_output": snapshot_path, "status": "training_finished", "message_log": append_message(state, f"Textual memory training completed: {len(result.new_step_records)} new training samples")}


def load_frozen_memory_node(state: SystemGraphState) -> SystemGraphState:
    artifact_root = framework_dir / "artifacts"
    snapshot_path = Path(state.get("frozen_snapshot_input") or artifact_root / "frozen_runtime_snapshot.json")
    payload = load_frozen_textual_memory_snapshot(snapshot_path)
    memory = payload["textual_theory_applicability_memory_train_final"]
    set_current_textual_theory_memory(memory)
    emit_runtime_stage("frozen_textual_memory_loaded", {"path": str(snapshot_path), "source_record_count": memory.get("source_record_count")})
    return {**state, "textual_theory_applicability_memory": memory, "status": "frozen_textual_memory_loaded", "message_log": append_message(state, f"Frozen textual memory loaded: {snapshot_path}")}


def existing_successful_prediction_step_ids(artifact_root: Path) -> set[str]:
    steps_dir = artifact_root / "prediction_history" / "steps"
    successful: set[str] = set()
    if not steps_dir.exists():
        return successful
    for step_path in steps_dir.glob("test_us_*.json"):
        try:
            record = json.loads(step_path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if record.get("status") == "success" and isinstance(record.get("sample_id"), str):
            successful.add(str(record["sample_id"]))
    return successful


def run_inference_node(state: SystemGraphState) -> SystemGraphState:
    artifact_root = framework_dir / "artifacts"
    run_id = str(state.get("runtime_run_id") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
    title = "predict"
    memory = state.get("textual_theory_applicability_memory") or get_current_textual_theory_memory()
    set_current_textual_theory_memory(memory)
    skip_sample_ids = existing_successful_prediction_step_ids(artifact_root)
    persist_prediction_run_record(artifact_root=artifact_root, run_id=run_id, title=title, status="running", step_ids=sorted(skip_sample_ids))
    emit_runtime_stage("prediction_schedule_build_started", {"run_id": run_id, "skip_successful_samples": len(skip_sample_ids), "frozen_textual_memory_schema": memory.get("schema_version"), "memory_source_record_count": memory.get("source_record_count")})
    test_schedule = state["application"].scheduler.test_schedule()
    schedule_payload = {"run_id": run_id, "total_samples": len(test_schedule), "skip_successful_samples": len(skip_sample_ids)}
    if test_schedule:
        schedule_payload.update({"first_sample_id": test_schedule[0].sample_id, "first_decision_date": test_schedule[0].decision_date.isoformat(), "last_sample_id": test_schedule[-1].sample_id, "last_decision_date": test_schedule[-1].decision_date.isoformat()})
    emit_runtime_stage("prediction_schedule_ready", schedule_payload)
    if skip_sample_ids:
        emit_runtime_stage("prediction_resume_plan", {"skip_successful_samples": len(skip_sample_ids), "skip_sample_ids": sorted(skip_sample_ids)})

    def _progress_callback(event: dict[str, Any]) -> None:
        if str(event.get("event", "")) in {"inference_started", "inference_sample_started", "inference_sample_skipped", "inference_sample_failed", "inference_sample_completed", "inference_finished"}:
            emit_runtime_stage(str(event["event"]), event)

    def _completed_item_callback(item) -> None:
        step_path = persist_prediction_step_item(item=item, artifact_root=artifact_root)
        persist_prediction_run_record(artifact_root=artifact_root, run_id=run_id, title=title, status="running", step_ids=sorted(successful_prediction_sample_ids(artifact_root)))
        emit_runtime_stage("prediction_step_persisted", {"sample_id": item.sample_ref.sample_id, "decision_date": item.sample_ref.decision_date.isoformat(), "path": str(step_path)})

    result = state["application"].run_direct_inference(progress_callback=_progress_callback, skip_sample_ids=skip_sample_ids, completed_item_callback=_completed_item_callback)
    persist_prediction_run_record(artifact_root=artifact_root, run_id=run_id, title=title, status="graph_finished", step_ids=sorted(successful_prediction_sample_ids(artifact_root)))
    return {**state, "inference_result": result, "status": "inference_finished", "message_log": append_message(state, f"Frozen-textual-memory test inference completed: {len(result.items)} new samples")}


def save_prediction_output_node(state: SystemGraphState) -> SystemGraphState:
    path = state["application"].save_prediction_output(state["prediction_output"], state["inference_result"])
    return {**state, "status": "prediction_output_saved", "message_log": append_message(state, f"Prediction output saved: {path}")}


def route_after_build(state: SystemGraphState) -> str:
    mode = state.get("mode", "train_then_predict")
    if mode == "train":
        return "run_training"
    if mode == "predict":
        return "load_frozen_memory"
    return "run_training"


def route_after_training(state: SystemGraphState) -> str:
    return END if state.get("mode") == "train" else "run_inference"


In [ ]:
graph_builder = StateGraph(SystemGraphState)
graph_builder.add_node("build_application", build_application_node)
graph_builder.add_node("run_training", run_training_node)
graph_builder.add_node("load_frozen_memory", load_frozen_memory_node)
graph_builder.add_node("run_inference", run_inference_node)
graph_builder.add_node("save_prediction_output", save_prediction_output_node)
graph_builder.add_edge(START, "build_application")
graph_builder.add_conditional_edges("build_application", route_after_build, {"run_training": "run_training", "load_frozen_memory": "load_frozen_memory"})
graph_builder.add_conditional_edges("run_training", route_after_training, {"run_inference": "run_inference", END: END})
graph_builder.add_edge("load_frozen_memory", "run_inference")
graph_builder.add_edge("run_inference", "save_prediction_output")
graph_builder.add_edge("save_prediction_output", END)
system_graph = graph_builder.compile()
print(system_graph)


## Run System C Training-Set Textual Memory System


In [ ]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

artifact_dir = framework_dir / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)
trace_artifact_dir = artifact_dir / "debug_traces"
trace_artifact_dir.mkdir(parents=True, exist_ok=True)
for path in (
    artifact_dir / "training_history" / "steps",
    artifact_dir / "training_history" / "reflection_notes",
    artifact_dir / "training_history" / "runs",
    artifact_dir / "training_history" / "summaries",
    artifact_dir / "prediction_history" / "steps",
    artifact_dir / "prediction_history" / "runs",
    artifact_dir / "prediction_history" / "summaries",
):
    path.mkdir(parents=True, exist_ok=True)

required_backend_api_key_env = "DEEPSEEK_API_KEY"
if not os.environ.get(required_backend_api_key_env):
    raise RuntimeError("Set DEEPSEEK_API_KEY in your environment or local .env file before continuing.")

shared_app_config = USRuntimeApplicationConfig(
    project_root=project_root,
    hp_lambda=129600.0,
    model_name="deepseek-chat",
    api_base="https://api.deepseek.com/v1",
    api_key_env="DEEPSEEK_API_KEY",
    timeout_seconds=120,
    temperature=0.2,
    max_output_tokens=None,
    max_retries=5,
    initial_backoff_seconds=2.0,
    backoff_multiplier=2.0,
    use_json_schema=False,
    fallback_to_json_object=True,
    include_response_format=True,
)

debug_application = build_us_runtime_application(shared_app_config)
default_prediction_output = artifact_dir / "prediction_results.json"
default_frozen_snapshot = artifact_dir / "frozen_runtime_snapshot.json"
debug_backend = debug_application.runtime_assembly.backend_bundle.theory_backend
print("Resolved backend config:")
print(json.dumps(debug_backend.resolved_config(), indent=2))
print("Notebook runtime revision:", NOTEBOOK_RUNTIME_REVISION)
print("System variant:", NOTEBOOK_SYSTEM_VARIANT)
print("Training enabled: True")
print("Textual memory enabled: True")
print("Memory method: per-case reflection notes + final Memory-Synthesis Agent")
print("Prompt optimization enabled: False")
print("Train/test split date:", TRAIN_TEST_SPLIT_DATE.isoformat())
print(f"Trace output dir: {trace_artifact_dir}")
print(f"Training history dir: {artifact_dir / 'training_history'}")
print(f"Prediction history dir: {artifact_dir / 'prediction_history'}")


def run_system_c_train_then_predict() -> dict[str, object]:
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    title = "train_then_predict"
    reset_runtime_trace(title)
    set_runtime_live_trace_path(trace_artifact_dir / f"{title}_trace_live.json")
    initial_state: SystemGraphState = {
        "mode": "train_then_predict",
        "runtime_title": "train",
        "runtime_run_id": run_id,
        "app_config": shared_app_config,
        "frozen_snapshot_input": default_frozen_snapshot,
        "frozen_snapshot_output": default_frozen_snapshot,
        "prediction_output": default_prediction_output,
        "message_log": [],
    }
    final_state = None
    try:
        for chunk in system_graph.stream(initial_state, stream_mode="updates"):
            for node_name, node_state in chunk.items():
                final_state = node_state
                print(f"[{title}] node={node_name} status={node_state.get('status')}")
                for message in node_state.get("message_log", []):
                    print(f"[{title}] message={message}")
        if final_state is None:
            final_state = system_graph.invoke(initial_state)
        result = final_state.get("inference_result")
        history_result = {"step_count": 0, "total_completed_step_count": len(load_successful_prediction_step_records(artifact_dir)), "run_record": None}
        accuracy = None
        total = 0
        correct = 0
        if result is not None:
            history_result = persist_prediction_result(result=result, artifact_root=artifact_dir, run_id=run_id, title="predict")
            total = len(result.items)
            correct = sum(1 for item in result.items if _label_value(item.pipeline_result.decision_state.decision_output.final_recommendation) == _label_value(item.actual_label))
            accuracy = None if total == 0 else correct / total
        live_trace_path = save_live_runtime_trace(title, trace_artifact_dir)
        trace_path = save_runtime_trace(title, trace_artifact_dir, run_id=run_id)
        memory = get_current_textual_theory_memory()
        print(f"[{title}] ended status={final_state.get('status')}")
        print(f"[{title}] new_prediction_samples={total} correct={correct} accuracy={accuracy}")
        print(f"[{title}] textual_memory_schema={memory.get('schema_version')} source_record_count={memory.get('source_record_count')}")
        print(f"[{title}] live_trace_saved={live_trace_path}")
        print(f"[{title}] trace_saved={trace_path}")
        print(f"[{title}] prediction_run_record={history_result['run_record']}")
        return {"final_state": final_state, "run_id": run_id, "new_prediction_samples": total, "correct": correct, "accuracy": accuracy, "live_trace_path": live_trace_path, "trace_path": trace_path, **history_result}
    except Exception as exc:
        persist_training_run_record(artifact_root=artifact_dir, run_id=run_id, title="train", status="failed", step_ids=sorted(successful_training_sample_ids(artifact_dir)), error_message=f"{type(exc).__name__}: {exc}")
        persist_prediction_run_record(artifact_root=artifact_dir, run_id=run_id, title="predict", status="failed", step_ids=sorted(successful_prediction_sample_ids(artifact_dir)), error_message=f"{type(exc).__name__}: {exc}")
        trace_path = save_runtime_trace(title, trace_artifact_dir, run_id=run_id)
        print(f"[{title}] exception={type(exc).__name__}: {exc}")
        print(f"[{title}] trace_saved={trace_path}")
        raise


def run_system_c_predict_only() -> dict[str, object]:
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    title = "predict_only"
    reset_runtime_trace(title)
    set_runtime_live_trace_path(trace_artifact_dir / f"{title}_trace_live.json")
    initial_state: SystemGraphState = {
        "mode": "predict",
        "runtime_title": "predict",
        "runtime_run_id": run_id,
        "app_config": shared_app_config,
        "frozen_snapshot_input": default_frozen_snapshot,
        "prediction_output": default_prediction_output,
        "message_log": [],
    }
    final_state = None
    try:
        for chunk in system_graph.stream(initial_state, stream_mode="updates"):
            for node_name, node_state in chunk.items():
                final_state = node_state
                print(f"[{title}] node={node_name} status={node_state.get('status')}")
                for message in node_state.get("message_log", []):
                    print(f"[{title}] message={message}")
        if final_state is None:
            final_state = system_graph.invoke(initial_state)
        result = final_state.get("inference_result")
        history_result = {"step_count": 0, "total_completed_step_count": len(load_successful_prediction_step_records(artifact_dir)), "run_record": None}
        accuracy = None
        total = 0
        correct = 0
        if result is not None:
            history_result = persist_prediction_result(result=result, artifact_root=artifact_dir, run_id=run_id, title="predict")
            total = len(result.items)
            correct = sum(1 for item in result.items if _label_value(item.pipeline_result.decision_state.decision_output.final_recommendation) == _label_value(item.actual_label))
            accuracy = None if total == 0 else correct / total
        live_trace_path = save_live_runtime_trace(title, trace_artifact_dir)
        trace_path = save_runtime_trace(title, trace_artifact_dir, run_id=run_id)
        memory = get_current_textual_theory_memory()
        print(f"[{title}] ended status={final_state.get('status')}")
        print(f"[{title}] new_prediction_samples={total} correct={correct} accuracy={accuracy}")
        print(f"[{title}] textual_memory_schema={memory.get('schema_version')} source_record_count={memory.get('source_record_count')}")
        print(f"[{title}] live_trace_saved={live_trace_path}")
        print(f"[{title}] trace_saved={trace_path}")
        print(f"[{title}] prediction_run_record={history_result['run_record']}")
        return {"final_state": final_state, "run_id": run_id, "new_prediction_samples": total, "correct": correct, "accuracy": accuracy, "live_trace_path": live_trace_path, "trace_path": trace_path, **history_result}
    except Exception as exc:
        persist_prediction_run_record(artifact_root=artifact_dir, run_id=run_id, title="predict", status="failed", step_ids=sorted(successful_prediction_sample_ids(artifact_dir)), error_message=f"{type(exc).__name__}: {exc}")
        trace_path = save_runtime_trace(title, trace_artifact_dir, run_id=run_id)
        print(f"[{title}] exception={type(exc).__name__}: {exc}")
        print(f"[{title}] trace_saved={trace_path}")
        raise


# Explicitly opt in to the full training and prediction run after reviewing README.md.
# system_c_result = run_system_c_train_then_predict()
